In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("javaidahmadwani/lc25000")

print("Path to dataset files:", path)

In [ ]:
import os
from pathlib import Path
from collections import Counter

# Dataset directory
dataset_path = Path("/kaggle/input/datasets/javaidahmadwani/lc25000/lung_colon_image_set/Train and Validation Set")

# Supported image extensions
image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp", ".gif"
}

# Check whether the dataset path exists
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset path not found: {dataset_path}")

print(f"Dataset path: {dataset_path}")
print("=" * 80)

# Show directory structure and image count for every folder
total_images = 0
folder_results = []

for root, dirs, files in os.walk(dataset_path):
    root_path = Path(root)

    image_files = [
        file for file in files
        if Path(file).suffix.lower() in image_extensions
    ]

    image_count = len(image_files)
    total_images += image_count

    relative_path = root_path.relative_to(dataset_path)
    display_path = str(relative_path) if str(relative_path) != "." else dataset_path.name

    folder_results.append((display_path, image_count, len(files), len(dirs)))

print("FOLDER STRUCTURE AND IMAGE COUNT")
print("-" * 80)

for folder, image_count, file_count, subfolder_count in folder_results:
    print(f"Folder     : {folder}")
    print(f"Subfolders : {subfolder_count}")
    print(f"All files  : {file_count}")
    print(f"Images     : {image_count}")
    print("-" * 80)

print(f"\nTotal number of images in the dataset: {total_images}")

In [ ]:
# Install once in Kaggle
!pip install -q ImageHash

import hashlib
import shutil
from collections import defaultdict
from pathlib import Path

import imagehash
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True


# ============================================================
# SETTINGS
# ============================================================

SOURCE_DIR = Path(
    "/kaggle/input/datasets/javaidahmadwani/lc25000/"
    "lung_colon_image_set/Train and Validation Set"
)

OUTPUT_DIR = Path(
    "/kaggle/working/LC25000_Deduplicated"
)

REPORT_DIR = Path(
    "/kaggle/working/LC25000_Duplicate_Reports"
)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

# Perceptual-hash Hamming distance.
# 0 means visually identical; 4–6 is a reasonable near-duplicate range.
NEAR_DUPLICATE_THRESHOLD = 5

# True: compare near duplicates across the entire dataset.
# False: compare only within the same class folder.
COMPARE_NEAR_DUPLICATES_ACROSS_CLASSES = True

# Remove old output before creating the cleaned dataset.
RESET_OUTPUT = True


# ============================================================
# INITIALIZATION
# ============================================================

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Dataset not found:\n{SOURCE_DIR}")

if RESET_OUTPUT:
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    if REPORT_DIR.exists():
        shutil.rmtree(REPORT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

image_paths = sorted([
    path for path in SOURCE_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
])

print("Dataset path:", SOURCE_DIR)
print("Total images found:", len(image_paths))


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def calculate_sha256(image_path, chunk_size=1024 * 1024):
    """Calculate exact file-content hash."""

    sha256 = hashlib.sha256()

    with open(image_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


def calculate_pixel_hash(image_path):
    """
    Hash decoded RGB pixel values.

    This detects identical images saved with different JPEG
    compression levels or metadata only when decoded pixels match.
    """

    with Image.open(image_path) as image:
        image = image.convert("RGB")

        sha256 = hashlib.sha256()
        sha256.update(str(image.size).encode())
        sha256.update(image.tobytes())

        return sha256.hexdigest()


def calculate_perceptual_hash(image_path):
    """Calculate a 64-bit perceptual hash."""

    with Image.open(image_path) as image:
        image = image.convert("RGB")
        return imagehash.phash(image, hash_size=8)


def get_class_name(image_path):
    """
    Uses the immediate parent directory as the class name.

    Example:
    train/lung_aca/image.jpg -> lung_aca
    """

    return image_path.parent.name


def copy_unique_image(source_path):
    """Copy an image while preserving its relative folder structure."""

    relative_path = source_path.relative_to(SOURCE_DIR)
    destination_path = OUTPUT_DIR / relative_path

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        source_path,
        destination_path
    )

    return destination_path


def get_hash_chunks(hash_integer, chunks=8, bits_per_chunk=8):
    """
    Split a 64-bit perceptual hash into 8 chunks.

    Candidate search using chunks prevents comparison of every
    image against every other image.
    """

    masks = (1 << bits_per_chunk) - 1

    return [
        (hash_integer >> (index * bits_per_chunk)) & masks
        for index in range(chunks)
    ]


# ============================================================
# DUPLICATE DETECTION
# ============================================================

exact_file_hashes = {}
exact_pixel_hashes = {}

# Each bucket stores retained image indices that share one pHash chunk.
phash_buckets = defaultdict(list)

retained_images = []
duplicate_records = []
corrupted_records = []

counts = {
    "total": len(image_paths),
    "unique": 0,
    "exact_file_duplicates": 0,
    "exact_pixel_duplicates": 0,
    "near_duplicates": 0,
    "corrupted": 0
}


for image_path in tqdm(
    image_paths,
    desc="Finding and removing duplicates"
):
    relative_path = image_path.relative_to(SOURCE_DIR)
    class_name = get_class_name(image_path)

    try:
        # ----------------------------------------------------
        # 1. Exact duplicate by file bytes
        # ----------------------------------------------------

        file_hash = calculate_sha256(image_path)

        if file_hash in exact_file_hashes:
            original = exact_file_hashes[file_hash]

            duplicate_records.append({
                "Duplicate File": str(relative_path),
                "Retained File": str(
                    original["path"].relative_to(SOURCE_DIR)
                ),
                "Duplicate Type": "Exact file duplicate",
                "Hash Distance": 0,
                "Duplicate Class": class_name,
                "Retained Class": original["class_name"]
            })

            counts["exact_file_duplicates"] += 1
            continue

        # ----------------------------------------------------
        # 2. Exact decoded-pixel duplicate
        # ----------------------------------------------------

        pixel_hash = calculate_pixel_hash(image_path)

        if pixel_hash in exact_pixel_hashes:
            original = exact_pixel_hashes[pixel_hash]

            duplicate_records.append({
                "Duplicate File": str(relative_path),
                "Retained File": str(
                    original["path"].relative_to(SOURCE_DIR)
                ),
                "Duplicate Type": "Exact pixel duplicate",
                "Hash Distance": 0,
                "Duplicate Class": class_name,
                "Retained Class": original["class_name"]
            })

            counts["exact_pixel_duplicates"] += 1
            continue

        # ----------------------------------------------------
        # 3. Near duplicate using perceptual hash
        # ----------------------------------------------------

        perceptual_hash = calculate_perceptual_hash(image_path)
        hash_integer = int(str(perceptual_hash), 16)

        hash_chunks = get_hash_chunks(hash_integer)

        # Get possible matches from shared hash chunks.
        candidate_indices = set()

        for chunk_position, chunk_value in enumerate(hash_chunks):
            bucket_key = (
                chunk_position,
                chunk_value
            )

            candidate_indices.update(
                phash_buckets[bucket_key]
            )

        best_match = None
        best_distance = float("inf")

        for candidate_index in candidate_indices:
            candidate = retained_images[candidate_index]

            if (
                not COMPARE_NEAR_DUPLICATES_ACROSS_CLASSES
                and candidate["class_name"] != class_name
            ):
                continue

            distance = (
                perceptual_hash -
                candidate["perceptual_hash"]
            )

            if distance < best_distance:
                best_distance = distance
                best_match = candidate

        if (
            best_match is not None
            and best_distance <= NEAR_DUPLICATE_THRESHOLD
        ):
            duplicate_records.append({
                "Duplicate File": str(relative_path),
                "Retained File": str(
                    best_match["path"].relative_to(SOURCE_DIR)
                ),
                "Duplicate Type": "Near duplicate",
                "Hash Distance": int(best_distance),
                "Duplicate Class": class_name,
                "Retained Class": best_match["class_name"]
            })

            counts["near_duplicates"] += 1
            continue

        # ----------------------------------------------------
        # 4. Retain unique image
        # ----------------------------------------------------

        output_path = copy_unique_image(image_path)

        retained_index = len(retained_images)

        retained_images.append({
            "path": image_path,
            "output_path": output_path,
            "class_name": class_name,
            "file_hash": file_hash,
            "pixel_hash": pixel_hash,
            "perceptual_hash": perceptual_hash
        })

        exact_file_hashes[file_hash] = retained_images[-1]
        exact_pixel_hashes[pixel_hash] = retained_images[-1]

        for chunk_position, chunk_value in enumerate(hash_chunks):
            bucket_key = (
                chunk_position,
                chunk_value
            )

            phash_buckets[bucket_key].append(
                retained_index
            )

        counts["unique"] += 1

    except Exception as error:
        counts["corrupted"] += 1

        corrupted_records.append({
            "File": str(relative_path),
            "Error": str(error)
        })


# ============================================================
# SAVE REPORTS
# ============================================================

duplicate_columns = [
    "Duplicate File",
    "Retained File",
    "Duplicate Type",
    "Hash Distance",
    "Duplicate Class",
    "Retained Class"
]

duplicate_df = pd.DataFrame(
    duplicate_records,
    columns=duplicate_columns
)

corrupted_df = pd.DataFrame(
    corrupted_records,
    columns=["File", "Error"]
)

duplicate_df.to_csv(
    REPORT_DIR / "duplicate_images.csv",
    index=False
)

corrupted_df.to_csv(
    REPORT_DIR / "corrupted_images.csv",
    index=False
)


# ============================================================
# CLASS-WISE COUNTS
# ============================================================

original_class_counts = defaultdict(int)
cleaned_class_counts = defaultdict(int)

for path in image_paths:
    original_class_counts[get_class_name(path)] += 1

for item in retained_images:
    cleaned_class_counts[item["class_name"]] += 1

class_rows = []

all_classes = sorted(
    set(original_class_counts) |
    set(cleaned_class_counts)
)

for class_name in all_classes:
    original_count = original_class_counts[class_name]
    cleaned_count = cleaned_class_counts[class_name]

    class_rows.append({
        "Class": class_name,
        "Original Images": original_count,
        "Cleaned Images": cleaned_count,
        "Removed Images": original_count - cleaned_count
    })

class_count_df = pd.DataFrame(class_rows)

class_count_df.to_csv(
    REPORT_DIR / "class_wise_counts.csv",
    index=False
)


# ============================================================
# CROSS-CLASS DUPLICATE REPORT
# ============================================================

if not duplicate_df.empty:
    cross_class_df = duplicate_df[
        duplicate_df["Duplicate Class"] !=
        duplicate_df["Retained Class"]
    ].copy()
else:
    cross_class_df = pd.DataFrame(
        columns=duplicate_columns
    )

cross_class_df.to_csv(
    REPORT_DIR / "cross_class_duplicates.csv",
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

removed_total = (
    counts["exact_file_duplicates"]
    + counts["exact_pixel_duplicates"]
    + counts["near_duplicates"]
)

summary_df = pd.DataFrame([{
    "Original Images": counts["total"],
    "Unique Images Retained": counts["unique"],
    "Exact File Duplicates Removed":
        counts["exact_file_duplicates"],
    "Exact Pixel Duplicates Removed":
        counts["exact_pixel_duplicates"],
    "Near Duplicates Removed":
        counts["near_duplicates"],
    "Total Duplicates Removed": removed_total,
    "Corrupted Images Skipped": counts["corrupted"],
    "Cross-Class Duplicates":
        len(cross_class_df),
    "Near-Duplicate Threshold":
        NEAR_DUPLICATE_THRESHOLD
}])

summary_df.to_csv(
    REPORT_DIR / "deduplication_summary.csv",
    index=False
)

print("\n" + "=" * 70)
print("DEDUPLICATION COMPLETED")
print("=" * 70)

display(summary_df)
display(class_count_df)

print("\nCleaned dataset:")
print(OUTPUT_DIR)

print("\nReports:")
print(REPORT_DIR)

print("\nDuplicate report:")
print(REPORT_DIR / "duplicate_images.csv")

print("\nCross-class duplicate report:")
print(REPORT_DIR / "cross_class_duplicates.csv")

In [ ]:
import os
import random
import shutil

source_dir = "/kaggle/working/LC25000_Deduplicated"
output_dir = "/kaggle/working/LC25000_15Percent"

keep_ratio = 0.15
random.seed(42)

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

os.makedirs(output_dir, exist_ok=True)

for class_name in sorted(os.listdir(source_dir)):
    class_path = os.path.join(source_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    images = [
        file for file in os.listdir(class_path)
        if file.lower().endswith(image_extensions)
    ]

    random.shuffle(images)

    keep_count = max(1, round(len(images) * keep_ratio))
    selected_images = images[:keep_count]

    output_class_path = os.path.join(output_dir, class_name)
    os.makedirs(output_class_path, exist_ok=True)

    for image_name in selected_images:
        shutil.copy2(
            os.path.join(class_path, image_name),
            os.path.join(output_class_path, image_name)
        )

    print(
        f"{class_name}: Original = {len(images)}, "
        f"Selected = {len(selected_images)}"
    )

print(f"\n15% dataset saved at: {output_dir}")

In [ ]:
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm


# ============================================================
# PATHS AND SETTINGS
# ============================================================

input_dir = Path("/kaggle/working/LC25000_15Percent")
output_dir = Path("/kaggle/working/LC25000_Preprocessed1")

IMG_SIZE = 224

extensions = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}


# ============================================================
# HISTOPATHOLOGY PREPROCESSING
# ============================================================

def preprocess_histopathology_image(path):
    """
    Preprocessing suitable for RGB histopathology images:

    1. Read RGB tissue image
    2. Resize to 224 × 224
    3. Apply mild bilateral denoising
    4. Apply CLAHE only to luminance channel
    5. Apply mild gamma correction
    6. Preserve original stain/color information
    """

    img = cv2.imread(
        str(path),
        cv2.IMREAD_COLOR
    )

    if img is None:
        return None

    # ----------------------------------------
    # 1. Resize
    # ----------------------------------------

    img = cv2.resize(
        img,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_AREA
    )

    # ----------------------------------------
    # 2. Mild edge-preserving denoising
    # ----------------------------------------

    img = cv2.bilateralFilter(
        img,
        d=5,
        sigmaColor=25,
        sigmaSpace=25
    )

    # ----------------------------------------
    # 3. CLAHE on luminance only
    # ----------------------------------------

    lab = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2LAB
    )

    lightness, channel_a, channel_b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=1.8,
        tileGridSize=(8, 8)
    )

    enhanced_lightness = clahe.apply(
        lightness
    )

    enhanced_lab = cv2.merge([
        enhanced_lightness,
        channel_a,
        channel_b
    ])

    enhanced = cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )

    # ----------------------------------------
    # 4. Mild gamma correction
    # ----------------------------------------

    gamma = 1.05

    lookup_table = np.array([
        ((pixel / 255.0) ** (1.0 / gamma)) * 255
        for pixel in np.arange(256)
    ]).astype(np.uint8)

    processed = cv2.LUT(
        enhanced,
        lookup_table
    )

    return np.clip(
        processed,
        0,
        255
    ).astype(np.uint8)


# ============================================================
# FIND ALL IMAGES RECURSIVELY
# ============================================================

image_files = sorted([
    path
    for path in input_dir.rglob("*")
    if path.is_file()
    and path.suffix.lower() in extensions
])

print("Total images found:", len(image_files))


# ============================================================
# PREPROCESS AND SAVE
# ============================================================

saved_count = 0
failed_count = 0

for image_path in tqdm(
    image_files,
    desc="Preprocessing histopathology images"
):
    processed = preprocess_histopathology_image(
        image_path
    )

    if processed is None:
        failed_count += 1
        continue

    # Preserve original directory structure
    relative_path = image_path.relative_to(
        input_dir
    )

    save_path = (
        output_dir /
        relative_path.parent /
        f"{image_path.stem}.png"
    )

    save_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    success = cv2.imwrite(
        str(save_path),
        processed
    )

    if success:
        saved_count += 1
    else:
        failed_count += 1


print("\nPreprocessing completed.")
print("Images found:", len(image_files))
print("Images saved:", saved_count)
print("Failed images:", failed_count)
print("Output directory:", output_dir)


# ============================================================
# IDENTIFY CLASS FOLDERS
# ============================================================

# A class folder is treated as a folder that directly contains images.
class_folders = []

for folder in sorted([
    path
    for path in input_dir.rglob("*")
    if path.is_dir()
]):
    folder_images = [
        path
        for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in extensions
    ]

    if folder_images:
        class_folders.append(folder)

print("\nClass folders found:", len(class_folders))

for folder in class_folders:
    print(
        folder.relative_to(input_dir)
    )


# ============================================================
# PLOT ONE SAMPLE FROM EACH CLASS
# ============================================================

if len(class_folders) > 0:

    plt.figure(
        figsize=(12, 4 * len(class_folders))
    )

    for row, class_folder in enumerate(
        class_folders
    ):
        folder_images = [
            path
            for path in class_folder.iterdir()
            if path.is_file()
            and path.suffix.lower() in extensions
        ]

        sample_path = random.choice(
            folder_images
        )

        original = cv2.imread(
            str(sample_path)
        )

        processed = preprocess_histopathology_image(
            sample_path
        )

        original_rgb = cv2.cvtColor(
            original,
            cv2.COLOR_BGR2RGB
        )

        processed_rgb = cv2.cvtColor(
            processed,
            cv2.COLOR_BGR2RGB
        )

        class_name = str(
            class_folder.relative_to(input_dir)
        )

        # Original image
        plt.subplot(
            len(class_folders),
            2,
            row * 2 + 1
        )

        plt.imshow(original_rgb)
        plt.title(
            f"{class_name} — Original"
        )
        plt.axis("off")

        # Preprocessed image
        plt.subplot(
            len(class_folders),
            2,
            row * 2 + 2
        )

        plt.imshow(processed_rgb)
        plt.title(
            f"{class_name} — Preprocessed"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

else:
    print(
        "No class folders containing images were found."
    )

In [ ]:
import os
import random
import shutil
from pathlib import Path

# Paths
source_dir = Path("/kaggle/working/LC25000_Preprocessed1")
output_dir = Path("/kaggle/working/PVMD_Split")

# Split ratios
train_ratio = 0.80
val_ratio = 0.10
test_ratio = 0.10

# Reproducibility
random.seed(42)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Remove old output folder if it exists
if output_dir.exists():
    shutil.rmtree(output_dir)

for class_dir in source_dir.iterdir():
    if not class_dir.is_dir():
        continue

    images = [
        file for file in class_dir.iterdir()
        if file.is_file() and file.suffix.lower() in image_extensions
    ]

    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    splits = {
        "train": images[:train_end],
        "validation": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split_name, split_images in splits.items():
        destination = output_dir / split_name / class_dir.name
        destination.mkdir(parents=True, exist_ok=True)

        for image_path in split_images:
            shutil.copy2(image_path, destination / image_path.name)

    print(
        f"{class_dir.name}: "
        f"Train={len(splits['train'])}, "
        f"Validation={len(splits['validation'])}, "
        f"Test={len(splits['test'])}"
    )

print(f"\nSplit dataset saved at: {output_dir}")

In [ ]:
import shutil
from pathlib import Path

import numpy as np
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Input training folder
train_path = Path("/kaggle/working/PVMD_Split/train")

# Output augmented training folder
output_path = Path("/kaggle/working/PVMD_Train_Augmented")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Remove previous output
if output_path.exists():
    shutil.rmtree(output_path)

# Augmentation settings
augmenter = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

for class_folder in sorted(train_path.iterdir()):

    if not class_folder.is_dir():
        continue

    destination = output_path / class_folder.name
    destination.mkdir(parents=True, exist_ok=True)

    image_files = [
        file for file in class_folder.iterdir()
        if file.is_file() and file.suffix.lower() in image_extensions
    ]

    for image_file in image_files:

        try:
            # Read image
            image = Image.open(image_file).convert("RGB")
            image_array = np.array(image)

            # Save original image
            original_name = f"{image_file.stem}_original.jpg"
            image.save(destination / original_name, quality=95)

            # Add batch dimension
            image_batch = np.expand_dims(image_array, axis=0)

            # Generate exactly 2 augmented images
            augmentation_generator = augmenter.flow(
                image_batch,
                batch_size=1,
                shuffle=False
            )

            for augmentation_number in range(1, 3):
                augmented_image = next(augmentation_generator)[0]
                augmented_image = np.clip(
                    augmented_image, 0, 255
                ).astype(np.uint8)

                augmented_image = Image.fromarray(augmented_image)

                augmented_name = (
                    f"{image_file.stem}_aug_{augmentation_number}.jpg"
                )

                augmented_image.save(
                    destination / augmented_name,
                    quality=95
                )

        except Exception as error:
            print(f"Error processing {image_file}: {error}")

    final_count = len([
        file for file in destination.iterdir()
        if file.suffix.lower() in image_extensions
    ])

    print(
        f"{class_folder.name}: "
        f"{len(image_files)} original × 3 = {final_count} images"
    )

print(f"\nAugmented training dataset saved at:\n{output_path}")

In [ ]:


import contextlib
import copy
import gc
import json
import math
import platform
import os
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path

try:
    import timm
except ImportError:
    print("timm is not installed. Installing timm...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "timm"]
    )
    import timm

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    hamming_loss,
    jaccard_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

TRAIN_DIR = Path("/kaggle/working/PVMD_Train_Augmented")
VAL_DIR = Path("/kaggle/working/PVMD_Split/validation")
TEST_DIR = Path("/kaggle/working/PVMD_Split/test")

if not VAL_DIR.exists():
    alternative_val = Path("/kaggle/working/PVMD_Split/val")
    if alternative_val.exists():
        VAL_DIR = alternative_val

OUTPUT_DIR = Path("/kaggle/working/MaxViT_Tiny_GRFM_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_maxvit_tiny_grfm.pth"
FINAL_MODEL_PATH = OUTPUT_DIR / "final_maxvit_tiny_grfm.pth"

MODEL_NAME = "maxvit_tiny_tf_224.in1k"
MODEL_DISPLAY_NAME = "MaxViT-Tiny-224 + GRFM"

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

EPOCHS = 35
LEARNING_RATE = 1e-4
BACKBONE_LR = 2e-5
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 7
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
MIN_LR = 1e-7

LABEL_SMOOTHING = 0.10
GRADIENT_CLIP = 1.0

GRFM_DIM = 512
GRFM_DROPOUT = 0.30
GRFM_REDUCTION = 4

PRETRAINED = True
ECE_BINS = 15
BOOTSTRAP_N = 1000
TEMPERATURE_MAX_ITER = 100

# Complexity and efficiency profiling.
PROFILE_FLOPS = True
INFERENCE_WARMUP_RUNS = 20
INFERENCE_BENCHMARK_RUNS = 100
INFERENCE_BATCH_SIZES = (1, BATCH_SIZE)
CONFIDENCE_BINS = 10
TOP_MISCLASSIFICATIONS_TO_SAVE = 100

# For binary classification, set the exact positive class folder name.
# Leave as None to use class index 1.
POSITIVE_CLASS_NAME = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


# ============================================================
# 2. GENERAL UTILITIES
# ============================================================

def set_seed(seed: int) -> None:
    """Set random seeds for reproducible training."""
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # benchmark=False is more reproducible than benchmark=True.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def validate_directories() -> None:
    missing = [
        str(path)
        for path in (TRAIN_DIR, VAL_DIR, TEST_DIR)
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "The following dataset folders were not found:\n- "
            + "\n- ".join(missing)
        )


def amp_context():
    """Return an AMP context on CUDA and a no-op context on CPU."""
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        )
    return contextlib.nullcontext()


def print_dataframe(
    dataframe: pd.DataFrame,
    title: str,
    max_rows: int | None = None,
) -> None:
    print(f"\n{title}")
    print("-" * len(title))

    shown = dataframe if max_rows is None else dataframe.head(max_rows)
    print(shown.to_string(index=False))


def make_json_serializable(value):
    """Recursively convert NumPy, Path and non-finite values for JSON."""
    if isinstance(value, dict):
        return {
            str(key): make_json_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [make_json_serializable(item) for item in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (np.floating, np.integer)):
        value = value.item()

    if isinstance(value, float) and not np.isfinite(value):
        return None

    return value


def save_json(data: dict, path: Path) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(
            make_json_serializable(data),
            file,
            indent=2,
        )


# ============================================================
# 3. DATA TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomVerticalFlip(p=0.10),
    transforms.RandomRotation(degrees=10),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05,
        hue=0.02,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),

    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.08),
        ratio=(0.5, 2.0),
    ),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# ============================================================
# 4. DATASETS AND LOADERS
# ============================================================

def create_datasets():
    train_dataset = datasets.ImageFolder(
        root=str(TRAIN_DIR),
        transform=train_transform,
    )

    # Stable, non-random training-set evaluation.
    train_evaluation_dataset = datasets.ImageFolder(
        root=str(TRAIN_DIR),
        transform=evaluation_transform,
    )

    validation_dataset = datasets.ImageFolder(
        root=str(VAL_DIR),
        transform=evaluation_transform,
    )

    test_dataset = datasets.ImageFolder(
        root=str(TEST_DIR),
        transform=evaluation_transform,
    )

    class_names = train_dataset.classes
    class_to_idx = train_dataset.class_to_idx

    for split_name, dataset in [
        ("training-evaluation", train_evaluation_dataset),
        ("validation", validation_dataset),
        ("test", test_dataset),
    ]:
        if dataset.classes != class_names:
            raise ValueError(
                f"{split_name} classes {dataset.classes} do not match "
                f"training classes {class_names}."
            )

    return (
        train_dataset,
        train_evaluation_dataset,
        validation_dataset,
        test_dataset,
        class_names,
        class_to_idx,
    )


def create_loaders(
    train_dataset,
    train_evaluation_dataset,
    validation_dataset,
    test_dataset,
):
    common = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": NUM_WORKERS > 0,
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        **common,
    )

    train_evaluation_loader = DataLoader(
        train_evaluation_dataset,
        shuffle=False,
        **common,
    )

    validation_loader = DataLoader(
        validation_dataset,
        shuffle=False,
        **common,
    )

    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        **common,
    )

    return (
        train_loader,
        train_evaluation_loader,
        validation_loader,
        test_loader,
    )


# ============================================================
# 5. GRFM MODULE
# ============================================================

class GRFM(nn.Module):
    """
    Gated Residual Feature Modulation.

    Input:
        Global MaxViT feature vector [B, C].

    Output:
        Gated residual representation [B, output_dim].
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int = 512,
        reduction: int = 4,
        dropout: float = 0.30,
    ):
        super().__init__()

        hidden_dim = max(output_dim // reduction, 32)

        self.projection = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

        self.feature_transform = nn.Sequential(
            nn.Linear(output_dim, output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, output_dim),
        )

        self.feature_gate = nn.Sequential(
            nn.Linear(output_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

        self.output_norm = nn.LayerNorm(output_dim)
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x, return_gate: bool = False):
        projected_feature = self.projection(x)
        transformed_feature = self.feature_transform(projected_feature)
        modulation_gate = self.feature_gate(projected_feature)

        modulated_feature = transformed_feature * modulation_gate
        fused_feature = projected_feature + modulated_feature
        fused_feature = self.output_norm(fused_feature)
        fused_feature = self.output_dropout(fused_feature)

        if return_gate:
            return fused_feature, modulation_gate

        return fused_feature


# ============================================================
# 6. MAXVIT-TINY + GRFM MODEL
# ============================================================

class MaxViTTinyGRFM(nn.Module):

    def __init__(
        self,
        num_classes: int,
        grfm_dim: int = 512,
        dropout: float = 0.30,
        pretrained: bool = True,
    ):
        super().__init__()

        try:
            self.backbone = timm.create_model(
                MODEL_NAME,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as error:
            if not pretrained:
                raise

            print(
                "\nWarning: pretrained MaxViT weights could not be loaded.\n"
                f"Reason: {error}\n"
                "Falling back to pretrained=False."
            )

            self.backbone = timm.create_model(
                MODEL_NAME,
                pretrained=False,
                num_classes=0,
                global_pool="avg",
            )

        backbone_feature_dim = self.backbone.num_features

        self.grfm = GRFM(
            input_dim=backbone_feature_dim,
            output_dim=grfm_dim,
            reduction=GRFM_REDUCTION,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(grfm_dim, grfm_dim // 2),
            nn.LayerNorm(grfm_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(grfm_dim // 2, num_classes),
        )

        self.backbone_feature_dim = backbone_feature_dim

    def forward(
        self,
        images,
        return_features: bool = False,
        return_gate: bool = False,
    ):
        backbone_features = self.backbone(images)

        grfm_features, gate = self.grfm(
            backbone_features,
            return_gate=True,
        )

        logits = self.classifier(grfm_features)

        if return_features or return_gate:
            return {
                "logits": logits,
                "backbone_features": backbone_features,
                "grfm_features": grfm_features,
                "gate": gate,
            }

        return logits


# ============================================================
# 7. MODEL, LOSS, OPTIMIZER AND SCHEDULER
# ============================================================

def create_model_and_training_objects(num_classes: int):
    model = MaxViTTinyGRFM(
        num_classes=num_classes,
        grfm_dim=GRFM_DIM,
        dropout=GRFM_DROPOUT,
        pretrained=PRETRAINED,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING,
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.backbone.parameters(),
                "lr": BACKBONE_LR,
            },
            {
                "params": model.grfm.parameters(),
                "lr": LEARNING_RATE,
            },
            {
                "params": model.classifier.parameters(),
                "lr": LEARNING_RATE,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MIN_LR,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    return model, criterion, optimizer, scheduler, scaler


def print_parameter_summary(model: nn.Module) -> None:
    components = {
        "Total": sum(p.numel() for p in model.parameters()),
        "Trainable": sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        ),
        "Backbone": sum(
            p.numel()
            for p in model.backbone.parameters()
        ),
        "GRFM": sum(
            p.numel()
            for p in model.grfm.parameters()
        ),
        "Classifier": sum(
            p.numel()
            for p in model.classifier.parameters()
        ),
    }

    print("\nModel parameter summary")
    print("-----------------------")
    for name, count in components.items():
        print(f"{name:12s}: {count:,}")


# ============================================================
# 8. MODEL COMPLEXITY, MEMORY AND SPEED PROFILING
# ============================================================

def count_model_parameters(model: nn.Module) -> dict:
    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )
    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    total_buffers = sum(
        buffer.numel()
        for buffer in model.buffers()
    )

    parameter_bytes = sum(
        parameter.numel() * parameter.element_size()
        for parameter in model.parameters()
    )
    buffer_bytes = sum(
        buffer.numel() * buffer.element_size()
        for buffer in model.buffers()
    )

    return {
        "Total Parameters": int(total_parameters),
        "Trainable Parameters": int(trainable_parameters),
        "Non-Trainable Parameters": int(
            total_parameters - trainable_parameters
        ),
        "Total Buffers": int(total_buffers),
        "Model State Size MB": float(
            (parameter_bytes + buffer_bytes) / (1024 ** 2)
        ),
    }


def synchronize_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


THOP_BUFFER_NAMES = {"total_ops", "total_params"}


def remove_thop_buffers_from_model(model: nn.Module) -> None:
    """Remove profiling-only buffers that THOP may register in-place."""
    for module in model.modules():
        for buffer_name in tuple(module._buffers.keys()):
            if buffer_name in THOP_BUFFER_NAMES:
                module._buffers.pop(buffer_name, None)


def clean_checkpoint_state_dict(state_or_model) -> dict:
    """
    Return a clean CPU state dictionary.

    This removes THOP-only ``total_ops`` and ``total_params`` entries and
    clones every tensor outside inference mode. It therefore supports both
    newly created checkpoints and older checkpoints contaminated by THOP.
    """
    if isinstance(state_or_model, nn.Module):
        raw_state = state_or_model.state_dict()
    else:
        raw_state = state_or_model

    cleaned_state = {}

    for key, value in raw_state.items():
        final_name = key.rsplit(".", 1)[-1]

        if final_name in THOP_BUFFER_NAMES:
            continue

        if torch.is_tensor(value):
            # clone() converts inference tensors into ordinary tensors and
            # moving to CPU keeps checkpoints portable and saves GPU memory.
            cleaned_state[key] = value.detach().cpu().clone()
        else:
            cleaned_state[key] = copy.deepcopy(value)

    return cleaned_state


def load_clean_model_state(
    model: nn.Module,
    checkpoint_state: dict,
) -> None:
    """Load a normal or THOP-contaminated checkpoint safely."""
    remove_thop_buffers_from_model(model)
    cleaned_state = clean_checkpoint_state_dict(checkpoint_state)

    incompatible = model.load_state_dict(
        cleaned_state,
        strict=False,
    )

    meaningful_missing = [
        key
        for key in incompatible.missing_keys
        if key.rsplit(".", 1)[-1] not in THOP_BUFFER_NAMES
    ]

    meaningful_unexpected = [
        key
        for key in incompatible.unexpected_keys
        if key.rsplit(".", 1)[-1] not in THOP_BUFFER_NAMES
    ]

    if meaningful_missing or meaningful_unexpected:
        raise RuntimeError(
            "Checkpoint does not match the model architecture.\n"
            f"Missing keys: {meaningful_missing}\n"
            f"Unexpected keys: {meaningful_unexpected}"
        )


def profile_model_complexity(model: nn.Module) -> dict:
    """
    Estimate MACs and FLOPs without modifying the trainable model.

    THOP registers ``total_ops`` and ``total_params`` buffers in-place. To
    prevent those buffers from entering training checkpoints, all profiling
    is performed on a deep copy that is deleted immediately afterward.
    """
    summary = count_model_parameters(model)
    summary.update({
        "Input Shape": [1, 3, IMG_SIZE, IMG_SIZE],
        "MACs": np.nan,
        "MACs GMAC": np.nan,
        "Estimated FLOPs": np.nan,
        "Estimated FLOPs GFLOPs": np.nan,
        "Profiling Method": "Not run",
        "Profiling Note": (
            "FLOPs are estimated as 2 x MACs. Profiling uses a "
            "separate model copy so THOP buffers cannot contaminate "
            "training checkpoints."
        ),
    })

    if not PROFILE_FLOPS:
        summary["Profiling Method"] = "Disabled"
        return summary

    profiling_model = None
    dummy_input = None

    try:
        # IMPORTANT: never pass the trainable model directly to THOP.
        profiling_model = copy.deepcopy(model).to(DEVICE)
        remove_thop_buffers_from_model(profiling_model)
        profiling_model.eval()

        dummy_input = torch.randn(
            1,
            3,
            IMG_SIZE,
            IMG_SIZE,
            device=DEVICE,
        )

        try:
            from thop import profile as thop_profile
        except ImportError:
            print("\nthop is not installed. Installing thop...")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "thop"]
            )
            from thop import profile as thop_profile

        # Use no_grad rather than inference_mode because THOP registers and
        # updates profiling buffers during the forward pass.
        with torch.no_grad():
            macs, thop_parameters = thop_profile(
                profiling_model,
                inputs=(dummy_input,),
                verbose=False,
            )

        estimated_flops = 2.0 * float(macs)

        summary.update({
            "THOP Parameters": int(thop_parameters),
            "MACs": float(macs),
            "MACs GMAC": float(macs / 1e9),
            "Estimated FLOPs": estimated_flops,
            "Estimated FLOPs GFLOPs": float(estimated_flops / 1e9),
            "Profiling Method": "THOP on isolated model copy",
        })

    except Exception as error:
        thop_error = str(error)
        summary["THOP Error"] = thop_error

        print(f"\nWarning: THOP profiling failed: {thop_error}")
        print("Trying torch.profiler FLOPs estimation on a fresh copy...")

        # Delete any copy modified by a failed THOP run before fallback.
        if profiling_model is not None:
            del profiling_model
            profiling_model = None

        if dummy_input is not None:
            del dummy_input
            dummy_input = None

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        try:
            profiling_model = copy.deepcopy(model).to(DEVICE)
            remove_thop_buffers_from_model(profiling_model)
            profiling_model.eval()

            dummy_input = torch.randn(
                1,
                3,
                IMG_SIZE,
                IMG_SIZE,
                device=DEVICE,
            )

            activities = [torch.profiler.ProfilerActivity.CPU]
            if DEVICE.type == "cuda":
                activities.append(torch.profiler.ProfilerActivity.CUDA)

            with torch.no_grad():
                with torch.profiler.profile(
                    activities=activities,
                    record_shapes=False,
                    profile_memory=False,
                    with_flops=True,
                ) as profiler:
                    _ = profiling_model(dummy_input)

            estimated_flops = float(
                sum(event.flops or 0 for event in profiler.key_averages())
            )

            if estimated_flops <= 0:
                raise RuntimeError(
                    "torch.profiler returned zero counted FLOPs."
                )

            estimated_macs = estimated_flops / 2.0

            summary.update({
                "MACs": estimated_macs,
                "MACs GMAC": float(estimated_macs / 1e9),
                "Estimated FLOPs": estimated_flops,
                "Estimated FLOPs GFLOPs": float(estimated_flops / 1e9),
                "Profiling Method": "torch.profiler on isolated model copy",
            })

        except Exception as fallback_error:
            summary["Profiling Method"] = "Profiling failed"
            summary["Profiler Error"] = str(fallback_error)

            print(
                "Warning: torch.profiler FLOPs estimation also failed: "
                f"{fallback_error}"
            )

    finally:
        if profiling_model is not None:
            remove_thop_buffers_from_model(profiling_model)
            del profiling_model

        if dummy_input is not None:
            del dummy_input

        # Defensive cleanup in case a third-party profiler unexpectedly
        # touched the real model.
        remove_thop_buffers_from_model(model)

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    save_json(
        summary,
        OUTPUT_DIR / "model_complexity.json",
    )

    pd.DataFrame(
        list(summary.items()),
        columns=["Measure", "Value"],
    ).to_csv(
        OUTPUT_DIR / "model_complexity.csv",
        index=False,
    )

    print("\nModel complexity")
    print("----------------")
    print(f"Parameters       : {summary['Total Parameters']:,}")
    print(
        "Estimated size   : "
        f"{summary['Model State Size MB']:.3f} MB"
    )

    if np.isfinite(summary.get("MACs GMAC", np.nan)):
        print(f"MACs             : {summary['MACs GMAC']:.4f} GMAC")
        print(
            "Estimated FLOPs : "
            f"{summary['Estimated FLOPs GFLOPs']:.4f} GFLOPs"
        )
    else:
        print("MACs / FLOPs     : unavailable")

    return summary


@torch.inference_mode()
def benchmark_forward_latency(
    model: nn.Module,
    batch_size: int,
    warmup_runs: int = INFERENCE_WARMUP_RUNS,
    benchmark_runs: int = INFERENCE_BENCHMARK_RUNS,
) -> dict:
    """Measure model-only forward-pass latency using synthetic input."""
    model.eval()

    dummy_input = torch.randn(
        batch_size,
        3,
        IMG_SIZE,
        IMG_SIZE,
        device=DEVICE,
    )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    for _ in range(warmup_runs):
        with amp_context():
            _ = model(dummy_input)

    synchronize_device()
    latency_ms = []

    if DEVICE.type == "cuda":
        for _ in range(benchmark_runs):
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)

            start_event.record()
            with amp_context():
                _ = model(dummy_input)
            end_event.record()

            torch.cuda.synchronize()
            latency_ms.append(float(start_event.elapsed_time(end_event)))
    else:
        for _ in range(benchmark_runs):
            start_time = time.perf_counter()
            with amp_context():
                _ = model(dummy_input)
            latency_ms.append(
                float((time.perf_counter() - start_time) * 1000.0)
            )

    latency_array = np.asarray(latency_ms, dtype=np.float64)
    mean_batch_latency_ms = float(latency_array.mean())
    mean_image_latency_ms = float(
        mean_batch_latency_ms / batch_size
    )
    throughput = float(
        batch_size / (mean_batch_latency_ms / 1000.0)
    )

    result = {
        "Device": str(DEVICE),
        "AMP Enabled": bool(USE_AMP),
        "Batch Size": int(batch_size),
        "Warm-up Runs": int(warmup_runs),
        "Measured Runs": int(benchmark_runs),
        "Mean Batch Latency ms": mean_batch_latency_ms,
        "Median Batch Latency ms": float(
            np.median(latency_array)
        ),
        "Std Batch Latency ms": float(latency_array.std(ddof=0)),
        "P95 Batch Latency ms": float(
            np.percentile(latency_array, 95)
        ),
        "Mean Per-Image Latency ms": mean_image_latency_ms,
        "Model Throughput Images per Second": throughput,
    }

    if DEVICE.type == "cuda":
        result["Peak Inference Memory Allocated MB"] = float(
            torch.cuda.max_memory_allocated() / (1024 ** 2)
        )
        result["Peak Inference Memory Reserved MB"] = float(
            torch.cuda.max_memory_reserved() / (1024 ** 2)
        )

    del dummy_input
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result


@torch.inference_mode()
def benchmark_dataloader_inference(
    model: nn.Module,
    loader: DataLoader,
) -> dict:
    """
    Measure full test-loader inference, including loading, transfer and forward.
    """
    model.eval()
    total_images = 0

    # Start persistent workers and warm up the accelerator once.
    for warmup_images, _ in loader:
        warmup_images = warmup_images.to(
            DEVICE,
            non_blocking=True,
        )
        with amp_context():
            _ = model(warmup_images)
        break

    synchronize_device()
    start_time = time.perf_counter()

    for images, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        with amp_context():
            _ = model(images)
        total_images += images.size(0)

    synchronize_device()
    total_seconds = time.perf_counter() - start_time

    return {
        "Test Images": int(total_images),
        "End-to-End Test Inference Seconds": float(total_seconds),
        "End-to-End Per-Image Latency ms": float(
            1000.0 * total_seconds / max(total_images, 1)
        ),
        "End-to-End Throughput Images per Second": float(
            total_images / max(total_seconds, 1e-12)
        ),
    }


def benchmark_model_inference(
    model: nn.Module,
    test_loader: DataLoader,
) -> tuple[pd.DataFrame, dict]:
    rows = []

    for batch_size in sorted(set(INFERENCE_BATCH_SIZES)):
        try:
            rows.append(
                benchmark_forward_latency(
                    model=model,
                    batch_size=int(batch_size),
                )
            )
        except RuntimeError as error:
            if "out of memory" in str(error).lower():
                print(
                    f"\nWarning: latency benchmark skipped for "
                    f"batch size {batch_size} because of GPU OOM."
                )
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
            else:
                raise

    latency_dataframe = pd.DataFrame(rows)
    latency_dataframe.to_csv(
        OUTPUT_DIR / "model_forward_latency.csv",
        index=False,
    )

    end_to_end_summary = benchmark_dataloader_inference(
        model,
        test_loader,
    )
    save_json(
        end_to_end_summary,
        OUTPUT_DIR / "test_inference_efficiency.json",
    )

    print_dataframe(
        latency_dataframe.round(6),
        "Model-only inference benchmark",
    )
    print("\nEnd-to-end test inference")
    print("-------------------------")
    for key, value in end_to_end_summary.items():
        if isinstance(value, float):
            print(f"{key}: {value:.6f}")
        else:
            print(f"{key}: {value}")

    return latency_dataframe, end_to_end_summary


def collect_environment_information() -> dict:
    information = {
        "Python Version": platform.python_version(),
        "Platform": platform.platform(),
        "PyTorch Version": torch.__version__,
        "Torchvision Version": getattr(
            sys.modules.get("torchvision"),
            "__version__",
            "unknown",
        ),
        "timm Version": getattr(timm, "__version__", "unknown"),
        "Device": str(DEVICE),
        "CUDA Available": bool(torch.cuda.is_available()),
        "CUDA Version": torch.version.cuda,
        "cuDNN Version": (
            torch.backends.cudnn.version()
            if torch.cuda.is_available()
            else None
        ),
    }

    if torch.cuda.is_available():
        information["GPU Name"] = torch.cuda.get_device_name(0)
        properties = torch.cuda.get_device_properties(0)
        information["GPU Total Memory GB"] = float(
            properties.total_memory / (1024 ** 3)
        )

    save_json(
        information,
        OUTPUT_DIR / "environment_information.json",
    )
    return information


# ============================================================
# 9. TRAINING AND BASIC EVALUATION FUNCTIONS
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
):
    model.train()

    running_loss = 0.0
    labels_all = []
    predictions_all = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp_context():
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP,
        )

        scaler.step(optimizer)
        scaler.update()

        predictions = logits.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        labels_all.extend(labels.detach().cpu().numpy())
        predictions_all.extend(predictions.detach().cpu().numpy())

    labels_all = np.asarray(labels_all)
    predictions_all = np.asarray(predictions_all)

    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels_all, predictions_all),
        "balanced_accuracy": balanced_accuracy_score(
            labels_all,
            predictions_all,
        ),
        "macro_f1": f1_score(
            labels_all,
            predictions_all,
            average="macro",
            zero_division=0,
        ),
    }


@torch.inference_mode()
def evaluate_logits(
    model,
    loader,
    criterion=None,
):
    model.eval()

    running_loss = 0.0
    logits_all = []
    labels_all = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels_device = labels.to(DEVICE, non_blocking=True)

        with amp_context():
            logits = model(images)

            if criterion is not None:
                loss = criterion(logits, labels_device)
                running_loss += loss.item() * images.size(0)

        logits_all.append(logits.float().cpu())
        labels_all.extend(labels.numpy())

    logits_all = torch.cat(logits_all, dim=0)
    labels_all = np.asarray(labels_all, dtype=np.int64)

    average_loss = (
        running_loss / len(loader.dataset)
        if criterion is not None
        else np.nan
    )

    return average_loss, logits_all, labels_all


def probabilities_and_predictions(logits: torch.Tensor):
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()
    predictions = np.argmax(probabilities, axis=1)
    return probabilities, predictions


# ============================================================
# 9. TRAINING LOOP AND CHECKPOINTING
# ============================================================

def train_model(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    class_names,
    class_to_idx,
):
    history = []

    best_validation_f1 = -np.inf
    best_validation_loss = np.inf
    epochs_without_improvement = 0

    start_time = time.time()

    print("\nTraining started")
    print("=" * 132)

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()

        train_results = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
        )

        (
            validation_loss,
            validation_logits,
            validation_labels,
        ) = evaluate_logits(
            model=model,
            loader=validation_loader,
            criterion=criterion,
        )

        (
            validation_probabilities,
            validation_predictions,
        ) = probabilities_and_predictions(validation_logits)

        validation_accuracy = accuracy_score(
            validation_labels,
            validation_predictions,
        )

        validation_balanced_accuracy = balanced_accuracy_score(
            validation_labels,
            validation_predictions,
        )

        validation_macro_f1 = f1_score(
            validation_labels,
            validation_predictions,
            average="macro",
            zero_division=0,
        )

        scheduler.step(validation_macro_f1)

        backbone_lr = optimizer.param_groups[0]["lr"]
        head_lr = optimizer.param_groups[1]["lr"]
        epoch_seconds = time.time() - epoch_start

        history.append({
            "Epoch": epoch,
            "Backbone LR": backbone_lr,
            "GRFM Head LR": head_lr,
            "Train Loss": train_results["loss"],
            "Train Accuracy": train_results["accuracy"],
            "Train Balanced Accuracy":
                train_results["balanced_accuracy"],
            "Train Macro F1": train_results["macro_f1"],
            "Validation Loss": validation_loss,
            "Validation Accuracy": validation_accuracy,
            "Validation Balanced Accuracy":
                validation_balanced_accuracy,
            "Validation Macro F1": validation_macro_f1,
            "Epoch Time Seconds": epoch_seconds,
        })

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train loss {train_results['loss']:.5f} | "
            f"Train acc {train_results['accuracy']:.5f} | "
            f"Train F1 {train_results['macro_f1']:.5f} | "
            f"Val loss {validation_loss:.5f} | "
            f"Val acc {validation_accuracy:.5f} | "
            f"Val F1 {validation_macro_f1:.5f} | "
            f"Backbone LR {backbone_lr:.2e} | "
            f"Head LR {head_lr:.2e} | "
            f"{epoch_seconds:.1f}s"
        )

        improved = (
            validation_macro_f1 > best_validation_f1 + 1e-5
            or (
                abs(validation_macro_f1 - best_validation_f1) <= 1e-5
                and validation_loss < best_validation_loss
            )
        )

        if improved:
            best_validation_f1 = validation_macro_f1
            best_validation_loss = validation_loss
            epochs_without_improvement = 0

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": clean_checkpoint_state_dict(model),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_validation_macro_f1":
                        best_validation_f1,
                    "best_validation_loss":
                        best_validation_loss,
                    "class_names": class_names,
                    "class_to_idx": class_to_idx,
                    "configuration": {
                        "model_name": MODEL_NAME,
                        "image_size": IMG_SIZE,
                        "grfm_dim": GRFM_DIM,
                        "grfm_dropout": GRFM_DROPOUT,
                        "grfm_reduction": GRFM_REDUCTION,
                        "num_classes": len(class_names),
                        "batch_size": BATCH_SIZE,
                        "epochs": EPOCHS,
                        "backbone_lr": BACKBONE_LR,
                        "head_lr": LEARNING_RATE,
                        "weight_decay": WEIGHT_DECAY,
                        "label_smoothing": LABEL_SMOOTHING,
                        "seed": SEED,
                    },
                },
                BEST_MODEL_PATH,
            )

            print("  Best checkpoint saved.")
        else:
            epochs_without_improvement += 1
            print(
                "  No improvement: "
                f"{epochs_without_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping activated.")
            break

    total_training_seconds = time.time() - start_time
    history_df = pd.DataFrame(history)

    history_df.to_csv(
        OUTPUT_DIR / "training_history.csv",
        index=False,
    )

    print(
        f"\nTraining completed in "
        f"{total_training_seconds / 60:.2f} minutes."
    )

    return history_df, total_training_seconds


# ============================================================
# 10. TRAINING CURVES
# ============================================================

def save_training_curves(history_df: pd.DataFrame) -> None:
    curve_specs = [
        (
            "Train Loss",
            "Validation Loss",
            "Loss",
            "MaxViT-Tiny-GRFM Loss",
            "loss_curve.png",
        ),
        (
            "Train Accuracy",
            "Validation Accuracy",
            "Accuracy",
            "MaxViT-Tiny-GRFM Accuracy",
            "accuracy_curve.png",
        ),
        (
            "Train Macro F1",
            "Validation Macro F1",
            "Macro F1-score",
            "MaxViT-Tiny-GRFM Macro F1",
            "macro_f1_curve.png",
        ),
    ]

    for train_column, validation_column, ylabel, title, filename in curve_specs:
        plt.figure(figsize=(8, 5))

        plt.plot(
            history_df["Epoch"],
            history_df[train_column],
            marker="o",
            label=train_column,
        )

        plt.plot(
            history_df["Epoch"],
            history_df[validation_column],
            marker="s",
            label=validation_column,
        )

        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / filename, dpi=300)
        plt.close()


# ============================================================
# 11. TEMPERATURE SCALING
# ============================================================

class TemperatureScaler(nn.Module):

    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self):
        return torch.exp(self.log_temperature).clamp(
            min=1e-3,
            max=100.0,
        )

    def forward(self, logits):
        return logits / self.temperature


def fit_temperature_scaler(
    validation_logits: torch.Tensor,
    validation_labels: np.ndarray,
):
    scaler = TemperatureScaler().to(DEVICE)

    logits_device = validation_logits.to(DEVICE)
    labels_device = torch.as_tensor(
        validation_labels,
        dtype=torch.long,
        device=DEVICE,
    )

    nll_criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.LBFGS(
        [scaler.log_temperature],
        lr=0.05,
        max_iter=TEMPERATURE_MAX_ITER,
        line_search_fn="strong_wolfe",
    )

    before_nll = nll_criterion(
        logits_device,
        labels_device,
    ).item()

    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = nll_criterion(
            scaler(logits_device),
            labels_device,
        )
        loss.backward()
        return loss

    try:
        optimizer.step(closure)
    except RuntimeError as error:
        print(
            "\nWarning: temperature optimization did not fully converge:"
            f" {error}"
        )

    with torch.no_grad():
        after_nll = nll_criterion(
            scaler(logits_device),
            labels_device,
        ).item()

    temperature = float(scaler.temperature.detach().cpu().item())

    summary = {
        "Validation NLL Before": before_nll,
        "Validation NLL After": after_nll,
        "Temperature": temperature,
    }

    save_json(
        summary,
        OUTPUT_DIR / "temperature_scaling.json",
    )

    print("\nTemperature scaling")
    print("-------------------")
    print(f"NLL before : {before_nll:.6f}")
    print(f"NLL after  : {after_nll:.6f}")
    print(f"Temperature: {temperature:.6f}")

    return scaler


@torch.inference_mode()
def calibrate_logits(
    logits: torch.Tensor,
    scaler: TemperatureScaler,
):
    return scaler(logits.to(DEVICE)).cpu()


# ============================================================
# 12. CALIBRATION METRICS
# ============================================================

def one_hot(labels: np.ndarray, number_of_classes: int):
    encoded = np.zeros(
        (len(labels), number_of_classes),
        dtype=np.float64,
    )
    encoded[np.arange(len(labels)), labels] = 1.0
    return encoded


def multiclass_brier_score(labels, probabilities):
    targets = one_hot(labels, probabilities.shape[1])
    return float(
        np.mean(
            np.sum(
                (probabilities - targets) ** 2,
                axis=1,
            )
        )
    )


def compute_ece_mce_rmsce(
    probabilities,
    labels,
    number_of_bins=15,
):
    confidence = np.max(probabilities, axis=1)
    predictions = np.argmax(probabilities, axis=1)
    correct = (predictions == labels).astype(float)

    boundaries = np.linspace(0.0, 1.0, number_of_bins + 1)

    ece = 0.0
    mce = 0.0
    rmsce = 0.0
    records = []

    for index in range(number_of_bins):
        lower = boundaries[index]
        upper = boundaries[index + 1]

        if index == number_of_bins - 1:
            mask = (confidence >= lower) & (confidence <= upper)
        else:
            mask = (confidence >= lower) & (confidence < upper)

        count = int(mask.sum())
        weight = count / len(labels)

        if count > 0:
            bin_accuracy = float(correct[mask].mean())
            bin_confidence = float(confidence[mask].mean())
            gap = abs(bin_accuracy - bin_confidence)

            ece += weight * gap
            rmsce += weight * gap ** 2
            mce = max(mce, gap)
        else:
            bin_accuracy = np.nan
            bin_confidence = np.nan
            gap = np.nan

        records.append({
            "Bin Lower": lower,
            "Bin Upper": upper,
            "Count": count,
            "Weight": weight,
            "Accuracy": bin_accuracy,
            "Confidence": bin_confidence,
            "Gap": gap,
        })

    return (
        float(ece),
        float(mce),
        float(np.sqrt(rmsce)),
        pd.DataFrame(records),
    )


def compute_binary_ece_mce_rmsce(
    labels,
    probabilities,
    positive_index,
    number_of_bins=15,
):
    binary_labels = (labels == positive_index).astype(int)
    positive_scores = probabilities[:, positive_index]

    boundaries = np.linspace(0.0, 1.0, number_of_bins + 1)

    ece = 0.0
    mce = 0.0
    rmsce = 0.0
    records = []

    for index in range(number_of_bins):
        lower = boundaries[index]
        upper = boundaries[index + 1]

        if index == number_of_bins - 1:
            mask = (
                (positive_scores >= lower)
                & (positive_scores <= upper)
            )
        else:
            mask = (
                (positive_scores >= lower)
                & (positive_scores < upper)
            )

        count = int(mask.sum())
        weight = count / len(labels)

        if count > 0:
            observed_rate = float(binary_labels[mask].mean())
            predicted_rate = float(positive_scores[mask].mean())
            gap = abs(observed_rate - predicted_rate)

            ece += weight * gap
            rmsce += weight * gap ** 2
            mce = max(mce, gap)
        else:
            observed_rate = np.nan
            predicted_rate = np.nan
            gap = np.nan

        records.append({
            "Bin Lower": lower,
            "Bin Upper": upper,
            "Count": count,
            "Weight": weight,
            "Observed Positive Rate": observed_rate,
            "Mean Predicted Probability": predicted_rate,
            "Calibration Gap": gap,
        })

    return (
        float(ece),
        float(mce),
        float(np.sqrt(rmsce)),
        pd.DataFrame(records),
    )


def adaptive_calibration_error(
    labels,
    probabilities,
    positive_index=None,
    number_of_bins=15,
):
    if probabilities.shape[1] == 2 and positive_index is not None:
        outcomes = (labels == positive_index).astype(float)
        scores = probabilities[:, positive_index]
    else:
        scores = np.max(probabilities, axis=1)
        predictions = np.argmax(probabilities, axis=1)
        outcomes = (predictions == labels).astype(float)

    ordered_indices = np.argsort(scores)
    groups = np.array_split(ordered_indices, number_of_bins)

    weighted_gaps = []

    for indices in groups:
        if len(indices) == 0:
            continue

        observed = float(outcomes[indices].mean())
        expected = float(scores[indices].mean())
        weight = len(indices) / len(labels)

        weighted_gaps.append(
            weight * abs(observed - expected)
        )

    return float(np.sum(weighted_gaps))


# ============================================================
# 13. COMPREHENSIVE METRICS
# ============================================================

def safe_divide(numerator, denominator):
    if denominator == 0:
        return np.nan
    return float(numerator / denominator)


def per_class_specificity(confusion):
    values = []

    for class_index in range(confusion.shape[0]):
        true_positive = confusion[class_index, class_index]
        false_positive = (
            confusion[:, class_index].sum() - true_positive
        )
        false_negative = (
            confusion[class_index, :].sum() - true_positive
        )
        true_negative = (
            confusion.sum()
            - true_positive
            - false_positive
            - false_negative
        )

        values.append(
            safe_divide(
                true_negative,
                true_negative + false_positive,
            )
        )

    return np.asarray(values, dtype=float)


def probability_diagnostics(
    labels: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
) -> dict:
    """Return confidence, uncertainty and ranking diagnostics."""
    number_of_classes = probabilities.shape[1]
    row_indices = np.arange(len(labels))

    confidence = np.max(probabilities, axis=1)
    true_class_probability = probabilities[row_indices, labels]

    sorted_probabilities = np.sort(probabilities, axis=1)
    if number_of_classes >= 2:
        second_highest = sorted_probabilities[:, -2]
    else:
        second_highest = np.zeros_like(confidence)

    confidence_margin = confidence - second_highest

    clipped_probabilities = np.clip(
        probabilities,
        1e-12,
        1.0,
    )
    entropy = -np.sum(
        clipped_probabilities * np.log(clipped_probabilities),
        axis=1,
    )
    if number_of_classes > 1:
        normalized_entropy = entropy / np.log(number_of_classes)
    else:
        normalized_entropy = np.zeros_like(entropy)

    correct_mask = predictions == labels
    incorrect_mask = ~correct_mask

    return {
        "confidence": confidence,
        "true_class_probability": true_class_probability,
        "second_highest_probability": second_highest,
        "confidence_margin": confidence_margin,
        "normalized_entropy": normalized_entropy,
        "correct_mask": correct_mask,
        "incorrect_mask": incorrect_mask,
    }


def compute_metric_dict(
    labels,
    predictions,
    probabilities,
    class_names,
    positive_index=None,
    number_of_bins=15,
    binary_threshold=None,
):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = np.asarray(predictions, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if probabilities.ndim != 2:
        raise ValueError("probabilities must be a 2D array.")

    number_of_classes = probabilities.shape[1]

    if len(labels) != len(predictions) or len(labels) != len(probabilities):
        raise ValueError("labels, predictions and probabilities must align.")

    if len(labels) == 0:
        raise ValueError("The evaluation split is empty.")

    if len(class_names) != number_of_classes:
        raise ValueError(
            "The number of class names must equal probability columns."
        )

    if np.any(labels < 0) or np.any(labels >= number_of_classes):
        raise ValueError("labels contain an invalid class index.")

    # Optional binary thresholding.
    if (
        number_of_classes == 2
        and positive_index is not None
        and binary_threshold is not None
    ):
        negative_index = 1 - positive_index
        positive_prediction = (
            probabilities[:, positive_index] >= binary_threshold
        )
        predictions = np.where(
            positive_prediction,
            positive_index,
            negative_index,
        )

    diagnostics = probability_diagnostics(
        labels=labels,
        predictions=predictions,
        probabilities=probabilities,
    )

    confidence = diagnostics["confidence"]
    true_class_probability = diagnostics[
        "true_class_probability"
    ]
    confidence_margin = diagnostics["confidence_margin"]
    normalized_entropy = diagnostics["normalized_entropy"]
    correct_mask = diagnostics["correct_mask"]
    incorrect_mask = diagnostics["incorrect_mask"]

    confusion = confusion_matrix(
        labels,
        predictions,
        labels=np.arange(number_of_classes),
    )

    specificity_by_class = per_class_specificity(confusion)

    metric = {
        "N": int(len(labels)),
        "Accuracy": accuracy_score(labels, predictions),
        "Balanced Accuracy": balanced_accuracy_score(
            labels,
            predictions,
        ),
        "Macro Precision": precision_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "Macro Recall / Sensitivity": recall_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "Macro Specificity": float(
            np.nanmean(specificity_by_class)
        ),
        "Macro F1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "Weighted Precision": precision_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "Weighted Recall": recall_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "Weighted F1": f1_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "Micro F1": f1_score(
            labels,
            predictions,
            average="micro",
            zero_division=0,
        ),
        "MCC": (
            matthews_corrcoef(labels, predictions)
            if len(np.unique(labels)) > 1
            else np.nan
        ),
        "Cohen Kappa": (
            cohen_kappa_score(labels, predictions)
            if len(np.unique(labels)) > 1
            else np.nan
        ),
        "Hamming Loss": hamming_loss(labels, predictions),
        "Jaccard Macro": jaccard_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "Multiclass Brier Score": multiclass_brier_score(
            labels,
            probabilities,
        ),
        "Mean Confidence": float(np.mean(confidence)),
        "Median Confidence": float(np.median(confidence)),
        "Mean True-Class Probability": float(
            np.mean(true_class_probability)
        ),
        "Mean Confidence Margin": float(
            np.mean(confidence_margin)
        ),
        "Mean Normalized Predictive Entropy": float(
            np.mean(normalized_entropy)
        ),
        "Mean Confidence Correct Predictions": (
            float(np.mean(confidence[correct_mask]))
            if np.any(correct_mask)
            else np.nan
        ),
        "Mean Confidence Incorrect Predictions": (
            float(np.mean(confidence[incorrect_mask]))
            if np.any(incorrect_mask)
            else np.nan
        ),
    }

    if number_of_classes > 2:
        sorted_indices = np.argsort(
            probabilities,
            axis=1,
        )[:, ::-1]

        for top_k in range(2, min(5, number_of_classes) + 1):
            top_k_correct = np.any(
                sorted_indices[:, :top_k] == labels[:, None],
                axis=1,
            )
            metric[f"Top-{top_k} Accuracy"] = float(
                np.mean(top_k_correct)
            )

    try:
        metric["Log Loss / NLL"] = log_loss(
            labels,
            probabilities,
            labels=list(range(number_of_classes)),
        )
    except ValueError:
        metric["Log Loss / NLL"] = np.nan

    if (
        np.any(correct_mask)
        and np.any(incorrect_mask)
    ):
        metric["Correct-Incorrect Confidence Gap"] = float(
            np.mean(confidence[correct_mask])
            - np.mean(confidence[incorrect_mask])
        )
    else:
        metric["Correct-Incorrect Confidence Gap"] = np.nan

    ece, mce, rmsce, calibration_bins = compute_ece_mce_rmsce(
        probabilities,
        labels,
        number_of_bins=number_of_bins,
    )

    metric["ECE"] = ece
    metric["MCE"] = mce
    metric["RMSCE"] = rmsce
    metric["ACE"] = adaptive_calibration_error(
        labels,
        probabilities,
        positive_index=positive_index,
        number_of_bins=number_of_bins,
    )

    binary_calibration_bins = None

    if number_of_classes == 2 and positive_index is not None:
        negative_index = 1 - positive_index

        true_binary = (labels == positive_index).astype(int)
        predicted_binary = (
            predictions == positive_index
        ).astype(int)

        binary_confusion = confusion_matrix(
            true_binary,
            predicted_binary,
            labels=[0, 1],
        )

        true_negative, false_positive, false_negative, true_positive = (
            binary_confusion.ravel()
        )

        sensitivity = safe_divide(
            true_positive,
            true_positive + false_negative,
        )

        specificity = safe_divide(
            true_negative,
            true_negative + false_positive,
        )

        ppv = safe_divide(
            true_positive,
            true_positive + false_positive,
        )

        npv = safe_divide(
            true_negative,
            true_negative + false_negative,
        )

        fpr = safe_divide(
            false_positive,
            false_positive + true_negative,
        )

        fnr = safe_divide(
            false_negative,
            false_negative + true_positive,
        )

        fdr = safe_divide(
            false_positive,
            false_positive + true_positive,
        )

        false_omission_rate = safe_divide(
            false_negative,
            false_negative + true_negative,
        )

        lr_positive = (
            safe_divide(sensitivity, 1.0 - specificity)
            if np.isfinite(sensitivity)
            and np.isfinite(specificity)
            else np.nan
        )

        lr_negative = (
            safe_divide(1.0 - sensitivity, specificity)
            if np.isfinite(sensitivity)
            and np.isfinite(specificity)
            else np.nan
        )

        metric.update({
            "Positive Class": class_names[positive_index],
            "Negative Class": class_names[negative_index],
            "Threshold": (
                0.5
                if binary_threshold is None
                else float(binary_threshold)
            ),
            "Sensitivity / Recall": sensitivity,
            "Specificity": specificity,
            "Precision / PPV": ppv,
            "NPV": npv,
            "Binary F1": f1_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            ),
            "F0.5-score": fbeta_score(
                true_binary,
                predicted_binary,
                beta=0.5,
                zero_division=0,
            ),
            "F2-score": fbeta_score(
                true_binary,
                predicted_binary,
                beta=2.0,
                zero_division=0,
            ),
            "G-Mean": (
                float(np.sqrt(sensitivity * specificity))
                if np.isfinite(sensitivity)
                and np.isfinite(specificity)
                else np.nan
            ),
            "FPR": fpr,
            "FNR": fnr,
            "FDR": fdr,
            "False Omission Rate": false_omission_rate,
            "Youden Index": (
                sensitivity + specificity - 1.0
                if np.isfinite(sensitivity)
                and np.isfinite(specificity)
                else np.nan
            ),
            "Markedness": (
                ppv + npv - 1.0
                if np.isfinite(ppv) and np.isfinite(npv)
                else np.nan
            ),
            "LR+": lr_positive,
            "LR-": lr_negative,
            "Diagnostic Odds Ratio": (
                safe_divide(lr_positive, lr_negative)
                if np.isfinite(lr_positive)
                and np.isfinite(lr_negative)
                else np.nan
            ),
            "TP": int(true_positive),
            "TN": int(true_negative),
            "FP": int(false_positive),
            "FN": int(false_negative),
        })

        positive_scores = probabilities[:, positive_index]

        try:
            metric["ROC-AUC"] = roc_auc_score(
                true_binary,
                positive_scores,
            )
        except ValueError:
            metric["ROC-AUC"] = np.nan

        try:
            metric["PR-AUC / Average Precision"] = (
                average_precision_score(
                    true_binary,
                    positive_scores,
                )
            )
        except ValueError:
            metric["PR-AUC / Average Precision"] = np.nan

        try:
            metric["Binary Brier Score"] = brier_score_loss(
                true_binary,
                positive_scores,
            )
        except ValueError:
            metric["Binary Brier Score"] = np.nan

        (
            binary_ece,
            binary_mce,
            binary_rmsce,
            binary_calibration_bins,
        ) = compute_binary_ece_mce_rmsce(
            labels,
            probabilities,
            positive_index=positive_index,
            number_of_bins=number_of_bins,
        )

        metric["Binary ECE"] = binary_ece
        metric["Binary MCE"] = binary_mce
        metric["Binary RMSCE"] = binary_rmsce
        metric["Binary ACE"] = adaptive_calibration_error(
            labels,
            probabilities,
            positive_index=positive_index,
            number_of_bins=number_of_bins,
        )

    else:
        targets = one_hot(labels, number_of_classes)

        try:
            metric["Macro ROC-AUC OVR"] = roc_auc_score(
                labels,
                probabilities,
                average="macro",
                multi_class="ovr",
            )
        except ValueError:
            metric["Macro ROC-AUC OVR"] = np.nan

        try:
            metric["Weighted ROC-AUC OVR"] = roc_auc_score(
                labels,
                probabilities,
                average="weighted",
                multi_class="ovr",
            )
        except ValueError:
            metric["Weighted ROC-AUC OVR"] = np.nan

        try:
            metric["Micro ROC-AUC OVR"] = roc_auc_score(
                targets,
                probabilities,
                average="micro",
                multi_class="ovr",
            )
        except ValueError:
            metric["Micro ROC-AUC OVR"] = np.nan

        try:
            metric["Macro PR-AUC"] = average_precision_score(
                targets,
                probabilities,
                average="macro",
            )
        except ValueError:
            metric["Macro PR-AUC"] = np.nan

        try:
            metric["Weighted PR-AUC"] = average_precision_score(
                targets,
                probabilities,
                average="weighted",
            )
        except ValueError:
            metric["Weighted PR-AUC"] = np.nan

        try:
            metric["Micro PR-AUC"] = average_precision_score(
                targets,
                probabilities,
                average="micro",
            )
        except ValueError:
            metric["Micro PR-AUC"] = np.nan

    per_class_rows = []

    for class_index, class_name in enumerate(class_names):
        true_binary = (labels == class_index).astype(int)
        predicted_binary = (
            predictions == class_index
        ).astype(int)

        try:
            class_roc_auc = roc_auc_score(
                true_binary,
                probabilities[:, class_index],
            )
        except ValueError:
            class_roc_auc = np.nan

        try:
            class_pr_auc = average_precision_score(
                true_binary,
                probabilities[:, class_index],
            )
        except ValueError:
            class_pr_auc = np.nan

        class_confusion = confusion_matrix(
            true_binary,
            predicted_binary,
            labels=[0, 1],
        )
        class_tn, class_fp, class_fn, class_tp = (
            class_confusion.ravel()
        )

        class_npv = safe_divide(
            class_tn,
            class_tn + class_fn,
        )
        class_fpr = safe_divide(
            class_fp,
            class_fp + class_tn,
        )
        class_fnr = safe_divide(
            class_fn,
            class_fn + class_tp,
        )

        true_class_mask = labels == class_index
        predicted_class_mask = predictions == class_index
        correct_class_mask = (
            true_class_mask & (predictions == labels)
        )
        incorrect_class_mask = (
            true_class_mask & (predictions != labels)
        )

        per_class_rows.append({
            "Class": class_name,
            "Precision / PPV": precision_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            ),
            "Sensitivity / Recall": recall_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            ),
            "Specificity": specificity_by_class[class_index],
            "NPV": class_npv,
            "F1-score": f1_score(
                true_binary,
                predicted_binary,
                zero_division=0,
            ),
            "FPR": class_fpr,
            "FNR": class_fnr,
            "ROC-AUC": class_roc_auc,
            "PR-AUC": class_pr_auc,
            "Support": int(np.sum(true_class_mask)),
            "Predicted Count": int(np.sum(predicted_class_mask)),
            "Correct Count": int(np.sum(correct_class_mask)),
            "Mean Max Confidence": float(
                np.mean(confidence[true_class_mask])
            ),
            "Mean True-Class Probability": float(
                np.mean(
                    probabilities[
                        true_class_mask,
                        class_index,
                    ]
                )
            ),
            "Mean Confidence Correct": (
                float(np.mean(confidence[correct_class_mask]))
                if np.any(correct_class_mask)
                else np.nan
            ),
            "Mean Confidence Incorrect": (
                float(np.mean(confidence[incorrect_class_mask]))
                if np.any(incorrect_class_mask)
                else np.nan
            ),
            "Mean Normalized Entropy": float(
                np.mean(normalized_entropy[true_class_mask])
            ),
            "Mean Confidence Margin": float(
                np.mean(confidence_margin[true_class_mask])
            ),
            "TP": int(class_tp),
            "TN": int(class_tn),
            "FP": int(class_fp),
            "FN": int(class_fn),
        })

    per_class_dataframe = pd.DataFrame(per_class_rows)

    return (
        metric,
        confusion,
        calibration_bins,
        binary_calibration_bins,
        per_class_dataframe,
        predictions,
    )


# ============================================================
# 14. THRESHOLD ANALYSIS FOR BINARY CLASSIFICATION
# ============================================================

def threshold_analysis(
    labels,
    probabilities,
    positive_index,
    class_names,
):
    thresholds = np.linspace(0.05, 0.95, 91)
    rows = []

    for threshold in thresholds:
        (
            metric,
            _,
            _,
            _,
            _,
            _,
        ) = compute_metric_dict(
            labels=labels,
            predictions=np.argmax(probabilities, axis=1),
            probabilities=probabilities,
            class_names=class_names,
            positive_index=positive_index,
            number_of_bins=ECE_BINS,
            binary_threshold=float(threshold),
        )

        rows.append({
            "Threshold": float(threshold),
            "Accuracy": metric["Accuracy"],
            "Balanced Accuracy": metric["Balanced Accuracy"],
            "Macro F1": metric["Macro F1"],
            "Sensitivity": metric["Sensitivity / Recall"],
            "Specificity": metric["Specificity"],
            "Precision": metric["Precision / PPV"],
            "Binary F1": metric["Binary F1"],
            "F2": metric["F2-score"],
            "G-Mean": metric["G-Mean"],
            "Youden Index": metric["Youden Index"],
            "MCC": metric["MCC"],
        })

    dataframe = pd.DataFrame(rows)

    best_youden_row = dataframe.loc[
        dataframe["Youden Index"].idxmax()
    ]

    best_f1_row = dataframe.loc[
        dataframe["Binary F1"].idxmax()
    ]

    return (
        dataframe,
        best_youden_row.to_dict(),
        best_f1_row.to_dict(),
    )


def plot_threshold_analysis(dataframe: pd.DataFrame) -> None:
    plt.figure(figsize=(8, 5))

    for column in [
        "Sensitivity",
        "Specificity",
        "Binary F1",
        "Youden Index",
    ]:
        plt.plot(
            dataframe["Threshold"],
            dataframe[column],
            label=column,
        )

    plt.xlabel("Decision threshold")
    plt.ylabel("Metric value")
    plt.title("Validation-Based Threshold Analysis")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "validation_threshold_analysis.png",
        dpi=300,
    )
    plt.close()


# ============================================================
# 15. BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def bootstrap_confidence_intervals(
    labels,
    predictions,
    probabilities,
    class_names,
    positive_index=None,
    binary_threshold=None,
    number_of_bootstraps=1000,
    seed=42,
):
    rng = np.random.default_rng(seed)

    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    probabilities = np.asarray(probabilities)

    number_of_samples = len(labels)

    metric_names = [
        "Accuracy",
        "Balanced Accuracy",
        "Macro Precision",
        "Macro Recall / Sensitivity",
        "Macro Specificity",
        "Macro F1",
        "Weighted F1",
        "MCC",
        "Cohen Kappa",
        "Log Loss / NLL",
        "Multiclass Brier Score",
        "ECE",
        "Mean Confidence",
        "Mean True-Class Probability",
        "Mean Confidence Margin",
        "Mean Normalized Predictive Entropy",
    ]

    if probabilities.shape[1] == 2 and positive_index is not None:
        metric_names.extend([
            "Sensitivity / Recall",
            "Specificity",
            "Precision / PPV",
            "NPV",
            "Binary F1",
            "F2-score",
            "G-Mean",
            "Youden Index",
            "ROC-AUC",
            "PR-AUC / Average Precision",
            "Binary Brier Score",
            "Binary ECE",
        ])
    else:
        metric_names.extend([
            "Macro ROC-AUC OVR",
            "Weighted ROC-AUC OVR",
            "Micro ROC-AUC OVR",
            "Macro PR-AUC",
            "Weighted PR-AUC",
            "Micro PR-AUC",
        ])

    values = {name: [] for name in metric_names}

    for _ in range(number_of_bootstraps):
        indices = rng.choice(
            number_of_samples,
            size=number_of_samples,
            replace=True,
        )

        bootstrap_labels = labels[indices]

        if len(np.unique(bootstrap_labels)) < 2:
            continue

        bootstrap_probabilities = probabilities[indices]
        bootstrap_predictions = predictions[indices]

        try:
            (
                metric,
                _,
                _,
                _,
                _,
                _,
            ) = compute_metric_dict(
                labels=bootstrap_labels,
                predictions=bootstrap_predictions,
                probabilities=bootstrap_probabilities,
                class_names=class_names,
                positive_index=positive_index,
                number_of_bins=ECE_BINS,
                binary_threshold=binary_threshold,
            )
        except Exception:
            continue

        for name in metric_names:
            value = metric.get(name, np.nan)

            if isinstance(value, (int, float, np.number)):
                value = float(value)

                if np.isfinite(value):
                    values[name].append(value)

    rows = []

    for name, metric_values in values.items():
        if metric_values:
            lower, upper = np.percentile(
                metric_values,
                [2.5, 97.5],
            )

            rows.append({
                "Metric": name,
                "Bootstrap Mean": np.mean(metric_values),
                "95% CI Lower": lower,
                "95% CI Upper": upper,
                "Valid Bootstraps": len(metric_values),
            })
        else:
            rows.append({
                "Metric": name,
                "Bootstrap Mean": np.nan,
                "95% CI Lower": np.nan,
                "95% CI Upper": np.nan,
                "Valid Bootstraps": 0,
            })

    return pd.DataFrame(rows)


# ============================================================
# 16. EVALUATION PLOTS
# ============================================================

def plot_confusion(
    confusion,
    class_names,
    filename,
    title,
    normalize=None,
):
    """Plot a count or row-normalized confusion matrix."""
    confusion = np.asarray(confusion)

    if normalize == "true":
        row_totals = confusion.sum(axis=1, keepdims=True)
        displayed_matrix = np.divide(
            confusion,
            row_totals,
            out=np.zeros_like(confusion, dtype=float),
            where=row_totals != 0,
        )
        value_format = ".2%"
    elif normalize is None:
        displayed_matrix = confusion
        value_format = "d"
    else:
        raise ValueError("normalize must be None or 'true'.")

    plt.figure(figsize=(7.5, 6.5))
    plt.imshow(displayed_matrix, interpolation="nearest")
    plt.colorbar()

    positions = np.arange(len(class_names))

    plt.xticks(
        positions,
        class_names,
        rotation=35,
        ha="right",
    )
    plt.yticks(positions, class_names)

    threshold = (
        displayed_matrix.max() / 2.0
        if displayed_matrix.size > 0
        else 0
    )

    for row in range(displayed_matrix.shape[0]):
        for column in range(displayed_matrix.shape[1]):
            value = displayed_matrix[row, column]
            text = (
                format(int(value), value_format)
                if normalize is None
                else format(float(value), value_format)
            )
            plt.text(
                column,
                row,
                text,
                ha="center",
                va="center",
                color=("white" if value > threshold else "black"),
            )

    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300)
    plt.close()

def plot_roc_curves(
    labels,
    probabilities,
    class_names,
    positive_index=None,
):
    number_of_classes = probabilities.shape[1]

    plt.figure(figsize=(8.5, 6.5))

    if number_of_classes == 2 and positive_index is not None:
        binary_labels = (labels == positive_index).astype(int)
        positive_scores = probabilities[:, positive_index]

        false_positive_rate, true_positive_rate, _ = roc_curve(
            binary_labels,
            positive_scores,
        )

        auc_value = auc(
            false_positive_rate,
            true_positive_rate,
        )

        plt.plot(
            false_positive_rate,
            true_positive_rate,
            linewidth=2.5,
            label=(
                f"{class_names[positive_index]}: "
                f"AUC={auc_value:.4f}"
            ),
        )

    else:
        targets = one_hot(labels, number_of_classes)
        valid_fprs = []
        valid_tprs = []

        for class_index, class_name in enumerate(class_names):
            if len(np.unique(targets[:, class_index])) < 2:
                continue

            false_positive_rate, true_positive_rate, _ = roc_curve(
                targets[:, class_index],
                probabilities[:, class_index],
            )

            auc_value = auc(
                false_positive_rate,
                true_positive_rate,
            )

            valid_fprs.append(false_positive_rate)
            valid_tprs.append(true_positive_rate)

            plt.plot(
                false_positive_rate,
                true_positive_rate,
                linewidth=1.8,
                label=f"{class_name}: AUC={auc_value:.4f}",
            )

        # Micro-average ROC.
        micro_fpr, micro_tpr, _ = roc_curve(
            targets.ravel(),
            probabilities.ravel(),
        )
        micro_auc = auc(micro_fpr, micro_tpr)
        plt.plot(
            micro_fpr,
            micro_tpr,
            linestyle=":",
            linewidth=3,
            label=f"Micro-average: AUC={micro_auc:.4f}",
        )

        # Macro-average ROC from interpolated class curves.
        if valid_fprs:
            all_fpr = np.unique(np.concatenate(valid_fprs))
            mean_tpr = np.zeros_like(all_fpr)

            for class_fpr, class_tpr in zip(valid_fprs, valid_tprs):
                mean_tpr += np.interp(
                    all_fpr,
                    class_fpr,
                    class_tpr,
                )

            mean_tpr /= len(valid_fprs)
            macro_auc = auc(all_fpr, mean_tpr)
            plt.plot(
                all_fpr,
                mean_tpr,
                linestyle="--",
                linewidth=3,
                label=f"Macro-average: AUC={macro_auc:.4f}",
            )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Chance",
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{MODEL_DISPLAY_NAME} Test ROC Curves")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "test_roc_curve.png", dpi=300)
    plt.close()

def plot_precision_recall_curves(
    labels,
    probabilities,
    class_names,
    positive_index=None,
):
    number_of_classes = probabilities.shape[1]

    plt.figure(figsize=(8.5, 6.5))

    if number_of_classes == 2 and positive_index is not None:
        binary_labels = (labels == positive_index).astype(int)
        positive_scores = probabilities[:, positive_index]

        precision_values, recall_values, _ = precision_recall_curve(
            binary_labels,
            positive_scores,
        )

        average_precision = average_precision_score(
            binary_labels,
            positive_scores,
        )

        prevalence = float(binary_labels.mean())

        plt.plot(
            recall_values,
            precision_values,
            linewidth=2.5,
            label=(
                f"{class_names[positive_index]}: "
                f"AP={average_precision:.4f}"
            ),
        )
        plt.axhline(
            prevalence,
            linestyle="--",
            linewidth=1.5,
            label=f"No-skill prevalence={prevalence:.4f}",
        )

    else:
        targets = one_hot(labels, number_of_classes)

        for class_index, class_name in enumerate(class_names):
            if len(np.unique(targets[:, class_index])) < 2:
                continue

            precision_values, recall_values, _ = (
                precision_recall_curve(
                    targets[:, class_index],
                    probabilities[:, class_index],
                )
            )

            average_precision = average_precision_score(
                targets[:, class_index],
                probabilities[:, class_index],
            )

            plt.plot(
                recall_values,
                precision_values,
                linewidth=1.8,
                label=f"{class_name}: AP={average_precision:.4f}",
            )

        micro_precision, micro_recall, _ = precision_recall_curve(
            targets.ravel(),
            probabilities.ravel(),
        )
        micro_ap = average_precision_score(
            targets,
            probabilities,
            average="micro",
        )
        plt.plot(
            micro_recall,
            micro_precision,
            linestyle="--",
            linewidth=3,
            label=f"Micro-average: AP={micro_ap:.4f}",
        )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(
        f"{MODEL_DISPLAY_NAME} Test Precision-Recall Curves"
    )
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(loc="lower left", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "test_pr_curve.png", dpi=300)
    plt.close()

def plot_reliability_diagram(
    multiclass_bins,
    binary_bins=None,
    binary_ece=None,
):
    plt.figure(figsize=(7, 6))

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=2,
        label="Perfect calibration",
    )

    if binary_bins is not None:
        valid = binary_bins.dropna(
            subset=[
                "Mean Predicted Probability",
                "Observed Positive Rate",
            ]
        )

        plt.plot(
            valid["Mean Predicted Probability"],
            valid["Observed Positive Rate"],
            marker="o",
            linewidth=2.5,
            label=f"Binary ECE = {binary_ece:.4f}",
        )

        plt.xlabel("Mean predicted positive probability")
        plt.ylabel("Observed positive rate")

    else:
        valid = multiclass_bins.dropna(
            subset=["Confidence", "Accuracy"]
        )

        plt.plot(
            valid["Confidence"],
            valid["Accuracy"],
            marker="o",
            linewidth=2.5,
            label="Calibrated model",
        )

        plt.xlabel("Mean confidence")
        plt.ylabel("Accuracy")

    plt.title(f"{MODEL_DISPLAY_NAME} Reliability Diagram")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "test_reliability_diagram.png",
        dpi=300,
    )
    plt.close()


def plot_confidence_histogram(probabilities):
    confidence = np.max(probabilities, axis=1)

    plt.figure(figsize=(7, 5))
    plt.hist(confidence, bins=20, edgecolor="black", alpha=0.75)
    plt.xlabel("Maximum predicted probability")
    plt.ylabel("Number of test images")
    plt.title(f"{MODEL_DISPLAY_NAME} Confidence Distribution")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "test_confidence_histogram.png",
        dpi=300,
    )
    plt.close()


def build_confidence_analysis(
    labels,
    predictions,
    probabilities,
    class_names,
):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = np.asarray(predictions, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)

    diagnostics = probability_diagnostics(
        labels,
        predictions,
        probabilities,
    )

    confidence = diagnostics["confidence"]
    true_class_probability = diagnostics[
        "true_class_probability"
    ]
    margin = diagnostics["confidence_margin"]
    entropy = diagnostics["normalized_entropy"]

    class_rows = []

    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        correct_mask = mask & (predictions == labels)
        incorrect_mask = mask & (predictions != labels)

        class_rows.append({
            "Class": class_name,
            "Support": int(mask.sum()),
            "Class Accuracy": float(
                np.mean(predictions[mask] == labels[mask])
            ),
            "Mean Max Confidence": float(
                np.mean(confidence[mask])
            ),
            "Std Max Confidence": float(
                np.std(confidence[mask], ddof=0)
            ),
            "Median Max Confidence": float(
                np.median(confidence[mask])
            ),
            "Mean True-Class Probability": float(
                np.mean(true_class_probability[mask])
            ),
            "Mean Confidence Correct": (
                float(np.mean(confidence[correct_mask]))
                if np.any(correct_mask)
                else np.nan
            ),
            "Mean Confidence Incorrect": (
                float(np.mean(confidence[incorrect_mask]))
                if np.any(incorrect_mask)
                else np.nan
            ),
            "Mean Confidence Margin": float(
                np.mean(margin[mask])
            ),
            "Mean Normalized Entropy": float(
                np.mean(entropy[mask])
            ),
        })

    class_confidence_dataframe = pd.DataFrame(class_rows)

    bin_edges = np.linspace(0.0, 1.0, CONFIDENCE_BINS + 1)
    bin_rows = []

    for bin_index in range(CONFIDENCE_BINS):
        lower = bin_edges[bin_index]
        upper = bin_edges[bin_index + 1]

        if bin_index == CONFIDENCE_BINS - 1:
            mask = (confidence >= lower) & (confidence <= upper)
        else:
            mask = (confidence >= lower) & (confidence < upper)

        count = int(mask.sum())
        bin_rows.append({
            "Bin Lower": lower,
            "Bin Upper": upper,
            "Bin Label": f"{lower:.1f}-{upper:.1f}",
            "Count": count,
            "Accuracy": (
                float(np.mean(predictions[mask] == labels[mask]))
                if count > 0
                else np.nan
            ),
            "Mean Confidence": (
                float(np.mean(confidence[mask]))
                if count > 0
                else np.nan
            ),
            "Mean Entropy": (
                float(np.mean(entropy[mask]))
                if count > 0
                else np.nan
            ),
        })

    confidence_bin_dataframe = pd.DataFrame(bin_rows)

    probability_matrix = np.zeros(
        (len(class_names), len(class_names)),
        dtype=np.float64,
    )

    for class_index in range(len(class_names)):
        mask = labels == class_index
        if np.any(mask):
            probability_matrix[class_index] = np.mean(
                probabilities[mask],
                axis=0,
            )

    probability_matrix_dataframe = pd.DataFrame(
        probability_matrix,
        index=[f"True: {name}" for name in class_names],
        columns=[f"P({name})" for name in class_names],
    )

    return (
        class_confidence_dataframe,
        confidence_bin_dataframe,
        probability_matrix_dataframe,
    )


def plot_class_confidence_summary(
    class_confidence_dataframe: pd.DataFrame,
):
    positions = np.arange(len(class_confidence_dataframe))
    width = 0.36

    plt.figure(figsize=(max(8, len(positions) * 1.3), 5.5))
    plt.bar(
        positions - width / 2,
        class_confidence_dataframe["Mean Max Confidence"],
        width=width,
        label="Mean max confidence",
    )
    plt.bar(
        positions + width / 2,
        class_confidence_dataframe[
            "Mean True-Class Probability"
        ],
        width=width,
        label="Mean true-class probability",
    )
    plt.xticks(
        positions,
        class_confidence_dataframe["Class"],
        rotation=35,
        ha="right",
    )
    plt.ylim(0.0, 1.05)
    plt.ylabel("Probability")
    plt.title(f"{MODEL_DISPLAY_NAME} Class Confidence Summary")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "test_class_confidence_summary.png",
        dpi=300,
    )
    plt.close()


def plot_confidence_boxplot(
    labels,
    probabilities,
    class_names,
):
    confidence = np.max(probabilities, axis=1)
    grouped_confidence = [
        confidence[np.asarray(labels) == class_index]
        for class_index in range(len(class_names))
    ]

    plt.figure(figsize=(max(8, len(class_names) * 1.3), 5.5))
    plt.boxplot(
        grouped_confidence,
        labels=class_names,
        showmeans=True,
    )
    plt.xticks(rotation=35, ha="right")
    plt.ylim(0.0, 1.05)
    plt.ylabel("Maximum predicted probability")
    plt.title(f"{MODEL_DISPLAY_NAME} Confidence by True Class")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "test_confidence_boxplot_by_class.png",
        dpi=300,
    )
    plt.close()


def plot_confidence_accuracy_curve(
    confidence_bin_dataframe: pd.DataFrame,
):
    valid = confidence_bin_dataframe.dropna(
        subset=["Accuracy", "Mean Confidence"]
    )

    plt.figure(figsize=(7.5, 5.5))
    plt.plot(
        valid["Mean Confidence"],
        valid["Accuracy"],
        marker="o",
        linewidth=2.2,
        label="Observed accuracy",
    )
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect agreement",
    )
    plt.xlabel("Mean confidence in bin")
    plt.ylabel("Accuracy in bin")
    plt.title(f"{MODEL_DISPLAY_NAME} Confidence vs Accuracy")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "test_confidence_vs_accuracy.png",
        dpi=300,
    )
    plt.close()


def plot_class_probability_heatmap(
    probability_matrix_dataframe: pd.DataFrame,
):
    matrix = probability_matrix_dataframe.to_numpy(dtype=float)

    plt.figure(
        figsize=(
            max(7, matrix.shape[1] * 1.2),
            max(6, matrix.shape[0] * 1.0),
        )
    )
    plt.imshow(matrix, interpolation="nearest", vmin=0.0, vmax=1.0)
    plt.colorbar(label="Mean predicted probability")
    plt.xticks(
        np.arange(matrix.shape[1]),
        probability_matrix_dataframe.columns,
        rotation=35,
        ha="right",
    )
    plt.yticks(
        np.arange(matrix.shape[0]),
        probability_matrix_dataframe.index,
    )

    threshold = 0.5
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix[row, column]
            plt.text(
                column,
                row,
                f"{value:.3f}",
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
            )

    plt.xlabel("Predicted probability column")
    plt.ylabel("True class")
    plt.title(
        f"{MODEL_DISPLAY_NAME} Mean Class-Probability Matrix"
    )
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "test_mean_class_probability_heatmap.png",
        dpi=300,
    )
    plt.close()


def plot_correct_incorrect_confidence(
    labels,
    predictions,
    probabilities,
):
    confidence = np.max(probabilities, axis=1)
    correct_mask = np.asarray(predictions) == np.asarray(labels)

    plt.figure(figsize=(7.5, 5.5))
    if np.any(correct_mask):
        plt.hist(
            confidence[correct_mask],
            bins=20,
            alpha=0.65,
            label="Correct",
        )
    if np.any(~correct_mask):
        plt.hist(
            confidence[~correct_mask],
            bins=20,
            alpha=0.65,
            label="Incorrect",
        )
    plt.xlabel("Maximum predicted probability")
    plt.ylabel("Number of test images")
    plt.title(
        f"{MODEL_DISPLAY_NAME} Correct vs Incorrect Confidence"
    )
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "test_correct_incorrect_confidence.png",
        dpi=300,
    )
    plt.close()


# ============================================================
# 17. SPLIT EVALUATION AND RESULT SAVING
# ============================================================

def evaluate_split(
    split_name,
    labels,
    raw_logits,
    calibrated_logits,
    class_names,
    positive_index,
    sample_paths=None,
):
    raw_probabilities, raw_predictions = probabilities_and_predictions(
        raw_logits
    )

    (
        raw_metrics,
        _,
        _,
        _,
        _,
        raw_predictions,
    ) = compute_metric_dict(
        labels=labels,
        predictions=raw_predictions,
        probabilities=raw_probabilities,
        class_names=class_names,
        positive_index=positive_index,
        number_of_bins=ECE_BINS,
    )

    (
        calibrated_probabilities,
        calibrated_predictions,
    ) = probabilities_and_predictions(calibrated_logits)

    (
        calibrated_metrics,
        calibrated_confusion,
        calibration_bins,
        binary_calibration_bins,
        per_class_dataframe,
        calibrated_predictions,
    ) = compute_metric_dict(
        labels=labels,
        predictions=calibrated_predictions,
        probabilities=calibrated_probabilities,
        class_names=class_names,
        positive_index=positive_index,
        number_of_bins=ECE_BINS,
    )

    raw_metrics_dataframe = pd.DataFrame(
        list(raw_metrics.items()),
        columns=["Metric", "Value"],
    )

    calibrated_metrics_dataframe = pd.DataFrame(
        list(calibrated_metrics.items()),
        columns=["Metric", "Value"],
    )

    raw_metrics_dataframe.to_csv(
        OUTPUT_DIR / f"{split_name.lower()}_raw_metrics.csv",
        index=False,
    )

    calibrated_metrics_dataframe.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_calibrated_metrics.csv",
        index=False,
    )

    per_class_dataframe.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_per_class_metrics.csv",
        index=False,
    )

    classification_dataframe = pd.DataFrame(
        classification_report(
            labels,
            calibrated_predictions,
            labels=np.arange(len(class_names)),
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    classification_dataframe.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_classification_report.csv"
    )

    diagnostics = probability_diagnostics(
        labels=labels,
        predictions=calibrated_predictions,
        probabilities=calibrated_probabilities,
    )

    prediction_data = {
        "True Label": labels,
        "True Class": [class_names[index] for index in labels],
        "Predicted Label": calibrated_predictions,
        "Predicted Class": [
            class_names[index]
            for index in calibrated_predictions
        ],
        "Correct": (
            calibrated_predictions == labels
        ).astype(int),
        "Confidence": diagnostics["confidence"],
        "True-Class Probability": diagnostics[
            "true_class_probability"
        ],
        "Second-Highest Probability": diagnostics[
            "second_highest_probability"
        ],
        "Confidence Margin": diagnostics["confidence_margin"],
        "Normalized Predictive Entropy": diagnostics[
            "normalized_entropy"
        ],
    }

    if sample_paths is not None:
        if len(sample_paths) != len(labels):
            raise ValueError(
                f"{split_name}: sample_paths and labels must align."
            )
        prediction_data = {
            "Image Path": [str(path) for path in sample_paths],
            "Image Name": [Path(path).name for path in sample_paths],
            **prediction_data,
        }

    prediction_dataframe = pd.DataFrame(prediction_data)

    for class_index, class_name in enumerate(class_names):
        prediction_dataframe[
            f"Probability {class_name}"
        ] = calibrated_probabilities[:, class_index]

    prediction_dataframe.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_calibrated_predictions.csv",
        index=False,
    )

    calibration_bins.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_multiclass_calibration_bins.csv",
        index=False,
    )

    if binary_calibration_bins is not None:
        binary_calibration_bins.to_csv(
            OUTPUT_DIR
            / f"{split_name.lower()}_binary_calibration_bins.csv",
            index=False,
        )

    (
        class_confidence_dataframe,
        confidence_bin_dataframe,
        probability_matrix_dataframe,
    ) = build_confidence_analysis(
        labels=labels,
        predictions=calibrated_predictions,
        probabilities=calibrated_probabilities,
        class_names=class_names,
    )

    class_confidence_dataframe.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_class_confidence_summary.csv",
        index=False,
    )
    confidence_bin_dataframe.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_confidence_accuracy_bins.csv",
        index=False,
    )
    probability_matrix_dataframe.to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_mean_class_probability_matrix.csv"
    )

    misclassified_dataframe = prediction_dataframe.loc[
        prediction_dataframe["Correct"] == 0
    ].sort_values(
        by="Confidence",
        ascending=False,
    )
    misclassified_dataframe.head(
        TOP_MISCLASSIFICATIONS_TO_SAVE
    ).to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_highest_confidence_errors.csv",
        index=False,
    )

    lowest_confidence_dataframe = prediction_dataframe.sort_values(
        by="Confidence",
        ascending=True,
    )
    lowest_confidence_dataframe.head(
        TOP_MISCLASSIFICATIONS_TO_SAVE
    ).to_csv(
        OUTPUT_DIR
        / f"{split_name.lower()}_lowest_confidence_predictions.csv",
        index=False,
    )

    return {
        "raw_metrics": raw_metrics,
        "calibrated_metrics": calibrated_metrics,
        "labels": labels,
        "raw_probabilities": raw_probabilities,
        "raw_predictions": raw_predictions,
        "calibrated_probabilities": calibrated_probabilities,
        "calibrated_predictions": calibrated_predictions,
        "confusion": calibrated_confusion,
        "calibration_bins": calibration_bins,
        "binary_calibration_bins": binary_calibration_bins,
        "per_class": per_class_dataframe,
        "classification_report": classification_dataframe,
        "prediction_dataframe": prediction_dataframe,
        "class_confidence": class_confidence_dataframe,
        "confidence_bins": confidence_bin_dataframe,
        "probability_matrix": probability_matrix_dataframe,
    }


# ============================================================
# 18. MAIN PIPELINE
# ============================================================

def main():
    set_seed(SEED)
    validate_directories()
    environment_information = collect_environment_information()

    print("=" * 80)
    print(MODEL_DISPLAY_NAME)
    print("=" * 80)
    print(f"Device        : {DEVICE}")
    print(f"AMP enabled   : {USE_AMP}")
    print(f"Train folder  : {TRAIN_DIR}")
    print(f"Validation    : {VAL_DIR}")
    print(f"Test folder   : {TEST_DIR}")
    print(f"Output folder : {OUTPUT_DIR}")

    (
        train_dataset,
        train_evaluation_dataset,
        validation_dataset,
        test_dataset,
        class_names,
        class_to_idx,
    ) = create_datasets()

    number_of_classes = len(class_names)

    if number_of_classes < 2:
        raise ValueError(
            "At least two classes are required for classification."
        )

    print("\nDataset summary")
    print("---------------")
    print(f"Classes              : {class_names}")
    print(f"Class mapping        : {class_to_idx}")
    print(f"Training images      : {len(train_dataset)}")
    print(f"Validation images    : {len(validation_dataset)}")
    print(f"Test images          : {len(test_dataset)}")

    distribution_rows = []
    for split_name, dataset in [
        ("Train", train_evaluation_dataset),
        ("Validation", validation_dataset),
        ("Test", test_dataset),
    ]:
        targets = np.asarray(dataset.targets, dtype=np.int64)
        for class_index, class_name in enumerate(class_names):
            class_count = int(np.sum(targets == class_index))
            distribution_rows.append({
                "Split": split_name,
                "Class Index": class_index,
                "Class": class_name,
                "Count": class_count,
                "Percentage": float(
                    100.0 * class_count / max(len(dataset), 1)
                ),
            })

    dataset_distribution_dataframe = pd.DataFrame(distribution_rows)
    dataset_distribution_dataframe.to_csv(
        OUTPUT_DIR / "dataset_class_distribution.csv",
        index=False,
    )

    train_sample_paths = [
        sample_path
        for sample_path, _ in train_evaluation_dataset.samples
    ]
    validation_sample_paths = [
        sample_path
        for sample_path, _ in validation_dataset.samples
    ]
    test_sample_paths = [
        sample_path
        for sample_path, _ in test_dataset.samples
    ]

    (
        train_loader,
        train_evaluation_loader,
        validation_loader,
        test_loader,
    ) = create_loaders(
        train_dataset,
        train_evaluation_dataset,
        validation_dataset,
        test_dataset,
    )

    (
        model,
        criterion,
        optimizer,
        scheduler,
        scaler,
    ) = create_model_and_training_objects(number_of_classes)

    print(f"\nBackbone output dimension: {model.backbone_feature_dim}")
    print_parameter_summary(model)

    model_complexity = profile_model_complexity(model)
    remove_thop_buffers_from_model(model)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    history_df, total_training_seconds = train_model(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        class_names=class_names,
        class_to_idx=class_to_idx,
    )

    save_training_curves(history_df)

    training_peak_allocated_mb = (
        float(torch.cuda.max_memory_allocated() / (1024 ** 2))
        if DEVICE.type == "cuda"
        else np.nan
    )
    training_peak_reserved_mb = (
        float(torch.cuda.max_memory_reserved() / (1024 ** 2))
        if DEVICE.type == "cuda"
        else np.nan
    )

    training_efficiency = {
        "Completed Epochs": int(len(history_df)),
        "Total Training Time Seconds": float(total_training_seconds),
        "Total Training Time Minutes": float(
            total_training_seconds / 60.0
        ),
        "Average Epoch Time Seconds": float(
            history_df["Epoch Time Seconds"].mean()
        ),
        "Median Epoch Time Seconds": float(
            history_df["Epoch Time Seconds"].median()
        ),
        "Fastest Epoch Time Seconds": float(
            history_df["Epoch Time Seconds"].min()
        ),
        "Slowest Epoch Time Seconds": float(
            history_df["Epoch Time Seconds"].max()
        ),
        "Peak Training Memory Allocated MB":
            training_peak_allocated_mb,
        "Peak Training Memory Reserved MB":
            training_peak_reserved_mb,
    }
    save_json(
        training_efficiency,
        OUTPUT_DIR / "training_efficiency.json",
    )
    pd.DataFrame(
        list(training_efficiency.items()),
        columns=["Measure", "Value"],
    ).to_csv(
        OUTPUT_DIR / "training_efficiency.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Load the best checkpoint on the selected device.
    # --------------------------------------------------------

    checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
    )

    load_clean_model_state(model, checkpoint["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()

    print("\nBest checkpoint")
    print("---------------")
    print(f"Epoch                : {checkpoint['epoch']}")
    print(
        "Validation macro F1  : "
        f"{checkpoint['best_validation_macro_f1']:.6f}"
    )
    print(
        "Validation loss      : "
        f"{checkpoint['best_validation_loss']:.6f}"
    )
    print(
        "Model device         : "
        f"{next(model.parameters()).device}"
    )

    (
        inference_latency_dataframe,
        end_to_end_inference,
    ) = benchmark_model_inference(
        model=model,
        test_loader=test_loader,
    )

    # --------------------------------------------------------
    # Raw logits for stable train, validation and test loaders.
    # --------------------------------------------------------

    (
        train_loss,
        train_logits,
        train_labels,
    ) = evaluate_logits(
        model,
        train_evaluation_loader,
        criterion,
    )

    (
        validation_loss,
        validation_logits,
        validation_labels,
    ) = evaluate_logits(
        model,
        validation_loader,
        criterion,
    )

    (
        test_loss,
        test_logits,
        test_labels,
    ) = evaluate_logits(
        model,
        test_loader,
        criterion,
    )

    # --------------------------------------------------------
    # Fit calibration only on validation logits.
    # --------------------------------------------------------

    temperature_scaler = fit_temperature_scaler(
        validation_logits,
        validation_labels,
    )

    train_calibrated_logits = calibrate_logits(
        train_logits,
        temperature_scaler,
    )

    validation_calibrated_logits = calibrate_logits(
        validation_logits,
        temperature_scaler,
    )

    test_calibrated_logits = calibrate_logits(
        test_logits,
        temperature_scaler,
    )

    # Positive class is configurable for binary classification.
    if number_of_classes == 2:
        if POSITIVE_CLASS_NAME is None:
            positive_index = 1
        elif POSITIVE_CLASS_NAME in class_to_idx:
            positive_index = class_to_idx[POSITIVE_CLASS_NAME]
        else:
            raise ValueError(
                f"POSITIVE_CLASS_NAME={POSITIVE_CLASS_NAME!r} was not "
                f"found. Available classes: {class_names}"
            )
    else:
        positive_index = None

    train_result = evaluate_split(
        split_name="Train",
        labels=train_labels,
        raw_logits=train_logits,
        calibrated_logits=train_calibrated_logits,
        class_names=class_names,
        positive_index=positive_index,
        sample_paths=train_sample_paths,
    )

    validation_result = evaluate_split(
        split_name="Validation",
        labels=validation_labels,
        raw_logits=validation_logits,
        calibrated_logits=validation_calibrated_logits,
        class_names=class_names,
        positive_index=positive_index,
        sample_paths=validation_sample_paths,
    )

    test_result = evaluate_split(
        split_name="Test",
        labels=test_labels,
        raw_logits=test_logits,
        calibrated_logits=test_calibrated_logits,
        class_names=class_names,
        positive_index=positive_index,
        sample_paths=test_sample_paths,
    )

    train_result["calibrated_metrics"]["Cross-Entropy Loss"] = (
        train_loss
    )
    validation_result["calibrated_metrics"][
        "Cross-Entropy Loss"
    ] = validation_loss
    test_result["calibrated_metrics"]["Cross-Entropy Loss"] = (
        test_loss
    )

    # Re-save calibrated metric tables after adding split loss.
    for split_name, result in [
        ("Train", train_result),
        ("Validation", validation_result),
        ("Test", test_result),
    ]:
        pd.DataFrame(
            list(result["calibrated_metrics"].items()),
            columns=["Metric", "Value"],
        ).to_csv(
            OUTPUT_DIR
            / f"{split_name.lower()}_calibrated_metrics.csv",
            index=False,
        )

    # --------------------------------------------------------
    # Combined metric table.
    # --------------------------------------------------------

    summary_rows = []

    for split_name, result in [
        ("Train Raw", train_result["raw_metrics"]),
        (
            "Train Calibrated",
            train_result["calibrated_metrics"],
        ),
        (
            "Validation Raw",
            validation_result["raw_metrics"],
        ),
        (
            "Validation Calibrated",
            validation_result["calibrated_metrics"],
        ),
        ("Test Raw", test_result["raw_metrics"]),
        (
            "Test Calibrated",
            test_result["calibrated_metrics"],
        ),
    ]:
        summary_rows.append({
            "Split": split_name,
            **result,
        })

    summary_metrics_dataframe = pd.DataFrame(summary_rows)
    summary_metrics_dataframe.to_csv(
        OUTPUT_DIR / "overall_metrics_train_val_test.csv",
        index=False,
    )

    print_dataframe(
        pd.DataFrame(
            list(
                test_result[
                    "calibrated_metrics"
                ].items()
            ),
            columns=["Metric", "Value"],
        ),
        "Calibrated test metrics",
    )

    print_dataframe(
        test_result["per_class"].round(6),
        "Test per-class metrics",
    )

    print("\nTest classification report")
    print("--------------------------")
    print(
        classification_report(
            test_labels,
            test_result["calibrated_predictions"],
            labels=np.arange(len(class_names)),
            target_names=class_names,
            digits=6,
            zero_division=0,
        )
    )

    # --------------------------------------------------------
    # Validation-based threshold selection for binary data.
    # --------------------------------------------------------

    selected_test_threshold = None

    if number_of_classes == 2:
        (
            threshold_dataframe,
            best_youden,
            best_f1,
        ) = threshold_analysis(
            labels=validation_labels,
            probabilities=validation_result[
                "calibrated_probabilities"
            ],
            positive_index=positive_index,
            class_names=class_names,
        )

        threshold_dataframe.to_csv(
            OUTPUT_DIR
            / "validation_threshold_analysis.csv",
            index=False,
        )

        pd.DataFrame([best_youden]).to_csv(
            OUTPUT_DIR
            / "best_validation_youden_threshold.csv",
            index=False,
        )

        pd.DataFrame([best_f1]).to_csv(
            OUTPUT_DIR
            / "best_validation_f1_threshold.csv",
            index=False,
        )

        plot_threshold_analysis(threshold_dataframe)

        # Use the validation-selected Youden threshold on test.
        selected_test_threshold = float(
            best_youden["Threshold"]
        )

        (
            thresholded_test_metrics,
            thresholded_test_confusion,
            _,
            _,
            thresholded_per_class,
            thresholded_predictions,
        ) = compute_metric_dict(
            labels=test_labels,
            predictions=test_result[
                "calibrated_predictions"
            ],
            probabilities=test_result[
                "calibrated_probabilities"
            ],
            class_names=class_names,
            positive_index=positive_index,
            number_of_bins=ECE_BINS,
            binary_threshold=selected_test_threshold,
        )

        pd.DataFrame(
            list(thresholded_test_metrics.items()),
            columns=["Metric", "Value"],
        ).to_csv(
            OUTPUT_DIR
            / "test_metrics_validation_youden_threshold.csv",
            index=False,
        )

        thresholded_per_class.to_csv(
            OUTPUT_DIR
            / "test_per_class_validation_youden_threshold.csv",
            index=False,
        )

        plot_confusion(
            confusion=thresholded_test_confusion,
            class_names=class_names,
            filename=(
                "test_confusion_matrix_validation_"
                "youden_threshold.png"
            ),
            title=(
                f"{MODEL_DISPLAY_NAME} Test Confusion Matrix\n"
                f"Validation Youden threshold = "
                f"{selected_test_threshold:.2f}"
            ),
        )

        print(
            "\nValidation-selected binary threshold"
            "\n------------------------------------"
        )
        print(
            f"Youden threshold: "
            f"{selected_test_threshold:.2f}"
        )

    # --------------------------------------------------------
    # Test bootstrap confidence intervals.
    # --------------------------------------------------------

    print(
        f"\nRunning {BOOTSTRAP_N} bootstrap iterations "
        "for test 95% confidence intervals..."
    )

    bootstrap_dataframe = bootstrap_confidence_intervals(
        labels=test_labels,
        predictions=test_result["calibrated_predictions"],
        probabilities=test_result[
            "calibrated_probabilities"
        ],
        class_names=class_names,
        positive_index=positive_index,
        binary_threshold=selected_test_threshold,
        number_of_bootstraps=BOOTSTRAP_N,
        seed=SEED,
    )

    bootstrap_dataframe.to_csv(
        OUTPUT_DIR / "bootstrap_95ci_test_metrics.csv",
        index=False,
    )

    print_dataframe(
        bootstrap_dataframe.round(6),
        "Bootstrap 95% confidence intervals",
    )

    # --------------------------------------------------------
    # Standard test plots.
    # --------------------------------------------------------

    confusion_count_dataframe = pd.DataFrame(
        test_result["confusion"],
        index=[f"True: {name}" for name in class_names],
        columns=[f"Predicted: {name}" for name in class_names],
    )
    confusion_count_dataframe.to_csv(
        OUTPUT_DIR / "test_confusion_matrix_counts.csv"
    )

    row_totals = test_result["confusion"].sum(
        axis=1,
        keepdims=True,
    )
    confusion_normalized = np.divide(
        test_result["confusion"],
        row_totals,
        out=np.zeros_like(
            test_result["confusion"],
            dtype=float,
        ),
        where=row_totals != 0,
    )
    pd.DataFrame(
        confusion_normalized,
        index=[f"True: {name}" for name in class_names],
        columns=[f"Predicted: {name}" for name in class_names],
    ).to_csv(
        OUTPUT_DIR / "test_confusion_matrix_normalized.csv"
    )

    plot_confusion(
        confusion=test_result["confusion"],
        class_names=class_names,
        filename="test_confusion_matrix.png",
        title=f"{MODEL_DISPLAY_NAME} Test Confusion Matrix (Counts)",
        normalize=None,
    )

    plot_confusion(
        confusion=test_result["confusion"],
        class_names=class_names,
        filename="test_confusion_matrix_normalized.png",
        title=(
            f"{MODEL_DISPLAY_NAME} Test Confusion Matrix "
            "(Row-Normalized)"
        ),
        normalize="true",
    )

    plot_roc_curves(
        labels=test_labels,
        probabilities=test_result[
            "calibrated_probabilities"
        ],
        class_names=class_names,
        positive_index=positive_index,
    )

    plot_precision_recall_curves(
        labels=test_labels,
        probabilities=test_result[
            "calibrated_probabilities"
        ],
        class_names=class_names,
        positive_index=positive_index,
    )

    plot_reliability_diagram(
        multiclass_bins=test_result["calibration_bins"],
        binary_bins=test_result[
            "binary_calibration_bins"
        ],
        binary_ece=test_result[
            "calibrated_metrics"
        ].get("Binary ECE"),
    )

    plot_confidence_histogram(
        test_result["calibrated_probabilities"]
    )

    plot_class_confidence_summary(
        test_result["class_confidence"]
    )

    plot_confidence_boxplot(
        labels=test_labels,
        probabilities=test_result["calibrated_probabilities"],
        class_names=class_names,
    )

    plot_confidence_accuracy_curve(
        test_result["confidence_bins"]
    )

    plot_class_probability_heatmap(
        test_result["probability_matrix"]
    )

    plot_correct_incorrect_confidence(
        labels=test_labels,
        predictions=test_result["calibrated_predictions"],
        probabilities=test_result["calibrated_probabilities"],
    )

    # --------------------------------------------------------
    # Save final checkpoint including temperature.
    # --------------------------------------------------------

    torch.save(
        {
            "epoch": checkpoint["epoch"],
            "model_state_dict": clean_checkpoint_state_dict(model),
            "temperature_state_dict":
                temperature_scaler.state_dict(),
            "temperature": float(
                temperature_scaler.temperature
                .detach()
                .cpu()
                .item()
            ),
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "positive_index": positive_index,
            "validation_youden_threshold":
                selected_test_threshold,
            "best_validation_macro_f1":
                checkpoint["best_validation_macro_f1"],
            "best_validation_loss":
                checkpoint["best_validation_loss"],
            "configuration":
                checkpoint["configuration"],
        },
        FINAL_MODEL_PATH,
    )

    best_model_size_mb = float(
        BEST_MODEL_PATH.stat().st_size / (1024 ** 2)
    )
    final_model_size_mb = float(
        FINAL_MODEL_PATH.stat().st_size / (1024 ** 2)
    )

    batch_one_latency = {}
    if not inference_latency_dataframe.empty:
        batch_one_rows = inference_latency_dataframe.loc[
            inference_latency_dataframe["Batch Size"] == 1
        ]
        if not batch_one_rows.empty:
            batch_one_latency = batch_one_rows.iloc[0].to_dict()

    run_summary = {
        "Model": MODEL_DISPLAY_NAME,
        "Device": str(DEVICE),
        "Number of Classes": number_of_classes,
        "Classes": class_names,
        "Train Images": len(train_dataset),
        "Validation Images": len(validation_dataset),
        "Test Images": len(test_dataset),
        "Best Epoch": checkpoint["epoch"],
        "Best Validation Macro F1":
            checkpoint["best_validation_macro_f1"],
        "Best Validation Loss":
            checkpoint["best_validation_loss"],
        "Training Time Seconds": total_training_seconds,
        "Training Time Minutes":
            total_training_seconds / 60,
        "Average Epoch Time Seconds": training_efficiency[
            "Average Epoch Time Seconds"
        ],
        "Peak Training Memory Allocated MB":
            training_peak_allocated_mb,
        "Peak Training Memory Reserved MB":
            training_peak_reserved_mb,
        "Total Parameters": model_complexity[
            "Total Parameters"
        ],
        "Trainable Parameters": model_complexity[
            "Trainable Parameters"
        ],
        "MACs GMAC": model_complexity.get("MACs GMAC"),
        "Estimated FLOPs GFLOPs": model_complexity.get(
            "Estimated FLOPs GFLOPs"
        ),
        "Model-Only Batch-1 Latency ms": batch_one_latency.get(
            "Mean Batch Latency ms"
        ),
        "Model-Only Batch-1 Throughput Images per Second":
            batch_one_latency.get(
                "Model Throughput Images per Second"
            ),
        "End-to-End Test Inference Seconds":
            end_to_end_inference[
                "End-to-End Test Inference Seconds"
            ],
        "End-to-End Per-Image Latency ms":
            end_to_end_inference[
                "End-to-End Per-Image Latency ms"
            ],
        "End-to-End Throughput Images per Second":
            end_to_end_inference[
                "End-to-End Throughput Images per Second"
            ],
        "Temperature": float(
            temperature_scaler.temperature
            .detach()
            .cpu()
            .item()
        ),
        "Validation Youden Threshold":
            selected_test_threshold,
        "Test Accuracy":
            test_result["calibrated_metrics"]["Accuracy"],
        "Test Balanced Accuracy":
            test_result[
                "calibrated_metrics"
            ]["Balanced Accuracy"],
        "Test Macro F1":
            test_result["calibrated_metrics"]["Macro F1"],
        "Test MCC":
            test_result["calibrated_metrics"]["MCC"],
        "Test Cohen Kappa":
            test_result[
                "calibrated_metrics"
            ]["Cohen Kappa"],
        "Test Mean Confidence": test_result[
            "calibrated_metrics"
        ]["Mean Confidence"],
        "Test Mean True-Class Probability": test_result[
            "calibrated_metrics"
        ]["Mean True-Class Probability"],
        "Test ECE": test_result[
            "calibrated_metrics"
        ]["ECE"],
        "Test Log Loss / NLL": test_result[
            "calibrated_metrics"
        ]["Log Loss / NLL"],
        "Best Model Size MB": best_model_size_mb,
        "Final Model Size MB": final_model_size_mb,
        "Environment": environment_information,
        "Best Model Path": str(BEST_MODEL_PATH),
        "Final Model Path": str(FINAL_MODEL_PATH),
    }

    save_json(
        run_summary,
        OUTPUT_DIR / "run_summary.json",
    )
    pd.DataFrame(
        [
            {
                key: value
                for key, value in run_summary.items()
                if not isinstance(value, (dict, list))
            }
        ]
    ).to_csv(
        OUTPUT_DIR / "run_summary.csv",
        index=False,
    )

    print("\nCompleted successfully")
    print("======================")
    print(f"Best model : {BEST_MODEL_PATH}")
    print(f"Final model: {FINAL_MODEL_PATH}")
    print(f"Results    : {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# MAXVIT-TINY + GRFM CHECKPOINT VISUALIZATION
#
# Loads an existing checkpoint and explicitly displays:
#   1. Confusion matrix
#   2. Precision-recall / PR-AUC curves
#   3. t-SNE feature visualization
#   4. Confidence histogram
#
# It also saves every figure as a PNG.
# No model training is performed.
# ============================================================

import contextlib
import copy
import os
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm

from IPython.display import display
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.preprocessing import label_binarize, StandardScaler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

TEST_DIR = Path(
    "/kaggle/working/PVMD_Split/test"
)

RESULTS_DIR = Path(
    "/kaggle/working/MaxViT_Tiny_GRFM_Results"
)

FINAL_MODEL_PATH = (
    RESULTS_DIR / "final_maxvit_tiny_grfm.pth"
)

BEST_MODEL_PATH = (
    RESULTS_DIR / "best_maxvit_tiny_grfm.pth"
)

OUTPUT_DIR = (
    RESULTS_DIR / "Checkpoint_New_Plots"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_NAME = "maxvit_tiny_tf_224.in1k"
MODEL_DISPLAY_NAME = "MaxViT-Tiny-224 + GRFM"

IMG_SIZE = 224
BATCH_SIZE = 16

# Set this to 0 in Kaggle for the most reliable notebook display.
NUM_WORKERS = 0

GRFM_DIM = 512
GRFM_DROPOUT = 0.30
GRFM_REDUCTION = 4

TSNE_MAX_SAMPLES = 1000
TSNE_PCA_COMPONENTS = 50
TSNE_PERPLEXITY = 30
TSNE_ITERATIONS = 1500

CONFIDENCE_BINS = 20

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]

THOP_BUFFER_NAMES = {
    "total_ops",
    "total_params",
}


# ============================================================
# 2. UTILITIES
# ============================================================

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def amp_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return contextlib.nullcontext()


def show_and_save(
    figure,
    filename,
    dpi=300,
):
    """
    Explicitly display a Matplotlib figure in Kaggle/Jupyter and save it.
    """
    output_path = (
        OUTPUT_DIR / filename
    )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
    )

    # This is more reliable in notebooks than relying only on plt.show().
    display(figure)

    plt.close(figure)

    print(
        f"Saved: {output_path}"
    )


# ============================================================
# 3. DATASET
# ============================================================

evaluation_transform = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])


def create_test_loader():
    if not TEST_DIR.exists():
        raise FileNotFoundError(
            f"Test directory not found: "
            f"{TEST_DIR}"
        )

    dataset = datasets.ImageFolder(
        root=str(TEST_DIR),
        transform=evaluation_transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    return dataset, loader


# ============================================================
# 4. MODEL
# ============================================================

class GRFM(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim=512,
        reduction=4,
        dropout=0.30,
    ):
        super().__init__()

        hidden_dim = max(
            output_dim // reduction,
            32,
        )

        self.projection = nn.Sequential(
            nn.Linear(
                input_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
        )

        self.feature_transform = nn.Sequential(
            nn.Linear(
                output_dim,
                output_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                output_dim,
                output_dim,
            ),
        )

        self.feature_gate = nn.Sequential(
            nn.Linear(
                output_dim,
                hidden_dim,
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.Sigmoid(),
        )

        self.output_norm = nn.LayerNorm(
            output_dim
        )

        self.output_dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        features,
    ):
        projected = self.projection(
            features
        )

        transformed = self.feature_transform(
            projected
        )

        gate = self.feature_gate(
            projected
        )

        refined = (
            projected
            + transformed * gate
        )

        refined = self.output_norm(
            refined
        )

        refined = self.output_dropout(
            refined
        )

        return refined


class MaxViTTinyGRFM(nn.Module):

    def __init__(
        self,
        num_classes,
        grfm_dim=512,
        dropout=0.30,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            MODEL_NAME,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )

        self.grfm = GRFM(
            input_dim=(
                self.backbone.num_features
            ),
            output_dim=grfm_dim,
            reduction=GRFM_REDUCTION,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                grfm_dim,
                grfm_dim // 2,
            ),
            nn.LayerNorm(
                grfm_dim // 2
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                grfm_dim // 2,
                num_classes,
            ),
        )

    def forward(
        self,
        images,
        return_features=False,
    ):
        backbone_features = self.backbone(
            images
        )

        refined_features = self.grfm(
            backbone_features
        )

        logits = self.classifier(
            refined_features
        )

        if return_features:
            return {
                "logits":
                    logits,

                "backbone_features":
                    backbone_features,

                "refined_features":
                    refined_features,
            }

        return logits


# ============================================================
# 5. SAFE CHECKPOINT LOADING
# ============================================================

def clean_state_dict(
    raw_state_dict,
):
    cleaned = {}

    for key, value in (
        raw_state_dict.items()
    ):
        final_name = key.rsplit(
            ".",
            1,
        )[-1]

        if final_name in (
            THOP_BUFFER_NAMES
        ):
            continue

        if torch.is_tensor(value):
            cleaned[key] = (
                value.detach()
                .cpu()
                .clone()
            )
        else:
            cleaned[key] = (
                copy.deepcopy(value)
            )

    return cleaned


def load_checkpoint_model(
    dataset_classes,
):
    if FINAL_MODEL_PATH.exists():
        checkpoint_path = (
            FINAL_MODEL_PATH
        )

    elif BEST_MODEL_PATH.exists():
        checkpoint_path = (
            BEST_MODEL_PATH
        )

    else:
        raise FileNotFoundError(
            "No checkpoint found.\n"
            f"Checked: {FINAL_MODEL_PATH}\n"
            f"Checked: {BEST_MODEL_PATH}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    class_names = list(
        checkpoint.get(
            "class_names",
            dataset_classes,
        )
    )

    if list(dataset_classes) != class_names:
        raise ValueError(
            "Checkpoint classes do not "
            "match test-folder classes.\n"
            f"Checkpoint: {class_names}\n"
            f"Dataset: {list(dataset_classes)}"
        )

    configuration = checkpoint.get(
        "configuration",
        {},
    )

    grfm_dim = int(
        configuration.get(
            "grfm_dim",
            GRFM_DIM,
        )
    )

    dropout = float(
        configuration.get(
            "grfm_dropout",
            GRFM_DROPOUT,
        )
    )

    model = MaxViTTinyGRFM(
        num_classes=len(class_names),
        grfm_dim=grfm_dim,
        dropout=dropout,
    )

    raw_state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )

    incompatible = model.load_state_dict(
        clean_state_dict(
            raw_state_dict
        ),
        strict=False,
    )

    meaningful_missing = [
        key
        for key
        in incompatible.missing_keys
        if key.rsplit(".", 1)[-1]
        not in THOP_BUFFER_NAMES
    ]

    meaningful_unexpected = [
        key
        for key
        in incompatible.unexpected_keys
        if key.rsplit(".", 1)[-1]
        not in THOP_BUFFER_NAMES
    ]

    if (
        meaningful_missing
        or meaningful_unexpected
    ):
        raise RuntimeError(
            "Checkpoint architecture mismatch.\n"
            f"Missing keys: "
            f"{meaningful_missing}\n"
            f"Unexpected keys: "
            f"{meaningful_unexpected}"
        )

    model = model.to(DEVICE)
    model.eval()

    temperature = float(
        checkpoint.get(
            "temperature",
            1.0,
        )
    )

    if (
        not np.isfinite(temperature)
        or temperature <= 0
    ):
        temperature = 1.0

    print(
        f"Checkpoint: {checkpoint_path}"
    )

    print(
        f"Epoch     : "
        f"{checkpoint.get('epoch', 'unknown')}"
    )

    print(
        f"Classes   : {class_names}"
    )

    print(
        f"Device    : {DEVICE}"
    )

    return (
        model,
        class_names,
        checkpoint_path,
        temperature,
    )


# ============================================================
# 6. EXTRACT PREDICTIONS AND FEATURES
# ============================================================

@torch.inference_mode()
def extract_predictions_and_features(
    model,
    loader,
    temperature,
):
    labels_all = []
    logits_all = []
    backbone_features_all = []
    refined_features_all = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        with amp_context():
            outputs = model(
                images,
                return_features=True,
            )

        logits = (
            outputs["logits"].float()
            / temperature
        )

        logits_all.append(
            logits.cpu()
        )

        backbone_features_all.append(
            outputs[
                "backbone_features"
            ].float().cpu()
        )

        refined_features_all.append(
            outputs[
                "refined_features"
            ].float().cpu()
        )

        labels_all.append(
            labels.numpy()
        )

    logits = torch.cat(
        logits_all,
        dim=0,
    )

    probabilities = torch.softmax(
        logits,
        dim=1,
    ).numpy()

    predictions = probabilities.argmax(
        axis=1
    )

    labels = np.concatenate(
        labels_all
    ).astype(
        np.int64
    )

    backbone_features = torch.cat(
        backbone_features_all,
        dim=0,
    ).numpy()

    refined_features = torch.cat(
        refined_features_all,
        dim=0,
    ).numpy()

    return (
        labels,
        predictions,
        probabilities,
        backbone_features,
        refined_features,
    )


# ============================================================
# 7. CONFUSION MATRIX
# ============================================================

def plot_confusion(
    labels,
    predictions,
    class_names,
):
    confusion = confusion_matrix(
        labels,
        predictions,
        labels=np.arange(
            len(class_names)
        ),
    )

    figure, axis = plt.subplots(
        figsize=(8, 7)
    )

    image = axis.imshow(
        confusion,
        interpolation="nearest",
    )

    figure.colorbar(
        image,
        ax=axis,
    )

    positions = np.arange(
        len(class_names)
    )

    axis.set_xticks(
        positions
    )

    axis.set_yticks(
        positions
    )

    axis.set_xticklabels(
        class_names,
        rotation=35,
        ha="right",
    )

    axis.set_yticklabels(
        class_names
    )

    threshold = (
        confusion.max() / 2.0
        if confusion.size
        else 0.0
    )

    for row in range(
        confusion.shape[0]
    ):
        for column in range(
            confusion.shape[1]
        ):
            value = confusion[
                row,
                column,
            ]

            axis.text(
                column,
                row,
                str(value),
                ha="center",
                va="center",
                color=(
                    "white"
                    if value > threshold
                    else "black"
                ),
            )

    axis.set_xlabel(
        "Predicted class"
    )

    axis.set_ylabel(
        "True class"
    )

    axis.set_title(
        f"{MODEL_DISPLAY_NAME}\n"
        "Confusion matrix"
    )

    show_and_save(
        figure,
        "confusion_matrix.png",
    )

    pd.DataFrame(
        confusion,
        index=class_names,
        columns=class_names,
    ).to_csv(
        OUTPUT_DIR
        / "confusion_matrix.csv"
    )


# ============================================================
# 8. PR-AUC CURVE
# ============================================================

def plot_pr_auc(
    labels,
    probabilities,
    class_names,
):
    number_of_classes = (
        probabilities.shape[1]
    )

    targets = label_binarize(
        labels,
        classes=np.arange(
            number_of_classes
        ),
    )

    if (
        number_of_classes == 2
        and targets.shape[1] == 1
    ):
        targets = np.column_stack(
            [
                1 - targets[:, 0],
                targets[:, 0],
            ]
        )

    figure, axis = plt.subplots(
        figsize=(9, 7)
    )

    rows = []

    for class_index, class_name in enumerate(
        class_names
    ):
        class_targets = targets[
            :,
            class_index,
        ]

        if (
            len(
                np.unique(
                    class_targets
                )
            ) < 2
        ):
            continue

        precision_values, recall_values, _ = (
            precision_recall_curve(
                class_targets,
                probabilities[
                    :,
                    class_index,
                ],
            )
        )

        average_precision = (
            average_precision_score(
                class_targets,
                probabilities[
                    :,
                    class_index,
                ],
            )
        )

        axis.plot(
            recall_values,
            precision_values,
            linewidth=2,
            label=(
                f"{class_name}: "
                f"AP={average_precision:.4f}"
            ),
        )

        rows.append({
            "Class":
                class_name,

            "PR-AUC":
                average_precision,
        })

    micro_precision, micro_recall, _ = (
        precision_recall_curve(
            targets.ravel(),
            probabilities.ravel(),
        )
    )

    micro_ap = (
        average_precision_score(
            targets,
            probabilities,
            average="micro",
        )
    )

    macro_ap = (
        average_precision_score(
            targets,
            probabilities,
            average="macro",
        )
    )

    axis.plot(
        micro_recall,
        micro_precision,
        linestyle="--",
        linewidth=3,
        label=(
            "Micro-average: "
            f"AP={micro_ap:.4f}"
        ),
    )

    axis.set_xlabel(
        "Recall"
    )

    axis.set_ylabel(
        "Precision"
    )

    axis.set_xlim(
        0.0,
        1.01,
    )

    axis.set_ylim(
        0.0,
        1.05,
    )

    axis.grid(
        True,
        linestyle="--",
        alpha=0.4,
    )

    axis.legend(
        loc="lower left",
    )

    axis.set_title(
        f"{MODEL_DISPLAY_NAME}\n"
        "Precision-recall curves"
    )

    show_and_save(
        figure,
        "pr_auc_curve.png",
    )

    rows.extend([
        {
            "Class":
                "Macro average",

            "PR-AUC":
                macro_ap,
        },
        {
            "Class":
                "Micro average",

            "PR-AUC":
                micro_ap,
        },
    ])

    pd.DataFrame(
        rows
    ).to_csv(
        OUTPUT_DIR
        / "pr_auc_values.csv",
        index=False,
    )


# ============================================================
# 9. T-SNE
# ============================================================

def stratified_sample_indices(
    labels,
    maximum_samples,
):
    number_of_samples = len(
        labels
    )

    if number_of_samples <= maximum_samples:
        return np.arange(
            number_of_samples
        )

    random_generator = (
        np.random.default_rng(
            SEED
        )
    )

    unique_classes = np.unique(
        labels
    )

    samples_per_class = max(
        1,
        maximum_samples
        // len(unique_classes),
    )

    selected = []

    for class_index in unique_classes:
        class_indices = np.where(
            labels == class_index
        )[0]

        take = min(
            samples_per_class,
            len(class_indices),
        )

        selected.extend(
            random_generator.choice(
                class_indices,
                size=take,
                replace=False,
            ).tolist()
        )

    if len(selected) > maximum_samples:
        selected = (
            random_generator.choice(
                np.asarray(selected),
                size=maximum_samples,
                replace=False,
            ).tolist()
        )

    return np.asarray(
        selected,
        dtype=np.int64,
    )


def plot_tsne(
    features,
    labels,
    class_names,
):
    selected_indices = (
        stratified_sample_indices(
            labels,
            TSNE_MAX_SAMPLES,
        )
    )

    selected_features = features[
        selected_indices
    ]

    selected_labels = labels[
        selected_indices
    ]

    print(
        f"Running t-SNE on "
        f"{len(selected_indices)} samples..."
    )

    selected_features = (
        StandardScaler()
        .fit_transform(
            selected_features
        )
    )

    maximum_components = min(
        TSNE_PCA_COMPONENTS,
        selected_features.shape[1],
        len(selected_features) - 1,
    )

    if maximum_components >= 2:
        selected_features = PCA(
            n_components=maximum_components,
            random_state=SEED,
        ).fit_transform(
            selected_features
        )

    valid_perplexity = min(
        TSNE_PERPLEXITY,
        max(
            5,
            (
                len(selected_features)
                - 1
            )
            // 3,
        ),
    )

    embedding = TSNE(
        n_components=2,
        perplexity=valid_perplexity,
        max_iter=TSNE_ITERATIONS,
        init="pca",
        learning_rate="auto",
        random_state=SEED,
    ).fit_transform(
        selected_features
    )

    figure, axis = plt.subplots(
        figsize=(9, 7)
    )

    for class_index, class_name in enumerate(
        class_names
    ):
        mask = (
            selected_labels
            == class_index
        )

        axis.scatter(
            embedding[
                mask,
                0,
            ],
            embedding[
                mask,
                1,
            ],
            s=35,
            alpha=0.8,
            label=class_name,
        )

    axis.set_xlabel(
        "t-SNE dimension 1"
    )

    axis.set_ylabel(
        "t-SNE dimension 2"
    )

    axis.set_title(
        f"{MODEL_DISPLAY_NAME}\n"
        "t-SNE of GRFM-refined features"
    )

    axis.legend(
        loc="best",
    )

    axis.grid(
        True,
        linestyle="--",
        alpha=0.3,
    )

    show_and_save(
        figure,
        "tsne_grfm_features.png",
    )

    pd.DataFrame({
        "TSNE 1":
            embedding[:, 0],

        "TSNE 2":
            embedding[:, 1],

        "Label":
            selected_labels,

        "Class": [
            class_names[index]
            for index
            in selected_labels
        ],
    }).to_csv(
        OUTPUT_DIR
        / "tsne_coordinates.csv",
        index=False,
    )


# ============================================================
# 10. CONFIDENCE HISTOGRAM
# ============================================================

def plot_confidence_histogram(
    labels,
    predictions,
    probabilities,
):
    confidence = probabilities.max(
        axis=1
    )

    correct_mask = (
        predictions == labels
    )

    bins = np.linspace(
        0.0,
        1.0,
        CONFIDENCE_BINS + 1,
    )

    figure, axis = plt.subplots(
        figsize=(9, 6)
    )

    if correct_mask.any():
        axis.hist(
            confidence[
                correct_mask
            ],
            bins=bins,
            alpha=0.7,
            label=(
                "Correct "
                f"(n={correct_mask.sum()})"
            ),
        )

    if (~correct_mask).any():
        axis.hist(
            confidence[
                ~correct_mask
            ],
            bins=bins,
            alpha=0.7,
            label=(
                "Incorrect "
                f"(n={(~correct_mask).sum()})"
            ),
        )

    axis.axvline(
        confidence.mean(),
        linestyle="--",
        linewidth=2,
        label=(
            "Mean confidence = "
            f"{confidence.mean():.4f}"
        ),
    )

    axis.set_xlabel(
        "Maximum predicted probability"
    )

    axis.set_ylabel(
        "Number of images"
    )

    axis.set_title(
        f"{MODEL_DISPLAY_NAME}\n"
        "Confidence histogram"
    )

    axis.legend()

    show_and_save(
        figure,
        "confidence_histogram.png",
    )

    pd.DataFrame({
        "True Label":
            labels,

        "Predicted Label":
            predictions,

        "Correct":
            correct_mask.astype(int),

        "Confidence":
            confidence,
    }).to_csv(
        OUTPUT_DIR
        / "confidence_values.csv",
        index=False,
    )


# ============================================================
# 11. MAIN
# ============================================================

def main():
    set_seed(SEED)

    print("=" * 80)
    print(
        f"{MODEL_DISPLAY_NAME} "
        "checkpoint plotting"
    )
    print("=" * 80)

    dataset, loader = (
        create_test_loader()
    )

    (
        model,
        class_names,
        checkpoint_path,
        temperature,
    ) = load_checkpoint_model(
        dataset.classes
    )

    (
        labels,
        predictions,
        probabilities,
        backbone_features,
        refined_features,
    ) = extract_predictions_and_features(
        model=model,
        loader=loader,
        temperature=temperature,
    )

    accuracy = accuracy_score(
        labels,
        predictions,
    )

    print(
        f"\nTest accuracy: "
        f"{accuracy:.6f}"
    )

    print(
        "\nDisplaying confusion matrix..."
    )

    plot_confusion(
        labels=labels,
        predictions=predictions,
        class_names=class_names,
    )

    print(
        "\nDisplaying PR-AUC curve..."
    )

    plot_pr_auc(
        labels=labels,
        probabilities=probabilities,
        class_names=class_names,
    )

    print(
        "\nDisplaying t-SNE..."
    )

    plot_tsne(
        features=refined_features,
        labels=labels,
        class_names=class_names,
    )

    print(
        "\nDisplaying confidence histogram..."
    )

    plot_confidence_histogram(
        labels=labels,
        predictions=predictions,
        probabilities=probabilities,
    )

    print(
        "\nCompleted successfully."
    )

    print(
        f"Saved outputs: {OUTPUT_DIR}"
    )


if __name__ == "__main__":
    main()

In [ ]:
"""
MaxViT-Tiny + GRFM visual explanation pipeline
==============================================

This script loads a trained MaxViT-Tiny + GRFM checkpoint and generates:

1. GRFM-guided feature-refinement maps.
2. t-SNE plots before and after GRFM refinement.
3. Misclassified-image grids and a detailed CSV report.
4. Grad-CAM++ visualizations.
5. Input-gradient saliency maps.
6. Three representative images per class for explanation panels.

The script does NOT retrain the model.

Important scientific note
-------------------------
GRFM operates on globally pooled feature vectors rather than a spatial tensor.
Therefore, the "GRFM-guided refined feature map" is a spatial visualization
proxy: the last spatial backbone feature tensor is weighted by the learned GRFM
gate. This is useful for interpretation, but it should not be described as an
exact spatial output produced directly by GRFM.
"""

import contextlib
import copy
import math
import os
import random
import re
import subprocess
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

try:
    import timm
except ImportError:
    print("timm is not installed. Installing timm...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

# Update these paths if needed.
TRAIN_DIR = Path("/kaggle/working/PVMD_Train_Augmented")
VAL_DIR = Path("/kaggle/working/PVMD_Split/validation")
TEST_DIR = Path("/kaggle/working/PVMD_Split/test")

if not VAL_DIR.exists():
    alternative_val = Path("/kaggle/working/PVMD_Split/val")
    if alternative_val.exists():
        VAL_DIR = alternative_val

TRAINING_OUTPUT_DIR = Path("/kaggle/working/MaxViT_Tiny_GRFM_Results")
FINAL_MODEL_PATH = TRAINING_OUTPUT_DIR / "final_maxvit_tiny_grfm.pth"
BEST_MODEL_PATH = TRAINING_OUTPUT_DIR / "best_maxvit_tiny_grfm.pth"

VISUALIZATION_DIR = TRAINING_OUTPUT_DIR / "Visual_Explanations"
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "maxvit_tiny_tf_224.in1k"
MODEL_DISPLAY_NAME = "MaxViT-Tiny-224 + GRFM"

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

GRFM_DIM = 512
GRFM_DROPOUT = 0.30
GRFM_REDUCTION = 4

# Three images per class, as requested.
SAMPLES_PER_CLASS = 3

# Misclassification output controls.
MAX_MISCLASSIFIED_GRID_IMAGES = 24
MAX_MISCLASSIFIED_PER_TRUE_CLASS = 3

# t-SNE controls. Limiting samples avoids excessive runtime and memory use.
TSNE_MAX_SAMPLES_PER_CLASS = 250
TSNE_PCA_COMPONENTS = 50
TSNE_PERPLEXITY = 30
TSNE_ITERATIONS = 1500

# Grad-CAM++ target layer.
# Leave as None to automatically use the final convolutional layer.
# To select one manually, enter its exact model.named_modules() name.
TARGET_LAYER_NAME: Optional[str] = None

# Explanation target:
# "true"      -> explain the ground-truth class score.
# "predicted" -> explain the model-predicted class score.
EXPLANATION_TARGET = "true"

# Select correctly classified images for the three-per-class panels whenever
# possible. If a class has fewer than three correct samples, remaining samples
# are filled using the most confident available images from that class.
PREFER_CORRECT_EXPLANATION_SAMPLES = True

HEATMAP_ALPHA = 0.45
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP_FOR_FEATURE_EXTRACTION = DEVICE.type == "cuda"

IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)


# ============================================================
# 2. REPRODUCIBILITY AND HELPERS
# ============================================================

def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def amp_context():
    if USE_AMP_FOR_FEATURE_EXTRACTION:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()


def sanitize_filename(text: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text).strip())
    return cleaned.strip("_") or "class"


def normalize_map(array: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    array = np.asarray(array, dtype=np.float32)
    array = np.nan_to_num(array, nan=0.0, posinf=0.0, neginf=0.0)
    minimum = float(array.min())
    maximum = float(array.max())

    if maximum - minimum < eps:
        return np.zeros_like(array, dtype=np.float32)

    return (array - minimum) / (maximum - minimum)


def tensor_to_rgb(image_tensor: torch.Tensor) -> np.ndarray:
    """Undo ImageNet normalization and return an RGB float image in [0, 1]."""
    image = image_tensor.detach().cpu().float().numpy().transpose(1, 2, 0)
    image = image * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(image, 0.0, 1.0)


def resize_heatmap(heatmap: np.ndarray, output_size: Tuple[int, int]) -> np.ndarray:
    heatmap = normalize_map(heatmap)
    heatmap_uint8 = np.uint8(np.round(heatmap * 255.0))
    resized = Image.fromarray(heatmap_uint8).resize(
        output_size,
        resample=Image.Resampling.BILINEAR,
    )
    return np.asarray(resized, dtype=np.float32) / 255.0


def heatmap_to_rgb(heatmap: np.ndarray, cmap_name: str = "jet") -> np.ndarray:
    heatmap = normalize_map(heatmap)
    color_map = matplotlib.colormaps.get_cmap(cmap_name)
    return color_map(heatmap)[..., :3].astype(np.float32)


def overlay_heatmap(
    rgb_image: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = HEATMAP_ALPHA,
    cmap_name: str = "jet",
) -> np.ndarray:
    height, width = rgb_image.shape[:2]
    resized_heatmap = resize_heatmap(heatmap, (width, height))
    colored_heatmap = heatmap_to_rgb(resized_heatmap, cmap_name=cmap_name)
    overlay = (1.0 - alpha) * rgb_image + alpha * colored_heatmap
    return np.clip(overlay, 0.0, 1.0)


def save_rgb_image(image: np.ndarray, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    image_uint8 = np.uint8(np.round(np.clip(image, 0.0, 1.0) * 255.0))
    Image.fromarray(image_uint8).save(path)


def validate_inputs() -> None:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")

    if not FINAL_MODEL_PATH.exists() and not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(
            "No checkpoint was found. Expected one of:\n"
            f"- {FINAL_MODEL_PATH}\n"
            f"- {BEST_MODEL_PATH}"
        )


# ============================================================
# 3. DATASET
# ============================================================

evaluation_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN.tolist(),
        std=IMAGENET_STD.tolist(),
    ),
])


def create_test_dataset_and_loader():
    dataset = datasets.ImageFolder(
        root=str(TEST_DIR),
        transform=evaluation_transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

    return dataset, loader


# ============================================================
# 4. GRFM AND MODEL DEFINITIONS
# ============================================================

class GRFM(nn.Module):
    """Gated Residual Feature Modulation."""

    def __init__(
        self,
        input_dim: int,
        output_dim: int = 512,
        reduction: int = 4,
        dropout: float = 0.30,
    ):
        super().__init__()

        hidden_dim = max(output_dim // reduction, 32)

        self.projection = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

        self.feature_transform = nn.Sequential(
            nn.Linear(output_dim, output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, output_dim),
        )

        self.feature_gate = nn.Sequential(
            nn.Linear(output_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

        self.output_norm = nn.LayerNorm(output_dim)
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_gate: bool = False):
        projected_feature = self.projection(x)
        transformed_feature = self.feature_transform(projected_feature)
        modulation_gate = self.feature_gate(projected_feature)

        modulated_feature = transformed_feature * modulation_gate
        fused_feature = projected_feature + modulated_feature
        fused_feature = self.output_norm(fused_feature)
        fused_feature = self.output_dropout(fused_feature)

        if return_gate:
            return fused_feature, modulation_gate

        return fused_feature


class MaxViTTinyGRFM(nn.Module):
    def __init__(
        self,
        num_classes: int,
        grfm_dim: int = GRFM_DIM,
        dropout: float = GRFM_DROPOUT,
    ):
        super().__init__()

        # pretrained=False is essential here because checkpoint weights are loaded.
        self.backbone = timm.create_model(
            MODEL_NAME,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )

        backbone_feature_dim = self.backbone.num_features

        self.grfm = GRFM(
            input_dim=backbone_feature_dim,
            output_dim=grfm_dim,
            reduction=GRFM_REDUCTION,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(grfm_dim, grfm_dim // 2),
            nn.LayerNorm(grfm_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(grfm_dim // 2, num_classes),
        )

        self.backbone_feature_dim = backbone_feature_dim

    def forward(
        self,
        images: torch.Tensor,
        return_features: bool = False,
        return_gate: bool = False,
    ):
        backbone_features = self.backbone(images)
        grfm_features, gate = self.grfm(backbone_features, return_gate=True)
        logits = self.classifier(grfm_features)

        if return_features or return_gate:
            return {
                "logits": logits,
                "backbone_features": backbone_features,
                "grfm_features": grfm_features,
                "gate": gate,
            }

        return logits


# ============================================================
# 5. CHECKPOINT LOADING
# ============================================================

THOP_BUFFER_NAMES = {"total_ops", "total_params"}


def remove_thop_buffers_from_model(model: nn.Module) -> None:
    """Remove profiling-only buffers that THOP may register on modules."""
    for module in model.modules():
        for buffer_name in tuple(module._buffers.keys()):
            if buffer_name in THOP_BUFFER_NAMES:
                module._buffers.pop(buffer_name, None)


def clean_checkpoint_state_dict(state_dict: dict) -> dict:
    """
    Remove THOP-only keys and convert inference tensors to ordinary CPU tensors.

    This supports both new clean checkpoints and older checkpoints that were
    accidentally saved after THOP profiling.
    """
    cleaned_state = {}

    for key, value in state_dict.items():
        final_name = key.rsplit(".", 1)[-1]

        if final_name in THOP_BUFFER_NAMES:
            continue

        if torch.is_tensor(value):
            cleaned_state[key] = value.detach().cpu().clone()
        else:
            cleaned_state[key] = copy.deepcopy(value)

    return cleaned_state


def load_checkpoint_and_model(dataset_class_names: Sequence[str]):
    checkpoint_path = (
        FINAL_MODEL_PATH
        if FINAL_MODEL_PATH.exists()
        else BEST_MODEL_PATH
    )

    # Load on CPU first. This avoids unnecessary GPU allocation and makes
    # contaminated inference tensors safe to clone before model loading.
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    checkpoint_class_names = checkpoint.get("class_names")
    class_names = (
        list(dataset_class_names)
        if checkpoint_class_names is None
        else list(checkpoint_class_names)
    )

    if list(dataset_class_names) != class_names:
        raise ValueError(
            "Checkpoint classes do not match test-folder classes.\n"
            f"Checkpoint: {class_names}\n"
            f"Test folders: {list(dataset_class_names)}"
        )

    configuration = checkpoint.get("configuration", {})
    grfm_dim = int(configuration.get("grfm_dim", GRFM_DIM))
    grfm_dropout = float(
        configuration.get("grfm_dropout", GRFM_DROPOUT)
    )

    model = MaxViTTinyGRFM(
        num_classes=len(class_names),
        grfm_dim=grfm_dim,
        dropout=grfm_dropout,
    )

    remove_thop_buffers_from_model(model)

    raw_state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )
    clean_state_dict = clean_checkpoint_state_dict(raw_state_dict)

    incompatible = model.load_state_dict(
        clean_state_dict,
        strict=False,
    )

    meaningful_missing = [
        key
        for key in incompatible.missing_keys
        if key.rsplit(".", 1)[-1] not in THOP_BUFFER_NAMES
    ]
    meaningful_unexpected = [
        key
        for key in incompatible.unexpected_keys
        if key.rsplit(".", 1)[-1] not in THOP_BUFFER_NAMES
    ]

    if meaningful_missing or meaningful_unexpected:
        raise RuntimeError(
            "Checkpoint and model architecture do not match.\n"
            f"Missing keys: {meaningful_missing}\n"
            f"Unexpected keys: {meaningful_unexpected}"
        )

    model = model.to(DEVICE)
    model.eval()

    removed_thop_keys = len(raw_state_dict) - len(clean_state_dict)

    print(f"Loaded checkpoint : {checkpoint_path}")
    print(f"Checkpoint epoch  : {checkpoint.get('epoch', 'unknown')}")
    print(f"Classes           : {class_names}")
    print(f"Device            : {DEVICE}")

    if removed_thop_keys > 0:
        print(
            "Checkpoint cleanup : removed "
            f"{removed_thop_keys} THOP profiling entries"
        )

    return model, class_names, checkpoint_path


# ============================================================
# 6. PREDICTIONS AND FEATURE EXTRACTION
# ============================================================

@torch.inference_mode()
def extract_predictions_and_features(
    model: nn.Module,
    loader: DataLoader,
    dataset: datasets.ImageFolder,
):
    logits_all: List[torch.Tensor] = []
    backbone_features_all: List[torch.Tensor] = []
    refined_features_all: List[torch.Tensor] = []
    gate_all: List[torch.Tensor] = []
    labels_all: List[np.ndarray] = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)

        with amp_context():
            outputs = model(images, return_features=True)

        logits_all.append(outputs["logits"].float().cpu())
        backbone_features_all.append(outputs["backbone_features"].float().cpu())
        refined_features_all.append(outputs["grfm_features"].float().cpu())
        gate_all.append(outputs["gate"].float().cpu())
        labels_all.append(labels.numpy())

    logits = torch.cat(logits_all, dim=0)
    probabilities = torch.softmax(logits, dim=1).numpy()
    predictions = np.argmax(probabilities, axis=1)

    backbone_features = torch.cat(backbone_features_all, dim=0).numpy()
    refined_features = torch.cat(refined_features_all, dim=0).numpy()
    gates = torch.cat(gate_all, dim=0).numpy()
    labels = np.concatenate(labels_all).astype(np.int64)

    if len(dataset.samples) != len(labels):
        raise RuntimeError("Dataset sample paths and extracted labels do not align.")

    image_paths = [sample[0] for sample in dataset.samples]

    return {
        "logits": logits.numpy(),
        "probabilities": probabilities,
        "predictions": predictions,
        "labels": labels,
        "backbone_features": backbone_features,
        "refined_features": refined_features,
        "gates": gates,
        "image_paths": image_paths,
    }


# ============================================================
# 7. TARGET LAYER AND ACTIVATION HOOK
# ============================================================

def list_convolutional_layers(model: nn.Module) -> List[str]:
    return [
        name
        for name, module in model.named_modules()
        if isinstance(module, nn.Conv2d)
    ]


def resolve_target_layer(
    model: nn.Module,
    requested_name: Optional[str] = None,
) -> Tuple[str, nn.Module]:
    named_modules = dict(model.named_modules())

    if requested_name is not None:
        if requested_name not in named_modules:
            available = list_convolutional_layers(model)
            raise ValueError(
                f"TARGET_LAYER_NAME='{requested_name}' was not found.\n"
                "Available convolutional layers include:\n- "
                + "\n- ".join(available[-30:])
            )

        target_layer = named_modules[requested_name]
        if not isinstance(target_layer, nn.Conv2d):
            raise TypeError(
                f"Selected target layer is not Conv2d: {requested_name}"
            )
        return requested_name, target_layer

    convolutional_layers = [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Conv2d)
    ]

    if not convolutional_layers:
        raise RuntimeError("No convolutional layer was found for Grad-CAM++.")

    return convolutional_layers[-1]


class SpatialActivationCapture:
    """Capture a layer output and retain its gradient for Grad-CAM++."""

    def __init__(self, layer: nn.Module):
        self.activations: Optional[torch.Tensor] = None
        self.forward_handle = layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inputs, output):
        if isinstance(output, (tuple, list)):
            output = output[0]

        self.activations = output

        # During Grad-CAM++ the tensor participates in autograd. Retaining its
        # gradient avoids fragile module-level backward hooks. During ordinary
        # no-gradient feature extraction this branch is skipped.
        if isinstance(output, torch.Tensor) and output.requires_grad:
            output.retain_grad()

    @property
    def gradients(self) -> Optional[torch.Tensor]:
        if self.activations is None:
            return None
        return self.activations.grad

    def clear(self):
        self.activations = None

    def close(self):
        self.forward_handle.remove()


# ============================================================
# 8. GRAD-CAM++, SALIENCY, AND REFINEMENT MAPS
# ============================================================

def ensure_spatial_tensor(tensor: torch.Tensor, name: str) -> torch.Tensor:
    if tensor is None:
        raise RuntimeError(f"{name} was not captured.")

    if tensor.ndim != 4:
        raise RuntimeError(
            f"{name} must be a four-dimensional NCHW tensor, got {tensor.shape}."
        )

    return tensor


def compute_gradcam_plus_plus(
    model: nn.Module,
    image_batch: torch.Tensor,
    target_class: int,
    capture: SpatialActivationCapture,
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute Grad-CAM++ and return the normalized map and probabilities."""
    model.zero_grad(set_to_none=True)
    capture.clear()

    # Gradients are more stable in float32, so AMP is intentionally disabled.
    outputs = model(image_batch, return_features=True)
    logits = outputs["logits"]
    probabilities = torch.softmax(logits.detach(), dim=1)[0].cpu().numpy()

    score = logits[0, target_class]
    score.backward(retain_graph=False)

    activations = ensure_spatial_tensor(capture.activations, "Activations")
    gradients = ensure_spatial_tensor(capture.gradients, "Gradients")

    activations = activations.detach().float()
    gradients = gradients.detach().float()

    gradient_squared = gradients.pow(2)
    gradient_cubed = gradient_squared * gradients

    spatial_activation_sum = activations.sum(dim=(2, 3), keepdim=True)

    denominator = (
        2.0 * gradient_squared
        + spatial_activation_sum * gradient_cubed
    )
    denominator = torch.where(
        denominator.abs() > 1e-8,
        denominator,
        torch.ones_like(denominator),
    )

    alpha = gradient_squared / denominator
    alpha = torch.where(gradients != 0.0, alpha, torch.zeros_like(alpha))

    positive_gradients = F.relu(gradients)
    weights = (alpha * positive_gradients).sum(dim=(2, 3), keepdim=True)

    cam = F.relu((weights * activations).sum(dim=1, keepdim=True))
    cam = F.interpolate(
        cam,
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )

    cam_array = cam[0, 0].detach().cpu().numpy()
    return normalize_map(cam_array), probabilities


def compute_input_saliency(
    model: nn.Module,
    image_batch: torch.Tensor,
    target_class: int,
) -> np.ndarray:
    """Compute absolute input-gradient saliency for the selected class."""
    input_tensor = image_batch.detach().clone().requires_grad_(True)
    model.zero_grad(set_to_none=True)

    logits = model(input_tensor)
    score = logits[0, target_class]
    score.backward()

    if input_tensor.grad is None:
        raise RuntimeError("Input gradient was not produced for saliency.")

    saliency = input_tensor.grad.detach().abs().amax(dim=1)[0]
    saliency = saliency.cpu().numpy()
    return normalize_map(saliency)


@torch.no_grad()
def compute_feature_refinement_maps(
    model: nn.Module,
    image_batch: torch.Tensor,
    capture: SpatialActivationCapture,
) -> Dict[str, np.ndarray]:
    """
    Build a GRFM-gate-guided spatial refinement visualization.

    Because GRFM consumes pooled vectors, this is an interpretation proxy:
    the last spatial backbone activation is channel-weighted by the learned
    GRFM gate for the same image.
    """
    capture.clear()
    outputs = model(image_batch, return_features=True)

    spatial_features = ensure_spatial_tensor(
        capture.activations,
        "Spatial backbone activations",
    ).detach().float()

    gate = outputs["gate"].detach().float()

    feature_tensor = spatial_features[0]
    channels, feature_height, feature_width = feature_tensor.shape

    base_map = feature_tensor.abs().mean(dim=0)

    gate_vector = gate[0]
    if gate_vector.numel() != channels:
        gate_vector = F.interpolate(
            gate_vector.view(1, 1, -1),
            size=channels,
            mode="linear",
            align_corners=False,
        ).view(-1)

    # Normalize weights to preserve a comparable scale.
    gate_vector = gate_vector.clamp(min=0.0)
    gate_weights = gate_vector / (gate_vector.mean() + 1e-8)

    refined_map = (
        feature_tensor.abs()
        * gate_weights.view(channels, 1, 1)
    ).mean(dim=0)

    # Absolute modulation effect highlights where gating changed emphasis.
    refinement_change = (refined_map - base_map).abs()

    base_map = F.interpolate(
        base_map.view(1, 1, feature_height, feature_width),
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )[0, 0]

    refined_map = F.interpolate(
        refined_map.view(1, 1, feature_height, feature_width),
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )[0, 0]

    refinement_change = F.interpolate(
        refinement_change.view(1, 1, feature_height, feature_width),
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )[0, 0]

    return {
        "backbone_activation": normalize_map(base_map.cpu().numpy()),
        "grfm_refined": normalize_map(refined_map.cpu().numpy()),
        "refinement_change": normalize_map(refinement_change.cpu().numpy()),
        "gate_mean": float(gate_vector.mean().cpu().item()),
        "gate_std": float(gate_vector.std(unbiased=False).cpu().item()),
    }


# ============================================================
# 9. SAMPLE SELECTION
# ============================================================

def select_three_samples_per_class(
    labels: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    class_names: Sequence[str],
    samples_per_class: int = SAMPLES_PER_CLASS,
) -> Dict[int, List[int]]:
    confidence = probabilities.max(axis=1)
    selected: Dict[int, List[int]] = {}

    for class_index, class_name in enumerate(class_names):
        class_indices = np.where(labels == class_index)[0]

        if len(class_indices) == 0:
            print(f"Warning: class '{class_name}' has no test images.")
            selected[class_index] = []
            continue

        chosen: List[int] = []

        if PREFER_CORRECT_EXPLANATION_SAMPLES:
            correct_indices = class_indices[
                predictions[class_indices] == class_index
            ]
            correct_indices = correct_indices[
                np.argsort(confidence[correct_indices])[::-1]
            ]
            chosen.extend(correct_indices[:samples_per_class].tolist())

        if len(chosen) < samples_per_class:
            remaining = np.asarray(
                [index for index in class_indices if index not in chosen],
                dtype=np.int64,
            )
            remaining = remaining[np.argsort(confidence[remaining])[::-1]]
            needed = samples_per_class - len(chosen)
            chosen.extend(remaining[:needed].tolist())

        selected[class_index] = chosen

    return selected


# ============================================================
# 10. EXPLANATION PANELS: THREE IMAGES PER CLASS
# ============================================================

def generate_explanation_panels(
    model: nn.Module,
    dataset: datasets.ImageFolder,
    class_names: Sequence[str],
    extracted: Dict[str, np.ndarray],
    capture: SpatialActivationCapture,
) -> pd.DataFrame:
    labels = extracted["labels"]
    predictions = extracted["predictions"]
    probabilities = extracted["probabilities"]
    image_paths = extracted["image_paths"]

    selected_by_class = select_three_samples_per_class(
        labels=labels,
        predictions=predictions,
        probabilities=probabilities,
        class_names=class_names,
    )

    panel_dir = VISUALIZATION_DIR / "per_class_explanation_panels"
    individual_dir = VISUALIZATION_DIR / "individual_explanations"
    panel_dir.mkdir(parents=True, exist_ok=True)
    individual_dir.mkdir(parents=True, exist_ok=True)

    records = []

    column_titles = [
        "Original",
        "Backbone activation",
        "GRFM-guided refinement",
        "Refinement change",
        "Grad-CAM++",
        "Saliency map",
    ]

    for class_index, class_name in enumerate(class_names):
        selected_indices = selected_by_class[class_index]

        if not selected_indices:
            continue

        number_of_rows = len(selected_indices)
        figure, axes = plt.subplots(
            number_of_rows,
            len(column_titles),
            figsize=(19, 4.3 * number_of_rows),
            squeeze=False,
        )

        for column_index, title in enumerate(column_titles):
            axes[0, column_index].set_title(title, fontsize=12, fontweight="bold")

        class_output_dir = individual_dir / sanitize_filename(class_name)
        class_output_dir.mkdir(parents=True, exist_ok=True)

        for row_index, dataset_index in enumerate(selected_indices):
            image_tensor, true_label = dataset[dataset_index]
            image_batch = image_tensor.unsqueeze(0).to(DEVICE)
            rgb_image = tensor_to_rgb(image_tensor)

            predicted_label = int(predictions[dataset_index])
            predicted_confidence = float(probabilities[dataset_index, predicted_label])
            true_probability = float(probabilities[dataset_index, true_label])

            if EXPLANATION_TARGET.lower() == "predicted":
                target_class = predicted_label
            elif EXPLANATION_TARGET.lower() == "true":
                target_class = int(true_label)
            else:
                raise ValueError(
                    "EXPLANATION_TARGET must be either 'true' or 'predicted'."
                )

            refinement = compute_feature_refinement_maps(
                model=model,
                image_batch=image_batch,
                capture=capture,
            )

            gradcam_map, _ = compute_gradcam_plus_plus(
                model=model,
                image_batch=image_batch,
                target_class=target_class,
                capture=capture,
            )

            saliency_map = compute_input_saliency(
                model=model,
                image_batch=image_batch,
                target_class=target_class,
            )

            backbone_overlay = overlay_heatmap(
                rgb_image,
                refinement["backbone_activation"],
            )
            refined_overlay = overlay_heatmap(
                rgb_image,
                refinement["grfm_refined"],
            )
            change_overlay = overlay_heatmap(
                rgb_image,
                refinement["refinement_change"],
                cmap_name="magma",
            )
            gradcam_overlay = overlay_heatmap(rgb_image, gradcam_map)
            saliency_overlay = overlay_heatmap(
                rgb_image,
                saliency_map,
                cmap_name="inferno",
            )

            displayed_images = [
                rgb_image,
                backbone_overlay,
                refined_overlay,
                change_overlay,
                gradcam_overlay,
                saliency_overlay,
            ]

            for column_index, shown_image in enumerate(displayed_images):
                axes[row_index, column_index].imshow(shown_image)
                axes[row_index, column_index].axis("off")

            status = "Correct" if predicted_label == true_label else "Misclassified"
            axes[row_index, 0].set_ylabel(
                f"True: {class_names[true_label]}\n"
                f"Pred: {class_names[predicted_label]}\n"
                f"Conf: {predicted_confidence:.3f}\n{status}",
                fontsize=10,
            )

            base_name = f"sample_{row_index + 1:02d}_index_{dataset_index:05d}"
            save_rgb_image(rgb_image, class_output_dir / f"{base_name}_original.png")
            save_rgb_image(
                backbone_overlay,
                class_output_dir / f"{base_name}_backbone_activation.png",
            )
            save_rgb_image(
                refined_overlay,
                class_output_dir / f"{base_name}_grfm_refined.png",
            )
            save_rgb_image(
                change_overlay,
                class_output_dir / f"{base_name}_refinement_change.png",
            )
            save_rgb_image(
                gradcam_overlay,
                class_output_dir / f"{base_name}_gradcam_plus_plus.png",
            )
            save_rgb_image(
                saliency_overlay,
                class_output_dir / f"{base_name}_saliency.png",
            )

            records.append({
                "Dataset Index": int(dataset_index),
                "Image Path": image_paths[dataset_index],
                "True Class": class_names[true_label],
                "Predicted Class": class_names[predicted_label],
                "Correct": int(predicted_label == true_label),
                "Predicted Confidence": predicted_confidence,
                "True-Class Probability": true_probability,
                "Explanation Target": class_names[target_class],
                "GRFM Gate Mean": refinement["gate_mean"],
                "GRFM Gate Std": refinement["gate_std"],
            })

        figure.suptitle(
            f"{MODEL_DISPLAY_NAME}: visual explanations for class '{class_name}'",
            fontsize=16,
            fontweight="bold",
            y=1.01,
        )
        figure.tight_layout()
        figure.savefig(
            panel_dir / f"{sanitize_filename(class_name)}_three_images.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(figure)

    records_dataframe = pd.DataFrame(records)
    records_dataframe.to_csv(
        VISUALIZATION_DIR / "selected_explanation_samples.csv",
        index=False,
    )

    return records_dataframe


# ============================================================
# 11. t-SNE BEFORE AND AFTER GRFM
# ============================================================

def stratified_tsne_indices(
    labels: np.ndarray,
    max_samples_per_class: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    selected_indices = []

    for class_index in np.unique(labels):
        class_indices = np.where(labels == class_index)[0]
        if len(class_indices) > max_samples_per_class:
            class_indices = rng.choice(
                class_indices,
                size=max_samples_per_class,
                replace=False,
            )
        selected_indices.extend(class_indices.tolist())

    selected_indices = np.asarray(selected_indices, dtype=np.int64)
    rng.shuffle(selected_indices)
    return selected_indices


def prepare_features_for_tsne(features: np.ndarray) -> np.ndarray:
    features = np.asarray(features, dtype=np.float64)
    features = StandardScaler().fit_transform(features)

    maximum_components = min(
        TSNE_PCA_COMPONENTS,
        features.shape[0] - 1,
        features.shape[1],
    )

    if maximum_components >= 2 and features.shape[1] > maximum_components:
        features = PCA(
            n_components=maximum_components,
            random_state=SEED,
        ).fit_transform(features)

    return features


def run_tsne(features: np.ndarray) -> np.ndarray:
    number_of_samples = len(features)

    if number_of_samples < 3:
        raise ValueError("At least three samples are required for t-SNE.")

    perplexity = min(TSNE_PERPLEXITY, max(2, number_of_samples // 4))
    perplexity = min(perplexity, number_of_samples - 1)

    # sklearn changed n_iter to max_iter in newer versions. This fallback
    # keeps the script compatible with both APIs.
    common_arguments = {
        "n_components": 2,
        "perplexity": perplexity,
        "learning_rate": "auto",
        "init": "pca",
        "random_state": SEED,
        "method": "barnes_hut",
        "angle": 0.5,
    }

    try:
        tsne = TSNE(
            **common_arguments,
            max_iter=TSNE_ITERATIONS,
        )
    except TypeError:
        tsne = TSNE(
            **common_arguments,
            n_iter=TSNE_ITERATIONS,
        )

    return tsne.fit_transform(features)


def plot_tsne_embedding(
    embedding: np.ndarray,
    labels: np.ndarray,
    class_names: Sequence[str],
    title: str,
    filename: str,
) -> None:
    figure = plt.figure(figsize=(9, 7))

    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        if not np.any(mask):
            continue

        plt.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            s=28,
            alpha=0.78,
            label=f"{class_name} (n={int(mask.sum())})",
        )

    plt.xlabel("t-SNE component 1")
    plt.ylabel("t-SNE component 2")
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.25)
    plt.legend(loc="best", fontsize=9)
    plt.tight_layout()
    figure.savefig(VISUALIZATION_DIR / filename, dpi=300, bbox_inches="tight")
    plt.close(figure)


def generate_tsne_plots(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    labels = extracted["labels"]
    selected_indices = stratified_tsne_indices(
        labels=labels,
        max_samples_per_class=TSNE_MAX_SAMPLES_PER_CLASS,
        seed=SEED,
    )

    selected_labels = labels[selected_indices]
    backbone_features = extracted["backbone_features"][selected_indices]
    refined_features = extracted["refined_features"][selected_indices]

    print(
        f"Running t-SNE on {len(selected_indices)} stratified test samples..."
    )

    backbone_prepared = prepare_features_for_tsne(backbone_features)
    refined_prepared = prepare_features_for_tsne(refined_features)

    backbone_embedding = run_tsne(backbone_prepared)
    refined_embedding = run_tsne(refined_prepared)

    plot_tsne_embedding(
        embedding=backbone_embedding,
        labels=selected_labels,
        class_names=class_names,
        title=f"{MODEL_DISPLAY_NAME}: backbone feature t-SNE",
        filename="tsne_backbone_features.png",
    )

    plot_tsne_embedding(
        embedding=refined_embedding,
        labels=selected_labels,
        class_names=class_names,
        title=f"{MODEL_DISPLAY_NAME}: GRFM-refined feature t-SNE",
        filename="tsne_grfm_refined_features.png",
    )

    comparison_figure, axes = plt.subplots(1, 2, figsize=(18, 7))

    for class_index, class_name in enumerate(class_names):
        mask = selected_labels == class_index
        if not np.any(mask):
            continue

        axes[0].scatter(
            backbone_embedding[mask, 0],
            backbone_embedding[mask, 1],
            s=25,
            alpha=0.75,
            label=class_name,
        )
        axes[1].scatter(
            refined_embedding[mask, 0],
            refined_embedding[mask, 1],
            s=25,
            alpha=0.75,
            label=class_name,
        )

    axes[0].set_title("Before GRFM: backbone vectors")
    axes[1].set_title("After GRFM: refined vectors")

    for axis in axes:
        axis.set_xlabel("t-SNE component 1")
        axis.set_ylabel("t-SNE component 2")
        axis.grid(True, linestyle="--", alpha=0.25)
        axis.legend(loc="best", fontsize=8)

    comparison_figure.suptitle(
        f"{MODEL_DISPLAY_NAME}: feature-space refinement comparison",
        fontsize=16,
        fontweight="bold",
    )
    comparison_figure.tight_layout()
    comparison_figure.savefig(
        VISUALIZATION_DIR / "tsne_before_after_grfm_comparison.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(comparison_figure)

    tsne_dataframe = pd.DataFrame({
        "Dataset Index": selected_indices,
        "True Label": selected_labels,
        "True Class": [class_names[index] for index in selected_labels],
        "Backbone t-SNE 1": backbone_embedding[:, 0],
        "Backbone t-SNE 2": backbone_embedding[:, 1],
        "GRFM t-SNE 1": refined_embedding[:, 0],
        "GRFM t-SNE 2": refined_embedding[:, 1],
    })

    tsne_dataframe.to_csv(
        VISUALIZATION_DIR / "tsne_coordinates.csv",
        index=False,
    )

    return tsne_dataframe


# ============================================================
# 12. MISCLASSIFIED IMAGE ANALYSIS
# ============================================================

def create_misclassification_dataframe(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    labels = extracted["labels"]
    predictions = extracted["predictions"]
    probabilities = extracted["probabilities"]
    image_paths = extracted["image_paths"]

    misclassified_indices = np.where(predictions != labels)[0]
    rows = []

    for index in misclassified_indices:
        true_label = int(labels[index])
        predicted_label = int(predictions[index])
        sorted_probabilities = np.sort(probabilities[index])[::-1]
        margin = (
            float(sorted_probabilities[0] - sorted_probabilities[1])
            if probabilities.shape[1] > 1
            else float(sorted_probabilities[0])
        )

        row = {
            "Dataset Index": int(index),
            "Image Path": image_paths[index],
            "True Label": true_label,
            "True Class": class_names[true_label],
            "Predicted Label": predicted_label,
            "Predicted Class": class_names[predicted_label],
            "Predicted Confidence": float(probabilities[index, predicted_label]),
            "True-Class Probability": float(probabilities[index, true_label]),
            "Confidence Margin": margin,
        }

        for class_index, class_name in enumerate(class_names):
            row[f"Probability {class_name}"] = float(
                probabilities[index, class_index]
            )

        rows.append(row)

    dataframe = pd.DataFrame(rows)

    if not dataframe.empty:
        dataframe = dataframe.sort_values(
            "Predicted Confidence",
            ascending=False,
        ).reset_index(drop=True)

    dataframe.to_csv(
        VISUALIZATION_DIR / "misclassified_images.csv",
        index=False,
    )

    return dataframe


def plot_misclassified_grid(
    dataset: datasets.ImageFolder,
    misclassified_dataframe: pd.DataFrame,
    class_names: Sequence[str],
    filename: str,
    title: str,
    max_images: int,
) -> None:
    if misclassified_dataframe.empty:
        # A zero-error class is a valid result, not a pipeline failure.
        # Save an explicit placeholder so every requested output exists.
        figure, axis = plt.subplots(figsize=(8, 4.5))
        axis.axis("off")
        axis.text(
            0.5,
            0.56,
            "No misclassified images",
            ha="center",
            va="center",
            fontsize=18,
            fontweight="bold",
            transform=axis.transAxes,
        )
        axis.text(
            0.5,
            0.40,
            title,
            ha="center",
            va="center",
            fontsize=11,
            wrap=True,
            transform=axis.transAxes,
        )
        figure.tight_layout()
        figure.savefig(
            VISUALIZATION_DIR / filename,
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(figure)
        print(f"Zero misclassifications: {title}")
        return

    shown_dataframe = misclassified_dataframe.head(max_images)
    number_of_images = len(shown_dataframe)
    columns = min(4, number_of_images)
    rows = int(math.ceil(number_of_images / columns))

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(4.2 * columns, 4.2 * rows),
        squeeze=False,
    )
    axes_flat = axes.ravel()

    for axis in axes_flat:
        axis.axis("off")

    for plot_index, (_, row) in enumerate(shown_dataframe.iterrows()):
        dataset_index = int(row["Dataset Index"])
        image_tensor, _ = dataset[dataset_index]
        rgb_image = tensor_to_rgb(image_tensor)

        axes_flat[plot_index].imshow(rgb_image)
        axes_flat[plot_index].set_title(
            f"True: {row['True Class']}\n"
            f"Pred: {row['Predicted Class']}\n"
            f"Confidence: {row['Predicted Confidence']:.3f}",
            fontsize=10,
        )
        axes_flat[plot_index].axis("off")

    figure.suptitle(title, fontsize=16, fontweight="bold")
    figure.tight_layout()
    figure.savefig(
        VISUALIZATION_DIR / filename,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(figure)


def generate_misclassification_outputs(
    dataset: datasets.ImageFolder,
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    dataframe = create_misclassification_dataframe(
        extracted=extracted,
        class_names=class_names,
    )

    total_images = len(extracted["labels"])
    total_errors = len(dataframe)
    accuracy = accuracy_score(
        extracted["labels"],
        extracted["predictions"],
    )

    print(f"Test accuracy       : {accuracy:.6f}")
    print(f"Misclassified images: {total_errors}/{total_images}")

    plot_misclassified_grid(
        dataset=dataset,
        misclassified_dataframe=dataframe,
        class_names=class_names,
        filename="misclassified_highest_confidence_grid.png",
        title=(
            f"{MODEL_DISPLAY_NAME}: highest-confidence misclassified images"
        ),
        max_images=MAX_MISCLASSIFIED_GRID_IMAGES,
    )

    for class_name in class_names:
        class_errors = dataframe[
            dataframe["True Class"] == class_name
        ] if not dataframe.empty else dataframe

        plot_misclassified_grid(
            dataset=dataset,
            misclassified_dataframe=class_errors,
            class_names=class_names,
            filename=(
                "misclassified_true_class_"
                f"{sanitize_filename(class_name)}.png"
            ),
            title=(
                f"{MODEL_DISPLAY_NAME}: errors with true class '{class_name}'"
            ),
            max_images=MAX_MISCLASSIFIED_PER_TRUE_CLASS,
        )

    return dataframe


# ============================================================
# 13. OPTIONAL: FEATURE-GATE SUMMARY
# ============================================================

def plot_gate_summary(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    gates = extracted["gates"]
    labels = extracted["labels"]

    rows = []
    class_gate_means = []

    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        if not np.any(mask):
            continue

        mean_gate = gates[mask].mean(axis=0)
        class_gate_means.append((class_name, mean_gate))

        rows.append({
            "Class": class_name,
            "Images": int(mask.sum()),
            "Mean Gate Value": float(gates[mask].mean()),
            "Gate Standard Deviation": float(gates[mask].std()),
            "Minimum Gate Value": float(gates[mask].min()),
            "Maximum Gate Value": float(gates[mask].max()),
        })

    summary_dataframe = pd.DataFrame(rows)
    summary_dataframe.to_csv(
        VISUALIZATION_DIR / "grfm_gate_summary_by_class.csv",
        index=False,
    )

    if class_gate_means:
        matrix = np.stack([item[1] for item in class_gate_means], axis=0)
        names = [item[0] for item in class_gate_means]

        figure = plt.figure(figsize=(14, max(4, 0.8 * len(names))))
        plt.imshow(matrix, aspect="auto", interpolation="nearest", cmap="viridis")
        plt.colorbar(label="Mean GRFM gate value")
        plt.yticks(np.arange(len(names)), names)
        plt.xlabel("GRFM feature dimension")
        plt.ylabel("True class")
        plt.title(f"{MODEL_DISPLAY_NAME}: class-wise GRFM gate profiles")
        plt.tight_layout()
        figure.savefig(
            VISUALIZATION_DIR / "grfm_gate_profile_heatmap.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(figure)

    return summary_dataframe


# ============================================================
# 14. MAIN
# ============================================================

def main() -> None:
    set_seed(SEED)
    validate_inputs()

    print("=" * 84)
    print(f"{MODEL_DISPLAY_NAME} visual explanation pipeline")
    print("=" * 84)
    print(f"Test directory    : {TEST_DIR}")
    print(f"Output directory  : {VISUALIZATION_DIR}")
    print(f"Images per class  : {SAMPLES_PER_CLASS}")
    print(f"Explanation target: {EXPLANATION_TARGET}")

    test_dataset, test_loader = create_test_dataset_and_loader()
    model, class_names, checkpoint_path = load_checkpoint_and_model(
        test_dataset.classes
    )

    target_layer_name, target_layer = resolve_target_layer(
        model,
        TARGET_LAYER_NAME,
    )
    print(f"Grad-CAM++ layer  : {target_layer_name}")

    # Save the selected layer name for reproducibility.
    with (VISUALIZATION_DIR / "selected_target_layer.txt").open(
        "w",
        encoding="utf-8",
    ) as file:
        file.write(target_layer_name + "\n")

    capture = SpatialActivationCapture(target_layer)

    try:
        print("\nExtracting predictions and feature vectors...")
        extracted = extract_predictions_and_features(
            model=model,
            loader=test_loader,
            dataset=test_dataset,
        )

        print("\nGenerating t-SNE plots...")
        generate_tsne_plots(
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating misclassification outputs...")
        generate_misclassification_outputs(
            dataset=test_dataset,
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating class-wise GRFM gate summaries...")
        plot_gate_summary(
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating three-image explanation panels per class...")
        generate_explanation_panels(
            model=model,
            dataset=test_dataset,
            class_names=class_names,
            extracted=extracted,
            capture=capture,
        )

    finally:
        capture.close()

    print("\nCompleted successfully")
    print("=" * 84)
    print(f"Checkpoint used: {checkpoint_path}")
    print(f"All outputs are in: {VISUALIZATION_DIR}")
    print("\nMain generated files:")
    print("- tsne_backbone_features.png")
    print("- tsne_grfm_refined_features.png")
    print("- tsne_before_after_grfm_comparison.png")
    print("- misclassified_images.csv")
    print("- misclassified_highest_confidence_grid.png")
    print("- grfm_gate_profile_heatmap.png")
    print("- per_class_explanation_panels/<class>_three_images.png")
    print("- individual_explanations/<class>/*.png")


if __name__ == "__main__":
    main()


In [ ]:
"""
MaxViT-Tiny + GRFM visual explanation pipeline
==============================================

This script loads a trained MaxViT-Tiny + GRFM checkpoint and generates:

1. GRFM-guided feature-refinement maps.
2. t-SNE plots before and after GRFM refinement.
3. Misclassified-image grids and a detailed CSV report.
4. Grad-CAM++ visualizations.
5. Input-gradient saliency maps.
6. Three representative images per class for explanation panels.

The script does NOT retrain the model.

Important scientific note
-------------------------
GRFM operates on globally pooled feature vectors rather than a spatial tensor.
Therefore, the "GRFM-guided refined feature map" is a spatial visualization
proxy: the last spatial backbone feature tensor is weighted by the learned GRFM
gate. This is useful for interpretation, but it should not be described as an
exact spatial output produced directly by GRFM.
"""

import contextlib
import copy
import math
import os
import random
import re
import subprocess
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

try:
    import timm
except ImportError:
    print("timm is not installed. Installing timm...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from IPython.display import display, Image as IPythonImage, Markdown
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.preprocessing import label_binarize, StandardScaler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

# Update these paths if needed.
TRAIN_DIR = Path("/kaggle/working/PVMD_Train_Augmented")
VAL_DIR = Path("/kaggle/working/PVMD_Split/validation")
TEST_DIR = Path("/kaggle/working/PVMD_Split/test")

if not VAL_DIR.exists():
    alternative_val = Path("/kaggle/working/PVMD_Split/val")
    if alternative_val.exists():
        VAL_DIR = alternative_val

TRAINING_OUTPUT_DIR = Path("/kaggle/working/MaxViT_Tiny_GRFM_Results")
FINAL_MODEL_PATH = TRAINING_OUTPUT_DIR / "final_maxvit_tiny_grfm.pth"
BEST_MODEL_PATH = TRAINING_OUTPUT_DIR / "best_maxvit_tiny_grfm.pth"

VISUALIZATION_DIR = TRAINING_OUTPUT_DIR / "Visual_Explanations"
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "maxvit_tiny_tf_224.in1k"
MODEL_DISPLAY_NAME = "MaxViT-Tiny-224 + GRFM"

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

GRFM_DIM = 512
GRFM_DROPOUT = 0.30
GRFM_REDUCTION = 4

# Three images per class, as requested.
SAMPLES_PER_CLASS = 3

# Misclassification output controls.
MAX_MISCLASSIFIED_GRID_IMAGES = 24
MAX_MISCLASSIFIED_PER_TRUE_CLASS = 3

# t-SNE controls. Limiting samples avoids excessive runtime and memory use.
TSNE_MAX_SAMPLES_PER_CLASS = 250
TSNE_PCA_COMPONENTS = 50
TSNE_PERPLEXITY = 30
TSNE_ITERATIONS = 1500

# Grad-CAM++ target layer.
# Leave as None to automatically use the final convolutional layer.
# To select one manually, enter its exact model.named_modules() name.
TARGET_LAYER_NAME: Optional[str] = None

# Explanation target:
# "true"      -> explain the ground-truth class score.
# "predicted" -> explain the model-predicted class score.
EXPLANATION_TARGET = "true"

# Select correctly classified images for the three-per-class panels whenever
# possible. If a class has fewer than three correct samples, remaining samples
# are filled using the most confident available images from that class.
PREFER_CORRECT_EXPLANATION_SAMPLES = True

HEATMAP_ALPHA = 0.45

# Explicitly display generated figures in Kaggle/Jupyter while also saving.
SHOW_PLOTS_INLINE = True

# Evaluation-plot controls.
CONFIDENCE_BINS = 20
DISPLAY_LARGE_EXPLANATION_PANELS = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP_FOR_FEATURE_EXTRACTION = DEVICE.type == "cuda"

IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)


# ============================================================
# 2. REPRODUCIBILITY AND HELPERS
# ============================================================

def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def amp_context():
    if USE_AMP_FOR_FEATURE_EXTRACTION:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()


def sanitize_filename(text: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text).strip())
    return cleaned.strip("_") or "class"


def normalize_map(array: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    array = np.asarray(array, dtype=np.float32)
    array = np.nan_to_num(array, nan=0.0, posinf=0.0, neginf=0.0)
    minimum = float(array.min())
    maximum = float(array.max())

    if maximum - minimum < eps:
        return np.zeros_like(array, dtype=np.float32)

    return (array - minimum) / (maximum - minimum)


def tensor_to_rgb(image_tensor: torch.Tensor) -> np.ndarray:
    """Undo ImageNet normalization and return an RGB float image in [0, 1]."""
    image = image_tensor.detach().cpu().float().numpy().transpose(1, 2, 0)
    image = image * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(image, 0.0, 1.0)


def resize_heatmap(heatmap: np.ndarray, output_size: Tuple[int, int]) -> np.ndarray:
    heatmap = normalize_map(heatmap)
    heatmap_uint8 = np.uint8(np.round(heatmap * 255.0))
    resized = Image.fromarray(heatmap_uint8).resize(
        output_size,
        resample=Image.Resampling.BILINEAR,
    )
    return np.asarray(resized, dtype=np.float32) / 255.0


def heatmap_to_rgb(heatmap: np.ndarray, cmap_name: str = "jet") -> np.ndarray:
    heatmap = normalize_map(heatmap)
    color_map = matplotlib.colormaps.get_cmap(cmap_name)
    return color_map(heatmap)[..., :3].astype(np.float32)


def overlay_heatmap(
    rgb_image: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = HEATMAP_ALPHA,
    cmap_name: str = "jet",
) -> np.ndarray:
    height, width = rgb_image.shape[:2]
    resized_heatmap = resize_heatmap(heatmap, (width, height))
    colored_heatmap = heatmap_to_rgb(resized_heatmap, cmap_name=cmap_name)
    overlay = (1.0 - alpha) * rgb_image + alpha * colored_heatmap
    return np.clip(overlay, 0.0, 1.0)


def save_rgb_image(image: np.ndarray, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    image_uint8 = np.uint8(np.round(np.clip(image, 0.0, 1.0) * 255.0))
    Image.fromarray(image_uint8).save(path)


def save_and_display_figure(
    figure,
    path: Path,
    dpi: int = 300,
    display_inline: bool = True,
) -> None:
    """
    Save a Matplotlib figure and explicitly display it in Kaggle/Jupyter.

    ``IPython.display.display`` is used instead of relying only on
    ``plt.show()``, making notebook rendering more reliable.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    figure.tight_layout()
    figure.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight",
    )

    if SHOW_PLOTS_INLINE and display_inline:
        display(figure)

    plt.close(figure)
    print(f"Saved and displayed: {path}")



def display_saved_png(
    path: Path,
    title: Optional[str] = None,
    width: Optional[int] = None,
) -> None:
    """
    Explicitly display a saved PNG/JPEG in Kaggle or Jupyter.

    This is useful for figures that were already closed after saving or for
    individual explanation images created directly with Pillow.
    """
    path = Path(path)

    if not path.exists():
        print(f"Display warning: file not found: {path}")
        return

    if title:
        display(Markdown(f"### {title}"))

    display(
        IPythonImage(
            filename=str(path),
            width=width,
        )
    )


def display_csv_table(
    path: Path,
    title: str,
    max_rows: int = 100,
) -> Optional[pd.DataFrame]:
    """
    Read and explicitly display a generated CSV as a notebook table.
    """
    path = Path(path)

    if not path.exists():
        print(f"Display warning: CSV not found: {path}")
        return None

    dataframe = pd.read_csv(path)

    display(Markdown(f"### {title}"))

    if dataframe.empty:
        display(
            pd.DataFrame({
                "Result": [
                    "No records were generated."
                ]
            })
        )
    else:
        display(
            dataframe.head(max_rows)
        )

    return dataframe


def display_generated_visual_outputs(
    class_names: Sequence[str],
) -> None:
    """
    Display all requested visual-explanation outputs after they are generated.

    This includes:
      - misclassified_images.csv
      - misclassified_highest_confidence_grid.png
      - grfm_gate_profile_heatmap.png
      - every per-class three-image panel
      - every individual explanation PNG
    """
    display(
        Markdown(
            "## Generated visual-explanation outputs"
        )
    )

    # ----------------------------------------------------------
    # Misclassification CSV
    # ----------------------------------------------------------
    display_csv_table(
        VISUALIZATION_DIR
        / "misclassified_images.csv",
        title="Misclassified images",
        max_rows=100,
    )

    # ----------------------------------------------------------
    # Main misclassification grid
    # ----------------------------------------------------------
    display_saved_png(
        VISUALIZATION_DIR
        / "misclassified_highest_confidence_grid.png",
        title="Highest-confidence misclassified images",
        width=1100,
    )

    # Also display class-specific misclassification grids.
    for class_name in class_names:
        class_grid = (
            VISUALIZATION_DIR
            / (
                "misclassified_true_class_"
                f"{sanitize_filename(class_name)}.png"
            )
        )

        display_saved_png(
            class_grid,
            title=(
                "Misclassifications with true class: "
                f"{class_name}"
            ),
            width=1000,
        )

    # ----------------------------------------------------------
    # GRFM gate heatmap
    # ----------------------------------------------------------
    display_saved_png(
        VISUALIZATION_DIR
        / "grfm_gate_profile_heatmap.png",
        title="Class-wise GRFM gate profile heatmap",
        width=1200,
    )

    display_csv_table(
        VISUALIZATION_DIR
        / "grfm_gate_summary_by_class.csv",
        title="GRFM gate summary by class",
        max_rows=100,
    )

    # ----------------------------------------------------------
    # Per-class explanation panels
    # ----------------------------------------------------------
    panel_directory = (
        VISUALIZATION_DIR
        / "per_class_explanation_panels"
    )

    display(
        Markdown(
            "## Per-class explanation panels"
        )
    )

    for class_name in class_names:
        panel_path = (
            panel_directory
            / (
                f"{sanitize_filename(class_name)}"
                "_three_images.png"
            )
        )

        display_saved_png(
            panel_path,
            title=(
                f"{class_name}: three-image "
                "explanation panel"
            ),
            width=1500,
        )

    # ----------------------------------------------------------
    # Selected explanation sample table
    # ----------------------------------------------------------
    display_csv_table(
        VISUALIZATION_DIR
        / "selected_explanation_samples.csv",
        title="Selected explanation samples",
        max_rows=200,
    )

    # ----------------------------------------------------------
    # Individual explanation images
    # ----------------------------------------------------------
    individual_root = (
        VISUALIZATION_DIR
        / "individual_explanations"
    )

    display(
        Markdown(
            "## Individual explanation images"
        )
    )

    for class_name in class_names:
        class_directory = (
            individual_root
            / sanitize_filename(class_name)
        )

        display(
            Markdown(
                f"### {class_name}"
            )
        )

        if not class_directory.exists():
            print(
                "Display warning: directory "
                f"not found: {class_directory}"
            )
            continue

        image_paths = sorted(
            list(
                class_directory.glob(
                    "*.png"
                )
            )
        )

        if not image_paths:
            print(
                "No individual explanation "
                f"images found for {class_name}."
            )
            continue

        for image_path in image_paths:
            display_saved_png(
                image_path,
                title=image_path.stem.replace(
                    "_",
                    " ",
                ),
                width=650,
            )


def validate_inputs() -> None:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")

    if not FINAL_MODEL_PATH.exists() and not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(
            "No checkpoint was found. Expected one of:\n"
            f"- {FINAL_MODEL_PATH}\n"
            f"- {BEST_MODEL_PATH}"
        )


# ============================================================
# 3. DATASET
# ============================================================

evaluation_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN.tolist(),
        std=IMAGENET_STD.tolist(),
    ),
])


def create_test_dataset_and_loader():
    dataset = datasets.ImageFolder(
        root=str(TEST_DIR),
        transform=evaluation_transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

    return dataset, loader


# ============================================================
# 4. GRFM AND MODEL DEFINITIONS
# ============================================================

class GRFM(nn.Module):
    """Gated Residual Feature Modulation."""

    def __init__(
        self,
        input_dim: int,
        output_dim: int = 512,
        reduction: int = 4,
        dropout: float = 0.30,
    ):
        super().__init__()

        hidden_dim = max(output_dim // reduction, 32)

        self.projection = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

        self.feature_transform = nn.Sequential(
            nn.Linear(output_dim, output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, output_dim),
        )

        self.feature_gate = nn.Sequential(
            nn.Linear(output_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

        self.output_norm = nn.LayerNorm(output_dim)
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_gate: bool = False):
        projected_feature = self.projection(x)
        transformed_feature = self.feature_transform(projected_feature)
        modulation_gate = self.feature_gate(projected_feature)

        modulated_feature = transformed_feature * modulation_gate
        fused_feature = projected_feature + modulated_feature
        fused_feature = self.output_norm(fused_feature)
        fused_feature = self.output_dropout(fused_feature)

        if return_gate:
            return fused_feature, modulation_gate

        return fused_feature


class MaxViTTinyGRFM(nn.Module):
    def __init__(
        self,
        num_classes: int,
        grfm_dim: int = GRFM_DIM,
        dropout: float = GRFM_DROPOUT,
    ):
        super().__init__()

        # pretrained=False is essential here because checkpoint weights are loaded.
        self.backbone = timm.create_model(
            MODEL_NAME,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )

        backbone_feature_dim = self.backbone.num_features

        self.grfm = GRFM(
            input_dim=backbone_feature_dim,
            output_dim=grfm_dim,
            reduction=GRFM_REDUCTION,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(grfm_dim, grfm_dim // 2),
            nn.LayerNorm(grfm_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(grfm_dim // 2, num_classes),
        )

        self.backbone_feature_dim = backbone_feature_dim

    def forward(
        self,
        images: torch.Tensor,
        return_features: bool = False,
        return_gate: bool = False,
    ):
        backbone_features = self.backbone(images)
        grfm_features, gate = self.grfm(backbone_features, return_gate=True)
        logits = self.classifier(grfm_features)

        if return_features or return_gate:
            return {
                "logits": logits,
                "backbone_features": backbone_features,
                "grfm_features": grfm_features,
                "gate": gate,
            }

        return logits


# ============================================================
# 5. CHECKPOINT LOADING
# ============================================================

THOP_BUFFER_NAMES = {"total_ops", "total_params"}


def remove_thop_buffers_from_model(model: nn.Module) -> None:
    """Remove profiling-only buffers that THOP may register on modules."""
    for module in model.modules():
        for buffer_name in tuple(module._buffers.keys()):
            if buffer_name in THOP_BUFFER_NAMES:
                module._buffers.pop(buffer_name, None)


def clean_checkpoint_state_dict(state_dict: dict) -> dict:
    """
    Remove THOP-only keys and convert inference tensors to ordinary CPU tensors.

    This supports both new clean checkpoints and older checkpoints that were
    accidentally saved after THOP profiling.
    """
    cleaned_state = {}

    for key, value in state_dict.items():
        final_name = key.rsplit(".", 1)[-1]

        if final_name in THOP_BUFFER_NAMES:
            continue

        if torch.is_tensor(value):
            cleaned_state[key] = value.detach().cpu().clone()
        else:
            cleaned_state[key] = copy.deepcopy(value)

    return cleaned_state


def load_checkpoint_and_model(dataset_class_names: Sequence[str]):
    checkpoint_path = (
        FINAL_MODEL_PATH
        if FINAL_MODEL_PATH.exists()
        else BEST_MODEL_PATH
    )

    # Load on CPU first. This avoids unnecessary GPU allocation and makes
    # contaminated inference tensors safe to clone before model loading.
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    checkpoint_class_names = checkpoint.get("class_names")
    class_names = (
        list(dataset_class_names)
        if checkpoint_class_names is None
        else list(checkpoint_class_names)
    )

    if list(dataset_class_names) != class_names:
        raise ValueError(
            "Checkpoint classes do not match test-folder classes.\n"
            f"Checkpoint: {class_names}\n"
            f"Test folders: {list(dataset_class_names)}"
        )

    configuration = checkpoint.get("configuration", {})
    grfm_dim = int(configuration.get("grfm_dim", GRFM_DIM))
    grfm_dropout = float(
        configuration.get("grfm_dropout", GRFM_DROPOUT)
    )

    model = MaxViTTinyGRFM(
        num_classes=len(class_names),
        grfm_dim=grfm_dim,
        dropout=grfm_dropout,
    )

    remove_thop_buffers_from_model(model)

    raw_state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )
    clean_state_dict = clean_checkpoint_state_dict(raw_state_dict)

    incompatible = model.load_state_dict(
        clean_state_dict,
        strict=False,
    )

    meaningful_missing = [
        key
        for key in incompatible.missing_keys
        if key.rsplit(".", 1)[-1] not in THOP_BUFFER_NAMES
    ]
    meaningful_unexpected = [
        key
        for key in incompatible.unexpected_keys
        if key.rsplit(".", 1)[-1] not in THOP_BUFFER_NAMES
    ]

    if meaningful_missing or meaningful_unexpected:
        raise RuntimeError(
            "Checkpoint and model architecture do not match.\n"
            f"Missing keys: {meaningful_missing}\n"
            f"Unexpected keys: {meaningful_unexpected}"
        )

    model = model.to(DEVICE)
    model.eval()

    removed_thop_keys = len(raw_state_dict) - len(clean_state_dict)

    print(f"Loaded checkpoint : {checkpoint_path}")
    print(f"Checkpoint epoch  : {checkpoint.get('epoch', 'unknown')}")
    print(f"Classes           : {class_names}")
    print(f"Device            : {DEVICE}")

    if removed_thop_keys > 0:
        print(
            "Checkpoint cleanup : removed "
            f"{removed_thop_keys} THOP profiling entries"
        )

    return model, class_names, checkpoint_path


# ============================================================
# 6. PREDICTIONS AND FEATURE EXTRACTION
# ============================================================

@torch.inference_mode()
def extract_predictions_and_features(
    model: nn.Module,
    loader: DataLoader,
    dataset: datasets.ImageFolder,
):
    logits_all: List[torch.Tensor] = []
    backbone_features_all: List[torch.Tensor] = []
    refined_features_all: List[torch.Tensor] = []
    gate_all: List[torch.Tensor] = []
    labels_all: List[np.ndarray] = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)

        with amp_context():
            outputs = model(images, return_features=True)

        logits_all.append(outputs["logits"].float().cpu())
        backbone_features_all.append(outputs["backbone_features"].float().cpu())
        refined_features_all.append(outputs["grfm_features"].float().cpu())
        gate_all.append(outputs["gate"].float().cpu())
        labels_all.append(labels.numpy())

    logits = torch.cat(logits_all, dim=0)
    probabilities = torch.softmax(logits, dim=1).numpy()
    predictions = np.argmax(probabilities, axis=1)

    backbone_features = torch.cat(backbone_features_all, dim=0).numpy()
    refined_features = torch.cat(refined_features_all, dim=0).numpy()
    gates = torch.cat(gate_all, dim=0).numpy()
    labels = np.concatenate(labels_all).astype(np.int64)

    if len(dataset.samples) != len(labels):
        raise RuntimeError("Dataset sample paths and extracted labels do not align.")

    image_paths = [sample[0] for sample in dataset.samples]

    return {
        "logits": logits.numpy(),
        "probabilities": probabilities,
        "predictions": predictions,
        "labels": labels,
        "backbone_features": backbone_features,
        "refined_features": refined_features,
        "gates": gates,
        "image_paths": image_paths,
    }


# ============================================================
# 7. TARGET LAYER AND ACTIVATION HOOK
# ============================================================

def list_convolutional_layers(model: nn.Module) -> List[str]:
    return [
        name
        for name, module in model.named_modules()
        if isinstance(module, nn.Conv2d)
    ]


def resolve_target_layer(
    model: nn.Module,
    requested_name: Optional[str] = None,
) -> Tuple[str, nn.Module]:
    named_modules = dict(model.named_modules())

    if requested_name is not None:
        if requested_name not in named_modules:
            available = list_convolutional_layers(model)
            raise ValueError(
                f"TARGET_LAYER_NAME='{requested_name}' was not found.\n"
                "Available convolutional layers include:\n- "
                + "\n- ".join(available[-30:])
            )

        target_layer = named_modules[requested_name]
        if not isinstance(target_layer, nn.Conv2d):
            raise TypeError(
                f"Selected target layer is not Conv2d: {requested_name}"
            )
        return requested_name, target_layer

    convolutional_layers = [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Conv2d)
    ]

    if not convolutional_layers:
        raise RuntimeError("No convolutional layer was found for Grad-CAM++.")

    return convolutional_layers[-1]


class SpatialActivationCapture:
    """Capture a layer output and retain its gradient for Grad-CAM++."""

    def __init__(self, layer: nn.Module):
        self.activations: Optional[torch.Tensor] = None
        self.forward_handle = layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inputs, output):
        if isinstance(output, (tuple, list)):
            output = output[0]

        self.activations = output

        # During Grad-CAM++ the tensor participates in autograd. Retaining its
        # gradient avoids fragile module-level backward hooks. During ordinary
        # no-gradient feature extraction this branch is skipped.
        if isinstance(output, torch.Tensor) and output.requires_grad:
            output.retain_grad()

    @property
    def gradients(self) -> Optional[torch.Tensor]:
        if self.activations is None:
            return None
        return self.activations.grad

    def clear(self):
        self.activations = None

    def close(self):
        self.forward_handle.remove()


# ============================================================
# 8. GRAD-CAM++, SALIENCY, AND REFINEMENT MAPS
# ============================================================

def ensure_spatial_tensor(tensor: torch.Tensor, name: str) -> torch.Tensor:
    if tensor is None:
        raise RuntimeError(f"{name} was not captured.")

    if tensor.ndim != 4:
        raise RuntimeError(
            f"{name} must be a four-dimensional NCHW tensor, got {tensor.shape}."
        )

    return tensor


def compute_gradcam_plus_plus(
    model: nn.Module,
    image_batch: torch.Tensor,
    target_class: int,
    capture: SpatialActivationCapture,
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute Grad-CAM++ and return the normalized map and probabilities."""
    model.zero_grad(set_to_none=True)
    capture.clear()

    # Gradients are more stable in float32, so AMP is intentionally disabled.
    outputs = model(image_batch, return_features=True)
    logits = outputs["logits"]
    probabilities = torch.softmax(logits.detach(), dim=1)[0].cpu().numpy()

    score = logits[0, target_class]
    score.backward(retain_graph=False)

    activations = ensure_spatial_tensor(capture.activations, "Activations")
    gradients = ensure_spatial_tensor(capture.gradients, "Gradients")

    activations = activations.detach().float()
    gradients = gradients.detach().float()

    gradient_squared = gradients.pow(2)
    gradient_cubed = gradient_squared * gradients

    spatial_activation_sum = activations.sum(dim=(2, 3), keepdim=True)

    denominator = (
        2.0 * gradient_squared
        + spatial_activation_sum * gradient_cubed
    )
    denominator = torch.where(
        denominator.abs() > 1e-8,
        denominator,
        torch.ones_like(denominator),
    )

    alpha = gradient_squared / denominator
    alpha = torch.where(gradients != 0.0, alpha, torch.zeros_like(alpha))

    positive_gradients = F.relu(gradients)
    weights = (alpha * positive_gradients).sum(dim=(2, 3), keepdim=True)

    cam = F.relu((weights * activations).sum(dim=1, keepdim=True))
    cam = F.interpolate(
        cam,
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )

    cam_array = cam[0, 0].detach().cpu().numpy()
    return normalize_map(cam_array), probabilities


def compute_input_saliency(
    model: nn.Module,
    image_batch: torch.Tensor,
    target_class: int,
) -> np.ndarray:
    """Compute absolute input-gradient saliency for the selected class."""
    input_tensor = image_batch.detach().clone().requires_grad_(True)
    model.zero_grad(set_to_none=True)

    logits = model(input_tensor)
    score = logits[0, target_class]
    score.backward()

    if input_tensor.grad is None:
        raise RuntimeError("Input gradient was not produced for saliency.")

    saliency = input_tensor.grad.detach().abs().amax(dim=1)[0]
    saliency = saliency.cpu().numpy()
    return normalize_map(saliency)


@torch.no_grad()
def compute_feature_refinement_maps(
    model: nn.Module,
    image_batch: torch.Tensor,
    capture: SpatialActivationCapture,
) -> Dict[str, np.ndarray]:
    """
    Build a GRFM-gate-guided spatial refinement visualization.

    Because GRFM consumes pooled vectors, this is an interpretation proxy:
    the last spatial backbone activation is channel-weighted by the learned
    GRFM gate for the same image.
    """
    capture.clear()
    outputs = model(image_batch, return_features=True)

    spatial_features = ensure_spatial_tensor(
        capture.activations,
        "Spatial backbone activations",
    ).detach().float()

    gate = outputs["gate"].detach().float()

    feature_tensor = spatial_features[0]
    channels, feature_height, feature_width = feature_tensor.shape

    base_map = feature_tensor.abs().mean(dim=0)

    gate_vector = gate[0]
    if gate_vector.numel() != channels:
        gate_vector = F.interpolate(
            gate_vector.view(1, 1, -1),
            size=channels,
            mode="linear",
            align_corners=False,
        ).view(-1)

    # Normalize weights to preserve a comparable scale.
    gate_vector = gate_vector.clamp(min=0.0)
    gate_weights = gate_vector / (gate_vector.mean() + 1e-8)

    refined_map = (
        feature_tensor.abs()
        * gate_weights.view(channels, 1, 1)
    ).mean(dim=0)

    # Absolute modulation effect highlights where gating changed emphasis.
    refinement_change = (refined_map - base_map).abs()

    base_map = F.interpolate(
        base_map.view(1, 1, feature_height, feature_width),
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )[0, 0]

    refined_map = F.interpolate(
        refined_map.view(1, 1, feature_height, feature_width),
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )[0, 0]

    refinement_change = F.interpolate(
        refinement_change.view(1, 1, feature_height, feature_width),
        size=image_batch.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )[0, 0]

    return {
        "backbone_activation": normalize_map(base_map.cpu().numpy()),
        "grfm_refined": normalize_map(refined_map.cpu().numpy()),
        "refinement_change": normalize_map(refinement_change.cpu().numpy()),
        "gate_mean": float(gate_vector.mean().cpu().item()),
        "gate_std": float(gate_vector.std(unbiased=False).cpu().item()),
    }


# ============================================================
# 9. SAMPLE SELECTION
# ============================================================

def select_three_samples_per_class(
    labels: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
    class_names: Sequence[str],
    samples_per_class: int = SAMPLES_PER_CLASS,
) -> Dict[int, List[int]]:
    confidence = probabilities.max(axis=1)
    selected: Dict[int, List[int]] = {}

    for class_index, class_name in enumerate(class_names):
        class_indices = np.where(labels == class_index)[0]

        if len(class_indices) == 0:
            print(f"Warning: class '{class_name}' has no test images.")
            selected[class_index] = []
            continue

        chosen: List[int] = []

        if PREFER_CORRECT_EXPLANATION_SAMPLES:
            correct_indices = class_indices[
                predictions[class_indices] == class_index
            ]
            correct_indices = correct_indices[
                np.argsort(confidence[correct_indices])[::-1]
            ]
            chosen.extend(correct_indices[:samples_per_class].tolist())

        if len(chosen) < samples_per_class:
            remaining = np.asarray(
                [index for index in class_indices if index not in chosen],
                dtype=np.int64,
            )
            remaining = remaining[np.argsort(confidence[remaining])[::-1]]
            needed = samples_per_class - len(chosen)
            chosen.extend(remaining[:needed].tolist())

        selected[class_index] = chosen

    return selected


# ============================================================
# 10. EXPLANATION PANELS: THREE IMAGES PER CLASS
# ============================================================

def generate_explanation_panels(
    model: nn.Module,
    dataset: datasets.ImageFolder,
    class_names: Sequence[str],
    extracted: Dict[str, np.ndarray],
    capture: SpatialActivationCapture,
) -> pd.DataFrame:
    labels = extracted["labels"]
    predictions = extracted["predictions"]
    probabilities = extracted["probabilities"]
    image_paths = extracted["image_paths"]

    selected_by_class = select_three_samples_per_class(
        labels=labels,
        predictions=predictions,
        probabilities=probabilities,
        class_names=class_names,
    )

    panel_dir = VISUALIZATION_DIR / "per_class_explanation_panels"
    individual_dir = VISUALIZATION_DIR / "individual_explanations"
    panel_dir.mkdir(parents=True, exist_ok=True)
    individual_dir.mkdir(parents=True, exist_ok=True)

    records = []

    column_titles = [
        "Original",
        "Backbone activation",
        "GRFM-guided refinement",
        "Refinement change",
        "Grad-CAM++",
        "Saliency map",
    ]

    for class_index, class_name in enumerate(class_names):
        selected_indices = selected_by_class[class_index]

        if not selected_indices:
            continue

        number_of_rows = len(selected_indices)
        figure, axes = plt.subplots(
            number_of_rows,
            len(column_titles),
            figsize=(19, 4.3 * number_of_rows),
            squeeze=False,
        )

        for column_index, title in enumerate(column_titles):
            axes[0, column_index].set_title(title, fontsize=12, fontweight="bold")

        class_output_dir = individual_dir / sanitize_filename(class_name)
        class_output_dir.mkdir(parents=True, exist_ok=True)

        for row_index, dataset_index in enumerate(selected_indices):
            image_tensor, true_label = dataset[dataset_index]
            image_batch = image_tensor.unsqueeze(0).to(DEVICE)
            rgb_image = tensor_to_rgb(image_tensor)

            predicted_label = int(predictions[dataset_index])
            predicted_confidence = float(probabilities[dataset_index, predicted_label])
            true_probability = float(probabilities[dataset_index, true_label])

            if EXPLANATION_TARGET.lower() == "predicted":
                target_class = predicted_label
            elif EXPLANATION_TARGET.lower() == "true":
                target_class = int(true_label)
            else:
                raise ValueError(
                    "EXPLANATION_TARGET must be either 'true' or 'predicted'."
                )

            refinement = compute_feature_refinement_maps(
                model=model,
                image_batch=image_batch,
                capture=capture,
            )

            gradcam_map, _ = compute_gradcam_plus_plus(
                model=model,
                image_batch=image_batch,
                target_class=target_class,
                capture=capture,
            )

            saliency_map = compute_input_saliency(
                model=model,
                image_batch=image_batch,
                target_class=target_class,
            )

            backbone_overlay = overlay_heatmap(
                rgb_image,
                refinement["backbone_activation"],
            )
            refined_overlay = overlay_heatmap(
                rgb_image,
                refinement["grfm_refined"],
            )
            change_overlay = overlay_heatmap(
                rgb_image,
                refinement["refinement_change"],
                cmap_name="magma",
            )
            gradcam_overlay = overlay_heatmap(rgb_image, gradcam_map)
            saliency_overlay = overlay_heatmap(
                rgb_image,
                saliency_map,
                cmap_name="inferno",
            )

            displayed_images = [
                rgb_image,
                backbone_overlay,
                refined_overlay,
                change_overlay,
                gradcam_overlay,
                saliency_overlay,
            ]

            for column_index, shown_image in enumerate(displayed_images):
                axes[row_index, column_index].imshow(shown_image)
                axes[row_index, column_index].axis("off")

            status = "Correct" if predicted_label == true_label else "Misclassified"
            axes[row_index, 0].set_ylabel(
                f"True: {class_names[true_label]}\n"
                f"Pred: {class_names[predicted_label]}\n"
                f"Conf: {predicted_confidence:.3f}\n{status}",
                fontsize=10,
            )

            base_name = f"sample_{row_index + 1:02d}_index_{dataset_index:05d}"
            save_rgb_image(rgb_image, class_output_dir / f"{base_name}_original.png")
            save_rgb_image(
                backbone_overlay,
                class_output_dir / f"{base_name}_backbone_activation.png",
            )
            save_rgb_image(
                refined_overlay,
                class_output_dir / f"{base_name}_grfm_refined.png",
            )
            save_rgb_image(
                change_overlay,
                class_output_dir / f"{base_name}_refinement_change.png",
            )
            save_rgb_image(
                gradcam_overlay,
                class_output_dir / f"{base_name}_gradcam_plus_plus.png",
            )
            save_rgb_image(
                saliency_overlay,
                class_output_dir / f"{base_name}_saliency.png",
            )

            records.append({
                "Dataset Index": int(dataset_index),
                "Image Path": image_paths[dataset_index],
                "True Class": class_names[true_label],
                "Predicted Class": class_names[predicted_label],
                "Correct": int(predicted_label == true_label),
                "Predicted Confidence": predicted_confidence,
                "True-Class Probability": true_probability,
                "Explanation Target": class_names[target_class],
                "GRFM Gate Mean": refinement["gate_mean"],
                "GRFM Gate Std": refinement["gate_std"],
            })

        figure.suptitle(
            f"{MODEL_DISPLAY_NAME}: visual explanations for class '{class_name}'",
            fontsize=16,
            fontweight="bold",
            y=1.01,
        )
        save_and_display_figure(
            figure,
            panel_dir / f"{sanitize_filename(class_name)}_three_images.png",
            dpi=300,
            display_inline=DISPLAY_LARGE_EXPLANATION_PANELS,
        )

    records_dataframe = pd.DataFrame(records)
    records_dataframe.to_csv(
        VISUALIZATION_DIR / "selected_explanation_samples.csv",
        index=False,
    )

    return records_dataframe


# ============================================================
# 11. t-SNE BEFORE AND AFTER GRFM
# ============================================================

def stratified_tsne_indices(
    labels: np.ndarray,
    max_samples_per_class: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    selected_indices = []

    for class_index in np.unique(labels):
        class_indices = np.where(labels == class_index)[0]
        if len(class_indices) > max_samples_per_class:
            class_indices = rng.choice(
                class_indices,
                size=max_samples_per_class,
                replace=False,
            )
        selected_indices.extend(class_indices.tolist())

    selected_indices = np.asarray(selected_indices, dtype=np.int64)
    rng.shuffle(selected_indices)
    return selected_indices


def prepare_features_for_tsne(features: np.ndarray) -> np.ndarray:
    features = np.asarray(features, dtype=np.float64)
    features = StandardScaler().fit_transform(features)

    maximum_components = min(
        TSNE_PCA_COMPONENTS,
        features.shape[0] - 1,
        features.shape[1],
    )

    if maximum_components >= 2 and features.shape[1] > maximum_components:
        features = PCA(
            n_components=maximum_components,
            random_state=SEED,
        ).fit_transform(features)

    return features


def run_tsne(features: np.ndarray) -> np.ndarray:
    number_of_samples = len(features)

    if number_of_samples < 3:
        raise ValueError("At least three samples are required for t-SNE.")

    perplexity = min(TSNE_PERPLEXITY, max(2, number_of_samples // 4))
    perplexity = min(perplexity, number_of_samples - 1)

    # sklearn changed n_iter to max_iter in newer versions. This fallback
    # keeps the script compatible with both APIs.
    common_arguments = {
        "n_components": 2,
        "perplexity": perplexity,
        "learning_rate": "auto",
        "init": "pca",
        "random_state": SEED,
        "method": "barnes_hut",
        "angle": 0.5,
    }

    try:
        tsne = TSNE(
            **common_arguments,
            max_iter=TSNE_ITERATIONS,
        )
    except TypeError:
        tsne = TSNE(
            **common_arguments,
            n_iter=TSNE_ITERATIONS,
        )

    return tsne.fit_transform(features)


def plot_tsne_embedding(
    embedding: np.ndarray,
    labels: np.ndarray,
    class_names: Sequence[str],
    title: str,
    filename: str,
) -> None:
    figure = plt.figure(figsize=(9, 7))

    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        if not np.any(mask):
            continue

        plt.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            s=28,
            alpha=0.78,
            label=f"{class_name} (n={int(mask.sum())})",
        )

    plt.xlabel("t-SNE component 1")
    plt.ylabel("t-SNE component 2")
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.25)
    plt.legend(loc="best", fontsize=9)
    save_and_display_figure(
        figure,
        VISUALIZATION_DIR / filename,
        dpi=300,
        display_inline=True,
    )


def generate_tsne_plots(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    labels = extracted["labels"]
    selected_indices = stratified_tsne_indices(
        labels=labels,
        max_samples_per_class=TSNE_MAX_SAMPLES_PER_CLASS,
        seed=SEED,
    )

    selected_labels = labels[selected_indices]
    backbone_features = extracted["backbone_features"][selected_indices]
    refined_features = extracted["refined_features"][selected_indices]

    print(
        f"Running t-SNE on {len(selected_indices)} stratified test samples..."
    )

    backbone_prepared = prepare_features_for_tsne(backbone_features)
    refined_prepared = prepare_features_for_tsne(refined_features)

    backbone_embedding = run_tsne(backbone_prepared)
    refined_embedding = run_tsne(refined_prepared)

    plot_tsne_embedding(
        embedding=backbone_embedding,
        labels=selected_labels,
        class_names=class_names,
        title=f"{MODEL_DISPLAY_NAME}: backbone feature t-SNE",
        filename="tsne_backbone_features.png",
    )

    plot_tsne_embedding(
        embedding=refined_embedding,
        labels=selected_labels,
        class_names=class_names,
        title=f"{MODEL_DISPLAY_NAME}: GRFM-refined feature t-SNE",
        filename="tsne_grfm_refined_features.png",
    )

    comparison_figure, axes = plt.subplots(1, 2, figsize=(18, 7))

    for class_index, class_name in enumerate(class_names):
        mask = selected_labels == class_index
        if not np.any(mask):
            continue

        axes[0].scatter(
            backbone_embedding[mask, 0],
            backbone_embedding[mask, 1],
            s=25,
            alpha=0.75,
            label=class_name,
        )
        axes[1].scatter(
            refined_embedding[mask, 0],
            refined_embedding[mask, 1],
            s=25,
            alpha=0.75,
            label=class_name,
        )

    axes[0].set_title("Before GRFM: backbone vectors")
    axes[1].set_title("After GRFM: refined vectors")

    for axis in axes:
        axis.set_xlabel("t-SNE component 1")
        axis.set_ylabel("t-SNE component 2")
        axis.grid(True, linestyle="--", alpha=0.25)
        axis.legend(loc="best", fontsize=8)

    comparison_figure.suptitle(
        f"{MODEL_DISPLAY_NAME}: feature-space refinement comparison",
        fontsize=16,
        fontweight="bold",
    )
    save_and_display_figure(
        comparison_figure,
        VISUALIZATION_DIR / "tsne_before_after_grfm_comparison.png",
        dpi=300,
        display_inline=True,
    )

    tsne_dataframe = pd.DataFrame({
        "Dataset Index": selected_indices,
        "True Label": selected_labels,
        "True Class": [class_names[index] for index in selected_labels],
        "Backbone t-SNE 1": backbone_embedding[:, 0],
        "Backbone t-SNE 2": backbone_embedding[:, 1],
        "GRFM t-SNE 1": refined_embedding[:, 0],
        "GRFM t-SNE 2": refined_embedding[:, 1],
    })

    tsne_dataframe.to_csv(
        VISUALIZATION_DIR / "tsne_coordinates.csv",
        index=False,
    )

    return tsne_dataframe



# ============================================================
# 12. CHECKPOINT EVALUATION PLOTS
# ============================================================

def generate_confusion_matrix_plot(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    labels = extracted["labels"]
    predictions = extracted["predictions"]

    matrix = confusion_matrix(
        labels,
        predictions,
        labels=np.arange(len(class_names)),
    )

    figure, axis = plt.subplots(figsize=(8.5, 7.2))
    image = axis.imshow(matrix, interpolation="nearest")
    figure.colorbar(image, ax=axis)

    positions = np.arange(len(class_names))
    axis.set_xticks(positions)
    axis.set_yticks(positions)
    axis.set_xticklabels(class_names, rotation=35, ha="right")
    axis.set_yticklabels(class_names)

    threshold = matrix.max() / 2.0 if matrix.size else 0.0

    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = int(matrix[row, column])
            axis.text(
                column,
                row,
                str(value),
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
            )

    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    axis.set_title(f"{MODEL_DISPLAY_NAME}: confusion matrix")

    save_and_display_figure(
        figure,
        VISUALIZATION_DIR / "confusion_matrix.png",
        dpi=300,
        display_inline=True,
    )

    dataframe = pd.DataFrame(
        matrix,
        index=class_names,
        columns=class_names,
    )
    dataframe.to_csv(
        VISUALIZATION_DIR / "confusion_matrix.csv"
    )

    return dataframe


def generate_pr_auc_plot(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    labels = extracted["labels"]
    probabilities = extracted["probabilities"]
    number_of_classes = probabilities.shape[1]

    targets = label_binarize(
        labels,
        classes=np.arange(number_of_classes),
    )

    if number_of_classes == 2 and targets.shape[1] == 1:
        targets = np.column_stack([
            1 - targets[:, 0],
            targets[:, 0],
        ])

    figure, axis = plt.subplots(figsize=(9, 7))
    rows = []

    for class_index, class_name in enumerate(class_names):
        class_targets = targets[:, class_index]

        if len(np.unique(class_targets)) < 2:
            continue

        precision_values, recall_values, _ = precision_recall_curve(
            class_targets,
            probabilities[:, class_index],
        )

        average_precision = average_precision_score(
            class_targets,
            probabilities[:, class_index],
        )

        prevalence = float(class_targets.mean())

        axis.plot(
            recall_values,
            precision_values,
            linewidth=2,
            label=f"{class_name}: AP={average_precision:.4f}",
        )

        rows.append({
            "Class": class_name,
            "PR-AUC / Average Precision": float(average_precision),
            "Prevalence": prevalence,
        })

    micro_precision, micro_recall, _ = precision_recall_curve(
        targets.ravel(),
        probabilities.ravel(),
    )

    micro_ap = average_precision_score(
        targets,
        probabilities,
        average="micro",
    )

    macro_ap = average_precision_score(
        targets,
        probabilities,
        average="macro",
    )

    axis.plot(
        micro_recall,
        micro_precision,
        linestyle="--",
        linewidth=3,
        label=f"Micro-average: AP={micro_ap:.4f}",
    )

    axis.set_xlabel("Recall")
    axis.set_ylabel("Precision")
    axis.set_xlim(0.0, 1.01)
    axis.set_ylim(0.0, 1.05)
    axis.set_title(f"{MODEL_DISPLAY_NAME}: precision-recall curves")
    axis.grid(True, linestyle="--", alpha=0.4)
    axis.legend(loc="lower left", fontsize=9)

    save_and_display_figure(
        figure,
        VISUALIZATION_DIR / "pr_auc_curve.png",
        dpi=300,
        display_inline=True,
    )

    rows.extend([
        {
            "Class": "Macro average",
            "PR-AUC / Average Precision": float(macro_ap),
            "Prevalence": np.nan,
        },
        {
            "Class": "Micro average",
            "PR-AUC / Average Precision": float(micro_ap),
            "Prevalence": np.nan,
        },
    ])

    dataframe = pd.DataFrame(rows)
    dataframe.to_csv(
        VISUALIZATION_DIR / "pr_auc_values.csv",
        index=False,
    )

    return dataframe


def generate_confidence_histogram(
    extracted: Dict[str, np.ndarray],
) -> pd.DataFrame:
    labels = extracted["labels"]
    predictions = extracted["predictions"]
    probabilities = extracted["probabilities"]

    confidence = probabilities.max(axis=1)
    correct_mask = predictions == labels

    bins = np.linspace(
        0.0,
        1.0,
        CONFIDENCE_BINS + 1,
    )

    figure, axis = plt.subplots(figsize=(9, 6))

    if np.any(correct_mask):
        axis.hist(
            confidence[correct_mask],
            bins=bins,
            alpha=0.70,
            label=f"Correct (n={int(correct_mask.sum())})",
        )

    if np.any(~correct_mask):
        axis.hist(
            confidence[~correct_mask],
            bins=bins,
            alpha=0.75,
            label=f"Incorrect (n={int((~correct_mask).sum())})",
        )

    axis.axvline(
        float(confidence.mean()),
        linestyle="--",
        linewidth=2,
        label=f"Mean confidence={confidence.mean():.4f}",
    )

    axis.set_xlabel("Maximum predicted probability")
    axis.set_ylabel("Number of test images")
    axis.set_title(f"{MODEL_DISPLAY_NAME}: confidence histogram")
    axis.legend()

    save_and_display_figure(
        figure,
        VISUALIZATION_DIR / "confidence_histogram.png",
        dpi=300,
        display_inline=True,
    )

    dataframe = pd.DataFrame({
        "True Label": labels,
        "Predicted Label": predictions,
        "Correct": correct_mask.astype(int),
        "Confidence": confidence,
        "True-Class Probability": probabilities[
            np.arange(len(labels)),
            labels,
        ],
    })

    dataframe.to_csv(
        VISUALIZATION_DIR / "confidence_values.csv",
        index=False,
    )

    return dataframe


# ============================================================
# 13. MISCLASSIFIED IMAGE ANALYSIS
# ============================================================

def create_misclassification_dataframe(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    labels = extracted["labels"]
    predictions = extracted["predictions"]
    probabilities = extracted["probabilities"]
    image_paths = extracted["image_paths"]

    misclassified_indices = np.where(predictions != labels)[0]
    rows = []

    for index in misclassified_indices:
        true_label = int(labels[index])
        predicted_label = int(predictions[index])
        sorted_probabilities = np.sort(probabilities[index])[::-1]
        margin = (
            float(sorted_probabilities[0] - sorted_probabilities[1])
            if probabilities.shape[1] > 1
            else float(sorted_probabilities[0])
        )

        row = {
            "Dataset Index": int(index),
            "Image Path": image_paths[index],
            "True Label": true_label,
            "True Class": class_names[true_label],
            "Predicted Label": predicted_label,
            "Predicted Class": class_names[predicted_label],
            "Predicted Confidence": float(probabilities[index, predicted_label]),
            "True-Class Probability": float(probabilities[index, true_label]),
            "Confidence Margin": margin,
        }

        for class_index, class_name in enumerate(class_names):
            row[f"Probability {class_name}"] = float(
                probabilities[index, class_index]
            )

        rows.append(row)

    dataframe = pd.DataFrame(rows)

    if not dataframe.empty:
        dataframe = dataframe.sort_values(
            "Predicted Confidence",
            ascending=False,
        ).reset_index(drop=True)

    dataframe.to_csv(
        VISUALIZATION_DIR / "misclassified_images.csv",
        index=False,
    )

    return dataframe


def plot_misclassified_grid(
    dataset: datasets.ImageFolder,
    misclassified_dataframe: pd.DataFrame,
    class_names: Sequence[str],
    filename: str,
    title: str,
    max_images: int,
) -> None:
    if misclassified_dataframe.empty:
        # A zero-error class is a valid result, not a pipeline failure.
        # Save an explicit placeholder so every requested output exists.
        figure, axis = plt.subplots(figsize=(8, 4.5))
        axis.axis("off")
        axis.text(
            0.5,
            0.56,
            "No misclassified images",
            ha="center",
            va="center",
            fontsize=18,
            fontweight="bold",
            transform=axis.transAxes,
        )
        axis.text(
            0.5,
            0.40,
            title,
            ha="center",
            va="center",
            fontsize=11,
            wrap=True,
            transform=axis.transAxes,
        )
        figure.tight_layout()
        save_and_display_figure(
            figure,
            VISUALIZATION_DIR / filename,
            dpi=300,
            display_inline=True,
        )
        print(f"Zero misclassifications: {title}")
        return

    shown_dataframe = misclassified_dataframe.head(max_images)
    number_of_images = len(shown_dataframe)
    columns = min(4, number_of_images)
    rows = int(math.ceil(number_of_images / columns))

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(4.2 * columns, 4.2 * rows),
        squeeze=False,
    )
    axes_flat = axes.ravel()

    for axis in axes_flat:
        axis.axis("off")

    for plot_index, (_, row) in enumerate(shown_dataframe.iterrows()):
        dataset_index = int(row["Dataset Index"])
        image_tensor, _ = dataset[dataset_index]
        rgb_image = tensor_to_rgb(image_tensor)

        axes_flat[plot_index].imshow(rgb_image)
        axes_flat[plot_index].set_title(
            f"True: {row['True Class']}\n"
            f"Pred: {row['Predicted Class']}\n"
            f"Confidence: {row['Predicted Confidence']:.3f}",
            fontsize=10,
        )
        axes_flat[plot_index].axis("off")

    figure.suptitle(title, fontsize=16, fontweight="bold")
    save_and_display_figure(
        figure,
        VISUALIZATION_DIR / filename,
        dpi=300,
        display_inline=True,
    )


def generate_misclassification_outputs(
    dataset: datasets.ImageFolder,
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    dataframe = create_misclassification_dataframe(
        extracted=extracted,
        class_names=class_names,
    )

    total_images = len(extracted["labels"])
    total_errors = len(dataframe)
    accuracy = accuracy_score(
        extracted["labels"],
        extracted["predictions"],
    )

    print(f"Test accuracy       : {accuracy:.6f}")
    print(f"Misclassified images: {total_errors}/{total_images}")

    plot_misclassified_grid(
        dataset=dataset,
        misclassified_dataframe=dataframe,
        class_names=class_names,
        filename="misclassified_highest_confidence_grid.png",
        title=(
            f"{MODEL_DISPLAY_NAME}: highest-confidence misclassified images"
        ),
        max_images=MAX_MISCLASSIFIED_GRID_IMAGES,
    )

    for class_name in class_names:
        class_errors = dataframe[
            dataframe["True Class"] == class_name
        ] if not dataframe.empty else dataframe

        plot_misclassified_grid(
            dataset=dataset,
            misclassified_dataframe=class_errors,
            class_names=class_names,
            filename=(
                "misclassified_true_class_"
                f"{sanitize_filename(class_name)}.png"
            ),
            title=(
                f"{MODEL_DISPLAY_NAME}: errors with true class '{class_name}'"
            ),
            max_images=MAX_MISCLASSIFIED_PER_TRUE_CLASS,
        )

    return dataframe


# ============================================================
# 14. OPTIONAL: FEATURE-GATE SUMMARY
# ============================================================

def plot_gate_summary(
    extracted: Dict[str, np.ndarray],
    class_names: Sequence[str],
) -> pd.DataFrame:
    gates = extracted["gates"]
    labels = extracted["labels"]

    rows = []
    class_gate_means = []

    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        if not np.any(mask):
            continue

        mean_gate = gates[mask].mean(axis=0)
        class_gate_means.append((class_name, mean_gate))

        rows.append({
            "Class": class_name,
            "Images": int(mask.sum()),
            "Mean Gate Value": float(gates[mask].mean()),
            "Gate Standard Deviation": float(gates[mask].std()),
            "Minimum Gate Value": float(gates[mask].min()),
            "Maximum Gate Value": float(gates[mask].max()),
        })

    summary_dataframe = pd.DataFrame(rows)
    summary_dataframe.to_csv(
        VISUALIZATION_DIR / "grfm_gate_summary_by_class.csv",
        index=False,
    )

    if class_gate_means:
        matrix = np.stack([item[1] for item in class_gate_means], axis=0)
        names = [item[0] for item in class_gate_means]

        figure = plt.figure(figsize=(14, max(4, 0.8 * len(names))))
        plt.imshow(matrix, aspect="auto", interpolation="nearest", cmap="viridis")
        plt.colorbar(label="Mean GRFM gate value")
        plt.yticks(np.arange(len(names)), names)
        plt.xlabel("GRFM feature dimension")
        plt.ylabel("True class")
        plt.title(f"{MODEL_DISPLAY_NAME}: class-wise GRFM gate profiles")
        save_and_display_figure(
            figure,
            VISUALIZATION_DIR / "grfm_gate_profile_heatmap.png",
            dpi=300,
            display_inline=True,
        )

    return summary_dataframe


# ============================================================
# 15. MAIN
# ============================================================

def main() -> None:
    set_seed(SEED)
    validate_inputs()

    print("=" * 84)
    print(f"{MODEL_DISPLAY_NAME} visual explanation pipeline")
    print("=" * 84)
    print(f"Test directory    : {TEST_DIR}")
    print(f"Output directory  : {VISUALIZATION_DIR}")
    print(f"Images per class  : {SAMPLES_PER_CLASS}")
    print(f"Explanation target: {EXPLANATION_TARGET}")

    test_dataset, test_loader = create_test_dataset_and_loader()
    model, class_names, checkpoint_path = load_checkpoint_and_model(
        test_dataset.classes
    )

    target_layer_name, target_layer = resolve_target_layer(
        model,
        TARGET_LAYER_NAME,
    )
    print(f"Grad-CAM++ layer  : {target_layer_name}")

    # Save the selected layer name for reproducibility.
    with (VISUALIZATION_DIR / "selected_target_layer.txt").open(
        "w",
        encoding="utf-8",
    ) as file:
        file.write(target_layer_name + "\n")

    capture = SpatialActivationCapture(target_layer)

    try:
        print("\nExtracting predictions and feature vectors...")
        extracted = extract_predictions_and_features(
            model=model,
            loader=test_loader,
            dataset=test_dataset,
        )

        print("\nGenerating confusion matrix...")
        generate_confusion_matrix_plot(
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating PR-AUC curve...")
        generate_pr_auc_plot(
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating confidence histogram...")
        generate_confidence_histogram(
            extracted=extracted,
        )

        print("\nGenerating t-SNE plots...")
        generate_tsne_plots(
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating misclassification outputs...")
        generate_misclassification_outputs(
            dataset=test_dataset,
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating class-wise GRFM gate summaries...")
        plot_gate_summary(
            extracted=extracted,
            class_names=class_names,
        )

        print("\nGenerating three-image explanation panels per class...")
        generate_explanation_panels(
            model=model,
            dataset=test_dataset,
            class_names=class_names,
            extracted=extracted,
            capture=capture,
        )

        print(
            "\nDisplaying all saved "
            "visual-explanation outputs..."
        )

        display_generated_visual_outputs(
            class_names=class_names,
        )

    finally:
        capture.close()

    print("\nCompleted successfully")
    print("=" * 84)
    print(f"Checkpoint used: {checkpoint_path}")
    print(f"All outputs are in: {VISUALIZATION_DIR}")
    print("\nMain generated files:")
    print("- confusion_matrix.png")
    print("- pr_auc_curve.png")
    print("- confidence_histogram.png")
    print("- tsne_backbone_features.png")
    print("- tsne_grfm_refined_features.png")
    print("- tsne_before_after_grfm_comparison.png")
    print("- misclassified_images.csv")
    print("- misclassified_highest_confidence_grid.png")
    print("- grfm_gate_profile_heatmap.png")
    print("- per_class_explanation_panels/<class>_three_images.png")
    print("- individual_explanations/<class>/*.png")
    print(
        "\nAll requested CSV tables and PNG files "
        "were also displayed inline."
    )


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# ROBUSTNESS ANALYSIS FOR MAXVIT-TINY + GRFM
# ============================================================

import contextlib
import io
import json
import os
import random
import subprocess
import sys
import warnings
from pathlib import Path

try:
    import timm
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "timm",
    ])
    import timm

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image, ImageEnhance, ImageFilter

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

TEST_DIR = Path(
    "/kaggle/working/PVMD_Split/test"
)

RESULTS_DIR = Path(
    "/kaggle/working/MaxViT_Tiny_GRFM_Results"
)

FINAL_MODEL_PATH = (
    RESULTS_DIR / "final_maxvit_tiny_grfm.pth"
)

BEST_MODEL_PATH = (
    RESULTS_DIR / "best_maxvit_tiny_grfm.pth"
)

ROBUSTNESS_DIR = (
    RESULTS_DIR / "Robustness_Analysis"
)

ROBUSTNESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_NAME = "maxvit_tiny_tf_224.in1k"
MODEL_DISPLAY_NAME = "MaxViT-Tiny-224 + GRFM"

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

GRFM_DIM = 512
GRFM_DROPOUT = 0.30
GRFM_REDUCTION = 4

SEVERITIES = [1, 2, 3, 4, 5]

# For binary classification, specify the positive class folder.
# Leave None to use class index 1.
POSITIVE_CLASS_NAME = None

CORRUPTIONS = [
    "gaussian_noise",
    "salt_pepper_noise",
    "speckle_noise",
    "gaussian_blur",
    "motion_blur",
    "brightness_dark",
    "brightness_bright",
    "contrast_low",
    "jpeg_compression",
    "occlusion",
]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ============================================================
# 2. GENERAL UTILITIES
# ============================================================

def set_seed(seed=42):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def amp_context():

    if USE_AMP:

        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return contextlib.nullcontext()


def clean_json_value(value):

    if isinstance(value, dict):

        return {
            str(key): clean_json_value(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            clean_json_value(item)
            for item in value
        ]

    if isinstance(value, Path):
        return str(value)

    if isinstance(
        value,
        (np.integer, np.floating),
    ):
        value = value.item()

    if (
        isinstance(value, float)
        and not np.isfinite(value)
    ):
        return None

    return value


def save_json(data, path):

    with path.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            clean_json_value(data),
            file,
            indent=2,
        )


# ============================================================
# 3. GRFM MODULE
# ============================================================

class GRFM(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim=512,
        reduction=4,
        dropout=0.30,
    ):

        super().__init__()

        hidden_dim = max(
            output_dim // reduction,
            32,
        )

        self.projection = nn.Sequential(
            nn.Linear(
                input_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

        self.feature_transform = nn.Sequential(
            nn.Linear(
                output_dim,
                output_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                output_dim,
                output_dim,
            ),
        )

        self.feature_gate = nn.Sequential(
            nn.Linear(
                output_dim,
                hidden_dim,
            ),
            nn.GELU(),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.Sigmoid(),
        )

        self.output_norm = nn.LayerNorm(
            output_dim
        )

        self.output_dropout = nn.Dropout(
            dropout
        )

    def forward(self, features):

        projected = self.projection(
            features
        )

        transformed = self.feature_transform(
            projected
        )

        gate = self.feature_gate(
            projected
        )

        refined = (
            projected
            + transformed * gate
        )

        refined = self.output_norm(
            refined
        )

        refined = self.output_dropout(
            refined
        )

        return refined


# ============================================================
# 4. MAXVIT-TINY + GRFM MODEL
# ============================================================

class MaxViTTinyGRFM(nn.Module):

    def __init__(
        self,
        num_classes,
        grfm_dim=512,
        dropout=0.30,
    ):

        super().__init__()

        self.backbone = timm.create_model(
            MODEL_NAME,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )

        self.grfm = GRFM(
            input_dim=self.backbone.num_features,
            output_dim=grfm_dim,
            reduction=GRFM_REDUCTION,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                grfm_dim,
                grfm_dim // 2,
            ),
            nn.LayerNorm(
                grfm_dim // 2
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                grfm_dim // 2,
                num_classes,
            ),
        )

    def forward(self, images):

        backbone_features = self.backbone(
            images
        )

        refined_features = self.grfm(
            backbone_features
        )

        logits = self.classifier(
            refined_features
        )

        return logits


# ============================================================
# 5. IMAGE CONVERSION FUNCTIONS
# ============================================================

def pil_to_float_array(image):

    return (
        np.asarray(
            image.convert("RGB"),
            dtype=np.float32,
        )
        / 255.0
    )


def float_array_to_pil(array):

    array = np.clip(
        array * 255.0,
        0,
        255,
    ).astype(np.uint8)

    return Image.fromarray(
        array,
        mode="RGB",
    )


# ============================================================
# 6. MOTION BLUR
# ============================================================

def create_motion_blur_kernel(size):
    """
    Create a normalized horizontal motion-blur kernel.

    The size is forced to a positive odd integer. This helper is retained
    for documentation and optional inspection, while the actual filtering
    is performed by a NumPy implementation that supports arbitrary sizes.
    """
    size = max(1, int(size))

    if size % 2 == 0:
        size += 1

    kernel = np.zeros(
        (size, size),
        dtype=np.float32,
    )

    kernel[size // 2, :] = (
        1.0 / float(size)
    )

    return kernel


def _moving_average_along_axis(
    array,
    window_size,
    axis,
):
    """
    Apply an edge-padded moving average along one image axis.

    This supports arbitrary odd kernel sizes and avoids Pillow's
    ImageFilter.Kernel limitation for kernels larger than 5 x 5.
    """
    window_size = max(
        1,
        int(window_size),
    )

    if window_size % 2 == 0:
        window_size += 1

    if window_size == 1:
        return array.copy()

    padding = window_size // 2

    pad_width = [
        (0, 0)
        for _ in range(array.ndim)
    ]

    pad_width[axis] = (
        padding,
        padding,
    )

    padded = np.pad(
        array,
        pad_width=pad_width,
        mode="edge",
    )

    # Add a leading zero so consecutive cumulative-sum differences
    # return exactly one value for each original image position.
    cumulative = np.cumsum(
        padded,
        axis=axis,
        dtype=np.float64,
    )

    zero_shape = list(
        cumulative.shape
    )
    zero_shape[axis] = 1

    cumulative = np.concatenate(
        [
            np.zeros(
                zero_shape,
                dtype=np.float64,
            ),
            cumulative,
        ],
        axis=axis,
    )

    upper_slice = [
        slice(None)
        for _ in range(
            cumulative.ndim
        )
    ]

    lower_slice = [
        slice(None)
        for _ in range(
            cumulative.ndim
        )
    ]

    upper_slice[axis] = slice(
        window_size,
        None,
    )

    lower_slice[axis] = slice(
        None,
        -window_size,
    )

    filtered = (
        cumulative[
            tuple(upper_slice)
        ]
        - cumulative[
            tuple(lower_slice)
        ]
    ) / float(window_size)

    return filtered.astype(
        np.float32
    )


def apply_motion_blur(
    image,
    kernel_size,
    direction="horizontal",
):
    """
    Apply motion blur with an arbitrary odd kernel size.

    Parameters
    ----------
    image:
        PIL RGB image.
    kernel_size:
        Requested blur length. Even values are changed to the next odd value.
    direction:
        "horizontal", "vertical", or "diagonal".

    Unlike PIL.ImageFilter.Kernel, this implementation works with the
    3, 5, 7, 9, and 11 pixel kernels used by the severity settings.
    """
    kernel_size = max(
        1,
        int(kernel_size),
    )

    if kernel_size % 2 == 0:
        kernel_size += 1

    image_array = pil_to_float_array(
        image
    )

    if direction == "horizontal":
        blurred = _moving_average_along_axis(
            image_array,
            window_size=kernel_size,
            axis=1,
        )

    elif direction == "vertical":
        blurred = _moving_average_along_axis(
            image_array,
            window_size=kernel_size,
            axis=0,
        )

    elif direction == "diagonal":
        horizontal = _moving_average_along_axis(
            image_array,
            window_size=kernel_size,
            axis=1,
        )

        vertical = _moving_average_along_axis(
            image_array,
            window_size=kernel_size,
            axis=0,
        )

        blurred = (
            horizontal + vertical
        ) / 2.0

    else:
        raise ValueError(
            "direction must be 'horizontal', "
            "'vertical', or 'diagonal'."
        )

    return float_array_to_pil(
        blurred
    )


# ============================================================
# 7. CORRUPTION FUNCTIONS
# ============================================================

def apply_corruption(
    image,
    corruption,
    severity,
    rng,
):

    if corruption == "clean":
        return image.copy()

    severity = int(
        np.clip(
            severity,
            1,
            5,
        )
    )

    image_array = pil_to_float_array(
        image
    )

    # --------------------------------------------------------
    # Gaussian noise
    # --------------------------------------------------------

    if corruption == "gaussian_noise":

        standard_deviation = [
            0.02,
            0.04,
            0.06,
            0.09,
            0.12,
        ][severity - 1]

        noise = rng.normal(
            loc=0.0,
            scale=standard_deviation,
            size=image_array.shape,
        )

        corrupted = (
            image_array + noise
        )

        return float_array_to_pil(
            corrupted
        )

    # --------------------------------------------------------
    # Salt-and-pepper noise
    # --------------------------------------------------------

    if corruption == "salt_pepper_noise":

        amount = [
            0.005,
            0.010,
            0.020,
            0.035,
            0.050,
        ][severity - 1]

        corrupted = image_array.copy()

        random_mask = rng.random(
            image_array.shape[:2]
        )

        salt_mask = (
            random_mask < amount / 2
        )

        pepper_mask = (
            (random_mask >= amount / 2)
            & (random_mask < amount)
        )

        corrupted[salt_mask] = 1.0
        corrupted[pepper_mask] = 0.0

        return float_array_to_pil(
            corrupted
        )

    # --------------------------------------------------------
    # Speckle noise
    # --------------------------------------------------------

    if corruption == "speckle_noise":

        standard_deviation = [
            0.03,
            0.06,
            0.09,
            0.13,
            0.18,
        ][severity - 1]

        noise = rng.normal(
            loc=0.0,
            scale=standard_deviation,
            size=image_array.shape,
        )

        corrupted = (
            image_array
            + image_array * noise
        )

        return float_array_to_pil(
            corrupted
        )

    # --------------------------------------------------------
    # Gaussian blur
    # --------------------------------------------------------

    if corruption == "gaussian_blur":

        radius = [
            0.6,
            1.0,
            1.5,
            2.2,
            3.0,
        ][severity - 1]

        return image.filter(
            ImageFilter.GaussianBlur(
                radius=radius
            )
        )

    # --------------------------------------------------------
    # Motion blur
    # --------------------------------------------------------

    if corruption == "motion_blur":

        kernel_size = [
            3,
            5,
            7,
            9,
            11,
        ][severity - 1]

        return apply_motion_blur(
            image,
            kernel_size,
        )

    # --------------------------------------------------------
    # Dark brightness
    # --------------------------------------------------------

    if corruption == "brightness_dark":

        factor = [
            0.90,
            0.80,
            0.70,
            0.60,
            0.50,
        ][severity - 1]

        return ImageEnhance.Brightness(
            image
        ).enhance(factor)

    # --------------------------------------------------------
    # High brightness
    # --------------------------------------------------------

    if corruption == "brightness_bright":

        factor = [
            1.10,
            1.20,
            1.35,
            1.50,
            1.70,
        ][severity - 1]

        return ImageEnhance.Brightness(
            image
        ).enhance(factor)

    # --------------------------------------------------------
    # Low contrast
    # --------------------------------------------------------

    if corruption == "contrast_low":

        factor = [
            0.90,
            0.75,
            0.60,
            0.45,
            0.30,
        ][severity - 1]

        return ImageEnhance.Contrast(
            image
        ).enhance(factor)

    # --------------------------------------------------------
    # JPEG compression
    # --------------------------------------------------------

    if corruption == "jpeg_compression":

        quality = [
            80,
            60,
            40,
            25,
            10,
        ][severity - 1]

        memory_buffer = io.BytesIO()

        image.save(
            memory_buffer,
            format="JPEG",
            quality=quality,
        )

        memory_buffer.seek(0)

        compressed_image = (
            Image.open(memory_buffer)
            .convert("RGB")
            .copy()
        )

        memory_buffer.close()

        return compressed_image

    # --------------------------------------------------------
    # Random occlusion
    # --------------------------------------------------------

    if corruption == "occlusion":

        fraction = [
            0.08,
            0.12,
            0.18,
            0.24,
            0.30,
        ][severity - 1]

        corrupted = image_array.copy()

        height, width = (
            corrupted.shape[:2]
        )

        box_height = max(
            1,
            int(height * fraction),
        )

        box_width = max(
            1,
            int(width * fraction),
        )

        y_start = int(
            rng.integers(
                0,
                max(
                    1,
                    height - box_height + 1,
                ),
            )
        )

        x_start = int(
            rng.integers(
                0,
                max(
                    1,
                    width - box_width + 1,
                ),
            )
        )

        mean_value = corrupted.mean(
            axis=(0, 1),
            keepdims=True,
        )

        corrupted[
            y_start:y_start + box_height,
            x_start:x_start + box_width,
        ] = mean_value

        return float_array_to_pil(
            corrupted
        )

    raise ValueError(
        f"Unknown corruption: {corruption}"
    )


# ============================================================
# 8. CORRUPTED IMAGE DATASET
# ============================================================

class CorruptedImageFolder(Dataset):

    def __init__(
        self,
        root,
        corruption="clean",
        severity=0,
        seed=42,
    ):

        self.base_dataset = (
            datasets.ImageFolder(
                str(root)
            )
        )

        self.samples = (
            self.base_dataset.samples
        )

        self.targets = (
            self.base_dataset.targets
        )

        self.classes = (
            self.base_dataset.classes
        )

        self.class_to_idx = (
            self.base_dataset.class_to_idx
        )

        self.corruption = corruption
        self.severity = severity
        self.seed = seed

        self.resize = transforms.Resize(
            (IMG_SIZE, IMG_SIZE)
        )

        self.tensor_transform = (
            transforms.Compose([
                transforms.ToTensor(),

                transforms.Normalize(
                    mean=IMAGENET_MEAN,
                    std=IMAGENET_STD,
                ),
            ])
        )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = (
            self.samples[index]
        )

        with Image.open(
            image_path
        ) as opened_image:

            image = opened_image.convert(
                "RGB"
            )

        image = self.resize(image)

        # A fixed seed produces the same corruption for
        # the same image and severity during every run.
        random_generator = (
            np.random.default_rng(
                self.seed
                + index * 1009
                + self.severity * 9176
            )
        )

        image = apply_corruption(
            image=image,
            corruption=self.corruption,
            severity=self.severity,
            rng=random_generator,
        )

        image_tensor = (
            self.tensor_transform(image)
        )

        return (
            image_tensor,
            label,
            image_path,
        )


def create_loader(
    corruption="clean",
    severity=0,
):

    dataset = CorruptedImageFolder(
        root=TEST_DIR,
        corruption=corruption,
        severity=severity,
        seed=SEED,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

    return dataset, loader


# ============================================================
# 9. LOAD TRAINED MODEL
# ============================================================

def load_trained_model():

    if FINAL_MODEL_PATH.exists():

        checkpoint_path = (
            FINAL_MODEL_PATH
        )

    elif BEST_MODEL_PATH.exists():

        checkpoint_path = (
            BEST_MODEL_PATH
        )

    else:

        raise FileNotFoundError(
            "No trained checkpoint was found.\n"
            f"Checked: {FINAL_MODEL_PATH}\n"
            f"Checked: {BEST_MODEL_PATH}"
        )

    # Load on CPU first so checkpoint tensors are normal portable tensors.
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    class_names = checkpoint[
        "class_names"
    ]

    configuration = checkpoint.get(
        "configuration",
        {},
    )

    grfm_dimension = int(
        configuration.get(
            "grfm_dim",
            GRFM_DIM,
        )
    )

    dropout = float(
        configuration.get(
            "grfm_dropout",
            GRFM_DROPOUT,
        )
    )

    model = MaxViTTinyGRFM(
        num_classes=len(class_names),
        grfm_dim=grfm_dimension,
        dropout=dropout,
    ).to(DEVICE)

    raw_state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )

    # Remove profiling-only THOP buffers and clone tensors so older
    # contaminated checkpoints can also be loaded safely.
    cleaned_state_dict = {}

    for key, value in raw_state_dict.items():
        final_name = key.rsplit(".", 1)[-1]

        if final_name in {
            "total_ops",
            "total_params",
        }:
            continue

        if torch.is_tensor(value):
            cleaned_state_dict[key] = (
                value.detach()
                .cpu()
                .clone()
            )
        else:
            cleaned_state_dict[key] = value

    incompatible = model.load_state_dict(
        cleaned_state_dict,
        strict=False,
    )

    meaningful_missing = [
        key
        for key in incompatible.missing_keys
        if key.rsplit(".", 1)[-1]
        not in {
            "total_ops",
            "total_params",
        }
    ]

    meaningful_unexpected = [
        key
        for key in incompatible.unexpected_keys
        if key.rsplit(".", 1)[-1]
        not in {
            "total_ops",
            "total_params",
        }
    ]

    if (
        meaningful_missing
        or meaningful_unexpected
    ):
        raise RuntimeError(
            "Checkpoint does not match the model architecture.\n"
            f"Missing keys: {meaningful_missing}\n"
            f"Unexpected keys: {meaningful_unexpected}"
        )

    model = model.to(DEVICE)
    model.eval()

    temperature = float(
        checkpoint.get(
            "temperature",
            1.0,
        )
    )

    if (
        not np.isfinite(temperature)
        or temperature <= 0
    ):
        temperature = 1.0

    return (
        model,
        class_names,
        checkpoint_path,
        temperature,
    )


# ============================================================
# 10. MODEL PREDICTION
# ============================================================

@torch.inference_mode()
def predict_dataset(
    model,
    loader,
    temperature=1.0,
):

    labels_all = []
    probabilities_all = []
    paths_all = []

    for (
        images,
        labels,
        image_paths,
    ) in loader:

        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        with amp_context():

            logits = model(images)

        logits = (
            logits.float()
            / temperature
        )

        probabilities = torch.softmax(
            logits,
            dim=1,
        ).cpu().numpy()

        labels_all.extend(
            labels.numpy()
        )

        probabilities_all.append(
            probabilities
        )

        paths_all.extend(
            image_paths
        )

    labels = np.asarray(
        labels_all,
        dtype=np.int64,
    )

    probabilities = np.concatenate(
        probabilities_all,
        axis=0,
    )

    predictions = probabilities.argmax(
        axis=1
    )

    return (
        labels,
        predictions,
        probabilities,
        paths_all,
    )


# ============================================================
# 11. CALIBRATION METRICS
# ============================================================

def expected_calibration_error(
    labels,
    probabilities,
    number_of_bins=15,
):

    confidences = probabilities.max(
        axis=1
    )

    predictions = probabilities.argmax(
        axis=1
    )

    correctness = (
        predictions == labels
    ).astype(float)

    boundaries = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    ece = 0.0

    for index in range(
        number_of_bins
    ):

        lower = boundaries[index]
        upper = boundaries[index + 1]

        if index == number_of_bins - 1:

            mask = (
                (confidences >= lower)
                & (confidences <= upper)
            )

        else:

            mask = (
                (confidences >= lower)
                & (confidences < upper)
            )

        if mask.any():

            bin_accuracy = (
                correctness[mask].mean()
            )

            bin_confidence = (
                confidences[mask].mean()
            )

            ece += (
                mask.mean()
                * abs(
                    bin_accuracy
                    - bin_confidence
                )
            )

    return float(ece)


def multiclass_brier_score(
    labels,
    probabilities,
):

    one_hot_targets = np.eye(
        probabilities.shape[1]
    )[labels]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot_targets
                ) ** 2,
                axis=1,
            )
        )
    )


# ============================================================
# 12. ROBUSTNESS METRICS
# ============================================================

def compute_metrics(
    labels,
    predictions,
    probabilities,
    positive_index=None,
):

    number_of_classes = (
        probabilities.shape[1]
    )

    metrics = {
        "N": int(len(labels)),

        "Accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                labels,
                predictions,
            ),

        "Macro Precision":
            precision_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            ),

        "Macro Recall":
            recall_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            ),

        "Macro F1":
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            ),

        "Weighted F1":
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            ),

        "MCC":
            matthews_corrcoef(
                labels,
                predictions,
            ),

        "Cohen Kappa":
            cohen_kappa_score(
                labels,
                predictions,
            ),

        "Mean Confidence":
            float(
                probabilities.max(
                    axis=1
                ).mean()
            ),

        "Mean True-Class Probability":
            float(
                probabilities[
                    np.arange(len(labels)),
                    labels,
                ].mean()
            ),

        "ECE":
            expected_calibration_error(
                labels,
                probabilities,
            ),

        "Brier Score":
            multiclass_brier_score(
                labels,
                probabilities,
            ),
    }

    try:

        metrics["NLL"] = log_loss(
            labels,
            probabilities,
            labels=list(
                range(
                    number_of_classes
                )
            ),
        )

    except ValueError:

        metrics["NLL"] = np.nan

    if (
        number_of_classes == 2
        and positive_index is not None
    ):

        binary_labels = (
            labels == positive_index
        ).astype(int)

        positive_scores = (
            probabilities[
                :,
                positive_index,
            ]
        )

        try:

            metrics["ROC-AUC"] = (
                roc_auc_score(
                    binary_labels,
                    positive_scores,
                )
            )

        except ValueError:

            metrics["ROC-AUC"] = np.nan

    else:

        try:

            metrics[
                "Macro ROC-AUC OVR"
            ] = roc_auc_score(
                labels,
                probabilities,
                average="macro",
                multi_class="ovr",
            )

        except ValueError:

            metrics[
                "Macro ROC-AUC OVR"
            ] = np.nan

    return metrics


# ============================================================
# 13. CONFUSION-MATRIX PLOT
# ============================================================

def plot_confusion_matrix(
    confusion,
    class_names,
    corruption,
    severity,
):

    plt.figure(
        figsize=(7, 6)
    )

    plt.imshow(
        confusion,
        interpolation="nearest",
    )

    plt.colorbar()

    positions = np.arange(
        len(class_names)
    )

    plt.xticks(
        positions,
        class_names,
        rotation=35,
        ha="right",
    )

    plt.yticks(
        positions,
        class_names,
    )

    threshold = (
        confusion.max() / 2
        if confusion.max() > 0
        else 0
    )

    for row in range(
        confusion.shape[0]
    ):

        for column in range(
            confusion.shape[1]
        ):

            plt.text(
                column,
                row,
                str(
                    confusion[
                        row,
                        column,
                    ]
                ),
                ha="center",
                va="center",
                color=(
                    "white"
                    if confusion[
                        row,
                        column,
                    ] > threshold
                    else "black"
                ),
            )

    plt.xlabel(
        "Predicted class"
    )

    plt.ylabel(
        "True class"
    )

    plt.title(
        f"{corruption}, severity {severity}"
    )

    plt.tight_layout()

    plt.savefig(
        ROBUSTNESS_DIR
        / (
            f"confusion_{corruption}_"
            f"severity_{severity}.png"
        ),
        dpi=300,
    )

    plt.close()


# ============================================================
# 14. METRIC DEGRADATION CURVES
# ============================================================

def plot_metric_curves(
    results_dataframe,
    metric,
):

    plt.figure(
        figsize=(10, 6)
    )

    for corruption in CORRUPTIONS:

        subset = results_dataframe[
            results_dataframe[
                "Corruption"
            ] == corruption
        ].sort_values("Severity")

        plt.plot(
            subset["Severity"],
            subset[metric],
            marker="o",
            label=corruption,
        )

    plt.xlabel(
        "Severity level"
    )

    plt.ylabel(metric)

    plt.title(
        f"{MODEL_DISPLAY_NAME}: "
        f"{metric} under corruption"
    )

    plt.xticks(SEVERITIES)

    plt.grid(
        True,
        linestyle="--",
        alpha=0.4,
    )

    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    plt.tight_layout()

    filename = (
        metric.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    plt.savefig(
        ROBUSTNESS_DIR
        / f"{filename}_degradation.png",
        dpi=300,
    )

    plt.close()


# ============================================================
# 15. RELATIVE DEGRADATION HEATMAP
# ============================================================

def plot_relative_degradation(
    results_dataframe,
    metric="Accuracy",
):

    clean_value = float(
        results_dataframe.loc[
            results_dataframe[
                "Corruption"
            ] == "clean",
            metric,
        ].iloc[0]
    )

    corrupted = results_dataframe[
        results_dataframe[
            "Corruption"
        ] != "clean"
    ].copy()

    corrupted[
        "Relative Drop (%)"
    ] = (
        100.0
        * (
            clean_value
            - corrupted[metric]
        )
        / max(
            abs(clean_value),
            1e-12,
        )
    )

    pivot_table = corrupted.pivot(
        index="Corruption",
        columns="Severity",
        values="Relative Drop (%)",
    )

    plt.figure(
        figsize=(9, 7)
    )

    heatmap = plt.imshow(
        pivot_table.values,
        aspect="auto",
    )

    plt.colorbar(
        heatmap,
        label=(
            f"Relative {metric} drop (%)"
        ),
    )

    plt.xticks(
        np.arange(
            len(pivot_table.columns)
        ),
        pivot_table.columns,
    )

    plt.yticks(
        np.arange(
            len(pivot_table.index)
        ),
        pivot_table.index,
    )

    for row in range(
        pivot_table.shape[0]
    ):

        for column in range(
            pivot_table.shape[1]
        ):

            value = pivot_table.iloc[
                row,
                column,
            ]

            plt.text(
                column,
                row,
                f"{value:.1f}",
                ha="center",
                va="center",
            )

    plt.xlabel(
        "Severity level"
    )

    plt.ylabel(
        "Corruption"
    )

    plt.title(
        f"Relative {metric} degradation "
        "from clean performance"
    )

    plt.tight_layout()

    filename = (
        metric.lower()
        .replace(" ", "_")
    )

    plt.savefig(
        ROBUSTNESS_DIR
        / (
            f"{filename}_relative_"
            "degradation_heatmap.png"
        ),
        dpi=300,
    )

    plt.close()

    pivot_table.to_csv(
        ROBUSTNESS_DIR
        / (
            f"{filename}_relative_"
            "degradation_heatmap.csv"
        )
    )


# ============================================================
# 16. SAVE CORRUPTION EXAMPLES
# ============================================================

def save_corruption_examples(
    dataset,
    corruption,
    severity,
    number_of_images=3,
):

    examples_directory = (
        ROBUSTNESS_DIR
        / "corruption_examples"
    )

    examples_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    selected_indices = np.linspace(
        0,
        len(dataset) - 1,
        min(
            number_of_images,
            len(dataset),
        ),
        dtype=int,
    )

    figure, axes = plt.subplots(
        len(selected_indices),
        2,
        figsize=(
            7,
            3.2 * len(
                selected_indices
            ),
        ),
    )

    axes = np.atleast_2d(
        axes
    )

    resize = transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    )

    for row, index in enumerate(
        selected_indices
    ):

        image_path, label = (
            dataset.samples[index]
        )

        original_image = resize(
            Image.open(
                image_path
            ).convert("RGB")
        )

        random_generator = (
            np.random.default_rng(
                SEED
                + index * 1009
                + severity * 9176
            )
        )

        corrupted_image = (
            apply_corruption(
                image=original_image,
                corruption=corruption,
                severity=severity,
                rng=random_generator,
            )
        )

        axes[row, 0].imshow(
            original_image
        )

        axes[row, 0].set_title(
            f"Clean: "
            f"{dataset.classes[label]}"
        )

        axes[row, 1].imshow(
            corrupted_image
        )

        axes[row, 1].set_title(
            f"{corruption}, "
            f"severity {severity}"
        )

        axes[row, 0].axis("off")
        axes[row, 1].axis("off")

    plt.tight_layout()

    plt.savefig(
        examples_directory
        / (
            f"{corruption}_severity_"
            f"{severity}_examples.png"
        ),
        dpi=250,
    )

    plt.close()


# ============================================================
# 17. MAIN ROBUSTNESS ANALYSIS
# ============================================================

def main():

    set_seed(SEED)

    if not TEST_DIR.exists():

        raise FileNotFoundError(
            f"Test directory not found: "
            f"{TEST_DIR}"
        )

    (
        model,
        class_names,
        checkpoint_path,
        temperature,
    ) = load_trained_model()

    (
        clean_dataset,
        clean_loader,
    ) = create_loader(
        corruption="clean",
        severity=0,
    )

    if (
        clean_dataset.classes
        != class_names
    ):

        raise ValueError(
            "Test classes do not match "
            "checkpoint classes.\n"
            f"Test classes: "
            f"{clean_dataset.classes}\n"
            f"Checkpoint classes: "
            f"{class_names}"
        )

    number_of_classes = len(
        class_names
    )

    if number_of_classes == 2:

        if POSITIVE_CLASS_NAME is None:

            positive_index = 1

        else:

            if (
                POSITIVE_CLASS_NAME
                not in class_names
            ):

                raise ValueError(
                    "POSITIVE_CLASS_NAME must "
                    f"be one of {class_names}"
                )

            positive_index = (
                class_names.index(
                    POSITIVE_CLASS_NAME
                )
            )

    else:

        positive_index = None

    print("=" * 80)
    print(
        f"{MODEL_DISPLAY_NAME} "
        "Robustness Analysis"
    )
    print("=" * 80)

    print(f"Device      : {DEVICE}")
    print(
        f"Checkpoint  : "
        f"{checkpoint_path}"
    )
    print(
        f"Temperature : "
        f"{temperature:.6f}"
    )
    print(
        f"Classes     : "
        f"{class_names}"
    )
    print(
        f"Test images : "
        f"{len(clean_dataset)}"
    )

    result_rows = []
    prediction_rows = []

    # --------------------------------------------------------
    # Clean test performance
    # --------------------------------------------------------

    (
        clean_labels,
        clean_predictions,
        clean_probabilities,
        clean_paths,
    ) = predict_dataset(
        model=model,
        loader=clean_loader,
        temperature=temperature,
    )

    clean_metrics = compute_metrics(
        labels=clean_labels,
        predictions=clean_predictions,
        probabilities=clean_probabilities,
        positive_index=positive_index,
    )

    result_rows.append({
        "Corruption": "clean",
        "Severity": 0,
        **clean_metrics,
    })

    clean_prediction_information = {
        image_path: {
            "prediction": int(prediction),
            "confidence": float(confidence),
        }
        for (
            image_path,
            prediction,
            confidence,
        ) in zip(
            clean_paths,
            clean_predictions,
            clean_probabilities.max(
                axis=1
            ),
        )
    }

    print(
        "\nClean test results"
    )

    print(
        f"Accuracy : "
        f"{clean_metrics['Accuracy']:.6f}"
    )

    print(
        f"Macro F1 : "
        f"{clean_metrics['Macro F1']:.6f}"
    )

    print(
        f"MCC      : "
        f"{clean_metrics['MCC']:.6f}"
    )

    # --------------------------------------------------------
    # Corrupted evaluation
    # --------------------------------------------------------

    for corruption in CORRUPTIONS:

        for severity in SEVERITIES:

            dataset, loader = create_loader(
                corruption=corruption,
                severity=severity,
            )

            (
                labels,
                predictions,
                probabilities,
                image_paths,
            ) = predict_dataset(
                model=model,
                loader=loader,
                temperature=temperature,
            )

            metrics = compute_metrics(
                labels=labels,
                predictions=predictions,
                probabilities=probabilities,
                positive_index=positive_index,
            )

            result_rows.append({
                "Corruption": corruption,
                "Severity": severity,
                **metrics,
            })

            for (
                image_path,
                true_label,
                prediction,
                probability_vector,
            ) in zip(
                image_paths,
                labels,
                predictions,
                probabilities,
            ):

                clean_information = (
                    clean_prediction_information[
                        image_path
                    ]
                )

                prediction_rows.append({
                    "Corruption":
                        corruption,

                    "Severity":
                        severity,

                    "Image Path":
                        image_path,

                    "True Label":
                        int(true_label),

                    "True Class":
                        class_names[
                            true_label
                        ],

                    "Predicted Label":
                        int(prediction),

                    "Predicted Class":
                        class_names[
                            prediction
                        ],

                    "Correct":
                        int(
                            prediction
                            == true_label
                        ),

                    "Confidence":
                        float(
                            probability_vector.max()
                        ),

                    "True-Class Probability":
                        float(
                            probability_vector[
                                true_label
                            ]
                        ),

                    "Clean Predicted Class":
                        class_names[
                            clean_information[
                                "prediction"
                            ]
                        ],

                    "Clean Confidence":
                        clean_information[
                            "confidence"
                        ],

                    "Prediction Changed":
                        int(
                            prediction
                            != clean_information[
                                "prediction"
                            ]
                        ),
                })

            print(
                f"{corruption:22s} | "
                f"severity={severity} | "
                f"accuracy="
                f"{metrics['Accuracy']:.5f} | "
                f"F1="
                f"{metrics['Macro F1']:.5f} | "
                f"MCC="
                f"{metrics['MCC']:.5f}"
            )

            # Save the confusion matrix and image examples
            # at the strongest corruption severity.
            if severity == max(
                SEVERITIES
            ):

                confusion = confusion_matrix(
                    labels,
                    predictions,
                    labels=np.arange(
                        number_of_classes
                    ),
                )

                plot_confusion_matrix(
                    confusion=confusion,
                    class_names=class_names,
                    corruption=corruption,
                    severity=severity,
                )

                save_corruption_examples(
                    dataset=dataset,
                    corruption=corruption,
                    severity=severity,
                    number_of_images=3,
                )

    results_dataframe = pd.DataFrame(
        result_rows
    )

    predictions_dataframe = pd.DataFrame(
        prediction_rows
    )

    # --------------------------------------------------------
    # Calculate degradation compared with clean results
    # --------------------------------------------------------

    clean_result = (
        results_dataframe[
            results_dataframe[
                "Corruption"
            ] == "clean"
        ].iloc[0]
    )

    degradation_metrics = [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "MCC",
        "Mean Confidence",
    ]

    for metric in degradation_metrics:

        results_dataframe[
            f"{metric} Absolute Drop"
        ] = (
            clean_result[metric]
            - results_dataframe[metric]
        )

        results_dataframe[
            f"{metric} Relative Drop (%)"
        ] = (
            100.0
            * (
                clean_result[metric]
                - results_dataframe[metric]
            )
            / max(
                abs(
                    clean_result[metric]
                ),
                1e-12,
            )
        )

    results_dataframe.to_csv(
        ROBUSTNESS_DIR
        / (
            "robustness_metrics_"
            "all_corruptions.csv"
        ),
        index=False,
    )

    predictions_dataframe.to_csv(
        ROBUSTNESS_DIR
        / (
            "robustness_predictions_"
            "all_images.csv"
        ),
        index=False,
    )

    # --------------------------------------------------------
    # Summary by corruption
    # --------------------------------------------------------

    corrupted_results = (
        results_dataframe[
            results_dataframe[
                "Corruption"
            ] != "clean"
        ].copy()
    )

    corruption_summary = (
        corrupted_results
        .groupby("Corruption")
        .agg(
            Mean_Accuracy=(
                "Accuracy",
                "mean",
            ),

            Worst_Accuracy=(
                "Accuracy",
                "min",
            ),

            Mean_Macro_F1=(
                "Macro F1",
                "mean",
            ),

            Worst_Macro_F1=(
                "Macro F1",
                "min",
            ),

            Mean_MCC=(
                "MCC",
                "mean",
            ),

            Mean_Confidence=(
                "Mean Confidence",
                "mean",
            ),

            Mean_ECE=(
                "ECE",
                "mean",
            ),

            Mean_Accuracy_Relative_Drop_Pct=(
                "Accuracy Relative Drop (%)",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            "Mean_Accuracy"
        )
    )

    corruption_summary.to_csv(
        ROBUSTNESS_DIR
        / (
            "robustness_summary_"
            "by_corruption.csv"
        ),
        index=False,
    )

    # --------------------------------------------------------
    # Prediction-stability analysis
    # --------------------------------------------------------

    prediction_stability = (
        predictions_dataframe
        .groupby([
            "Corruption",
            "Severity",
        ])
        .agg(
            Prediction_Change_Rate=(
                "Prediction Changed",
                "mean",
            ),

            Corrupted_Error_Rate=(
                "Correct",
                lambda values:
                    1.0 - values.mean(),
            ),

            Mean_Confidence=(
                "Confidence",
                "mean",
            ),

            Mean_True_Class_Probability=(
                "True-Class Probability",
                "mean",
            ),
        )
        .reset_index()
    )

    prediction_stability.to_csv(
        ROBUSTNESS_DIR
        / (
            "prediction_stability_"
            "summary.csv"
        ),
        index=False,
    )

    # --------------------------------------------------------
    # Save robustness plots
    # --------------------------------------------------------

    metrics_to_plot = [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "MCC",
        "Mean Confidence",
        "ECE",
    ]

    for metric in metrics_to_plot:

        plot_metric_curves(
            results_dataframe,
            metric,
        )

    plot_relative_degradation(
        results_dataframe,
        metric="Accuracy",
    )

    plot_relative_degradation(
        results_dataframe,
        metric="Macro F1",
    )

    # --------------------------------------------------------
    # Final JSON summary
    # --------------------------------------------------------

    run_summary = {
        "Model":
            MODEL_DISPLAY_NAME,

        "Checkpoint":
            checkpoint_path,

        "Device":
            str(DEVICE),

        "Temperature":
            temperature,

        "Classes":
            class_names,

        "Test Images":
            len(clean_dataset),

        "Corruptions":
            CORRUPTIONS,

        "Severity Levels":
            SEVERITIES,

        "Clean Metrics":
            clean_metrics,

        "Most Harmful Corruption":
            corruption_summary.iloc[
                0
            ]["Corruption"],

        "Most Robust Corruption":
            corruption_summary.iloc[
                -1
            ]["Corruption"],
    }

    save_json(
        run_summary,
        ROBUSTNESS_DIR
        / "robustness_run_summary.json",
    )

    print(
        "\nRobustness analysis "
        "completed successfully."
    )

    print(
        f"Results saved to: "
        f"{ROBUSTNESS_DIR}"
    )

    print(
        "\nCorruptions ranked from "
        "most harmful to least harmful:"
    )

    print(
        corruption_summary.to_string(
            index=False
        )
    )


if __name__ == "__main__":

    main()

In [ ]:

# ============================================================
# MAXVIT-TINY + GRFM
# A) 3-FOLD CV ON AUGMENTED TRAINING PATH
# B) 3-SEED STABILITY ON FIXED TRAIN/VAL/TEST PATHS
#
# Metrics:
# Accuracy, Macro Precision, Macro F1, MCC, Cohen Kappa
#
# Epochs:
# 15 for every fold and every seed run
# ============================================================

import contextlib
import json
import os
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path

try:
    import timm
except ImportError:
    print("Installing timm...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "timm"]
    )
    import timm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Experiment A: 3-fold CV uses this augmented training folder.
# ------------------------------------------------------------
CV_AUGMENTED_TRAIN_DIR = Path(
    "/kaggle/working/PVMD_Train_Augmented"
)

# ------------------------------------------------------------
# Experiment B: three-seed stability uses fixed splits.
# ------------------------------------------------------------
TRAIN_DIR = Path(
    "/kaggle/working/PVMD_Train_Augmented"
)

VAL_DIR = Path(
    "/kaggle/working/PVMD_Split/validation"
)

TEST_DIR = Path(
    "/kaggle/working/PVMD_Split/test"
)

if not VAL_DIR.exists():
    alternate_val = Path(
        "/kaggle/working/PVMD_Split/val"
    )
    if alternate_val.exists():
        VAL_DIR = alternate_val

OUTPUT_DIR = Path(
    "/kaggle/working/MaxViT_Tiny_GRFM_CV_Seed_Results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CV_OUTPUT_DIR = OUTPUT_DIR / "Three_Fold_CV"
SEED_OUTPUT_DIR = OUTPUT_DIR / "Three_Seed_Stability"

CV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "maxvit_tiny_tf_224.in1k"
MODEL_DISPLAY_NAME = "MaxViT-Tiny-224 + GRFM"

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

# Requested by user.
EPOCHS = 15

LEARNING_RATE = 1e-4
BACKBONE_LR = 2e-5
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 7
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
MIN_LR = 1e-7

LABEL_SMOOTHING = 0.10
GRADIENT_CLIP = 1.0

GRFM_DIM = 512
GRFM_DROPOUT = 0.30
GRFM_REDUCTION = 4

PRETRAINED = True

N_SPLITS = 3
CV_RANDOM_SEED = 42

# Three independent fixed-split training runs.
STABILITY_SEEDS = [42, 123, 2024]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
USE_AMP = DEVICE.type == "cuda"


# ============================================================
# 2. REPRODUCIBILITY AND UTILITIES
# ============================================================

def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def amp_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        )

    return contextlib.nullcontext()


def save_json(data: dict, path: Path) -> None:
    cleaned = {}

    for key, value in data.items():
        if isinstance(value, Path):
            value = str(value)

        if isinstance(value, (np.integer, np.floating)):
            value = value.item()

        if isinstance(value, float) and not np.isfinite(value):
            value = None

        cleaned[key] = value

    with path.open("w", encoding="utf-8") as file:
        json.dump(cleaned, file, indent=2)


def validate_directories() -> None:
    required = [
        CV_AUGMENTED_TRAIN_DIR,
        TRAIN_DIR,
        VAL_DIR,
        TEST_DIR,
    ]

    missing = [
        str(path)
        for path in required
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "The following folders were not found:\n- "
            + "\n- ".join(missing)
        )


# ============================================================
# 3. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomVerticalFlip(p=0.10),
    transforms.RandomRotation(degrees=10),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05,
        hue=0.02,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),

    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.08),
        ratio=(0.5, 2.0),
    ),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# ============================================================
# 4. DATASETS
# ============================================================

class IndexedImageFolderDataset(Dataset):
    """
    Uses selected indices from an ImageFolder sample list.
    """

    def __init__(
        self,
        samples,
        indices,
        transform,
    ):
        self.samples = samples
        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        dataset_index = int(
            self.indices[item]
        )

        image_path, label = (
            self.samples[dataset_index]
        )

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        image = self.transform(image)

        return image, int(label)


def create_standard_datasets():
    train_dataset = datasets.ImageFolder(
        root=str(TRAIN_DIR),
        transform=train_transform,
    )

    validation_dataset = datasets.ImageFolder(
        root=str(VAL_DIR),
        transform=evaluation_transform,
    )

    test_dataset = datasets.ImageFolder(
        root=str(TEST_DIR),
        transform=evaluation_transform,
    )

    class_names = train_dataset.classes

    for split_name, dataset in [
        ("validation", validation_dataset),
        ("test", test_dataset),
    ]:
        if dataset.classes != class_names:
            raise ValueError(
                f"{split_name} classes do not match training classes.\n"
                f"Training: {class_names}\n"
                f"{split_name}: {dataset.classes}"
            )

    return (
        train_dataset,
        validation_dataset,
        test_dataset,
        class_names,
        train_dataset.class_to_idx,
    )


def create_loader(
    dataset,
    shuffle,
    seed,
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        worker_init_fn=seed_worker,
        generator=generator if shuffle else None,
    )


# ============================================================
# 5. GRFM MODULE
# ============================================================

class GRFM(nn.Module):

    def __init__(
        self,
        input_dim: int,
        output_dim: int = 512,
        reduction: int = 4,
        dropout: float = 0.30,
    ):
        super().__init__()

        hidden_dim = max(
            output_dim // reduction,
            32,
        )

        self.projection = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

        self.feature_transform = nn.Sequential(
            nn.Linear(output_dim, output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, output_dim),
        )

        self.feature_gate = nn.Sequential(
            nn.Linear(output_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

        self.output_norm = nn.LayerNorm(
            output_dim
        )

        self.output_dropout = nn.Dropout(
            dropout
        )

    def forward(self, features):
        projected = self.projection(
            features
        )

        transformed = self.feature_transform(
            projected
        )

        gate = self.feature_gate(
            projected
        )

        refined = (
            projected
            + transformed * gate
        )

        refined = self.output_norm(
            refined
        )

        refined = self.output_dropout(
            refined
        )

        return refined


# ============================================================
# 6. MODEL
# ============================================================

class MaxViTTinyGRFM(nn.Module):

    def __init__(
        self,
        num_classes: int,
        grfm_dim: int = 512,
        dropout: float = 0.30,
        pretrained: bool = True,
    ):
        super().__init__()

        try:
            self.backbone = timm.create_model(
                MODEL_NAME,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )

        except Exception as error:
            if not pretrained:
                raise

            print(
                "\nWarning: pretrained weights could not be loaded.\n"
                f"Reason: {error}\n"
                "Using pretrained=False."
            )

            self.backbone = timm.create_model(
                MODEL_NAME,
                pretrained=False,
                num_classes=0,
                global_pool="avg",
            )

        backbone_dim = (
            self.backbone.num_features
        )

        self.grfm = GRFM(
            input_dim=backbone_dim,
            output_dim=grfm_dim,
            reduction=GRFM_REDUCTION,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                grfm_dim,
                grfm_dim // 2,
            ),
            nn.LayerNorm(
                grfm_dim // 2
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                grfm_dim // 2,
                num_classes,
            ),
        )

    def forward(self, images):
        backbone_features = self.backbone(
            images
        )

        refined_features = self.grfm(
            backbone_features
        )

        logits = self.classifier(
            refined_features
        )

        return logits


# ============================================================
# 7. TRAINING OBJECTS
# ============================================================

def create_training_objects(
    num_classes,
):
    model = MaxViTTinyGRFM(
        num_classes=num_classes,
        grfm_dim=GRFM_DIM,
        dropout=GRFM_DROPOUT,
        pretrained=PRETRAINED,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING,
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params":
                    model.backbone.parameters(),
                "lr":
                    BACKBONE_LR,
            },
            {
                "params":
                    model.grfm.parameters(),
                "lr":
                    LEARNING_RATE,
            },
            {
                "params":
                    model.classifier.parameters(),
                "lr":
                    LEARNING_RATE,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=SCHEDULER_FACTOR,
            patience=SCHEDULER_PATIENCE,
            min_lr=MIN_LR,
        )
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    return (
        model,
        criterion,
        optimizer,
        scheduler,
        scaler,
    )


# ============================================================
# 8. METRICS
# ============================================================

def compute_metrics(
    labels,
    predictions,
):
    return {
        "Accuracy":
            accuracy_score(
                labels,
                predictions,
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                labels,
                predictions,
            ),

        "Macro Precision":
            precision_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            ),

        "Macro Recall":
            recall_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            ),

        "Macro F1":
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            ),

        "Weighted F1":
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            ),

        "MCC":
            matthews_corrcoef(
                labels,
                predictions,
            ),

        "Cohen Kappa":
            cohen_kappa_score(
                labels,
                predictions,
            ),
    }


# ============================================================
# 9. TRAINING AND EVALUATION
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
):
    model.train()

    running_loss = 0.0
    labels_all = []
    predictions_all = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with amp_context():
            logits = model(images)
            loss = criterion(
                logits,
                labels,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP,
        )

        scaler.step(optimizer)
        scaler.update()

        predictions = logits.argmax(
            dim=1
        )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        labels_all.extend(
            labels.detach()
            .cpu()
            .numpy()
        )

        predictions_all.extend(
            predictions.detach()
            .cpu()
            .numpy()
        )

    labels_all = np.asarray(
        labels_all
    )

    predictions_all = np.asarray(
        predictions_all
    )

    metrics = compute_metrics(
        labels_all,
        predictions_all,
    )

    metrics["Loss"] = (
        running_loss
        / len(loader.dataset)
    )

    return metrics


@torch.inference_mode()
def evaluate_model(
    model,
    loader,
    criterion,
):
    model.eval()

    running_loss = 0.0
    labels_all = []
    predictions_all = []
    probabilities_all = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels_device = labels.to(
            DEVICE,
            non_blocking=True,
        )

        with amp_context():
            logits = model(images)

            loss = criterion(
                logits,
                labels_device,
            )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = probabilities.argmax(
            dim=1
        )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        labels_all.extend(
            labels.numpy()
        )

        predictions_all.extend(
            predictions.cpu().numpy()
        )

        probabilities_all.append(
            probabilities.cpu().numpy()
        )

    labels_all = np.asarray(
        labels_all,
        dtype=np.int64,
    )

    predictions_all = np.asarray(
        predictions_all,
        dtype=np.int64,
    )

    probabilities_all = np.concatenate(
        probabilities_all,
        axis=0,
    )

    metrics = compute_metrics(
        labels_all,
        predictions_all,
    )

    metrics["Loss"] = (
        running_loss
        / len(loader.dataset)
    )

    return (
        metrics,
        labels_all,
        predictions_all,
        probabilities_all,
    )


# ============================================================
# 10. GENERIC TRAINING RUN
# ============================================================

def train_and_select_best(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    checkpoint_path,
    run_label,
):
    history = []

    best_f1 = -np.inf
    best_loss = np.inf
    no_improvement = 0

    run_start = time.time()

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        epoch_start = time.time()

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
        )

        (
            validation_metrics,
            _,
            _,
            _,
        ) = evaluate_model(
            model=model,
            loader=validation_loader,
            criterion=criterion,
        )

        scheduler.step(
            validation_metrics[
                "Macro F1"
            ]
        )

        epoch_seconds = (
            time.time()
            - epoch_start
        )

        history.append({
            "Epoch": epoch,

            "Train Loss":
                train_metrics["Loss"],

            "Train Accuracy":
                train_metrics["Accuracy"],

            "Train Macro Precision":
                train_metrics[
                    "Macro Precision"
                ],

            "Train Macro F1":
                train_metrics["Macro F1"],

            "Train MCC":
                train_metrics["MCC"],

            "Train Cohen Kappa":
                train_metrics[
                    "Cohen Kappa"
                ],

            "Validation Loss":
                validation_metrics["Loss"],

            "Validation Accuracy":
                validation_metrics[
                    "Accuracy"
                ],

            "Validation Macro Precision":
                validation_metrics[
                    "Macro Precision"
                ],

            "Validation Macro F1":
                validation_metrics[
                    "Macro F1"
                ],

            "Validation MCC":
                validation_metrics["MCC"],

            "Validation Cohen Kappa":
                validation_metrics[
                    "Cohen Kappa"
                ],

            "Backbone LR":
                optimizer.param_groups[
                    0
                ]["lr"],

            "Head LR":
                optimizer.param_groups[
                    1
                ]["lr"],

            "Epoch Time Seconds":
                epoch_seconds,
        })

        print(
            f"{run_label} | "
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Acc="
            f"{train_metrics['Accuracy']:.5f} | "
            f"Val Acc="
            f"{validation_metrics['Accuracy']:.5f} | "
            f"Val Pre="
            f"{validation_metrics['Macro Precision']:.5f} | "
            f"Val F1="
            f"{validation_metrics['Macro F1']:.5f} | "
            f"Val MCC="
            f"{validation_metrics['MCC']:.5f} | "
            f"Val Kappa="
            f"{validation_metrics['Cohen Kappa']:.5f}"
        )

        improved = (
            validation_metrics[
                "Macro F1"
            ] > best_f1 + 1e-5
            or (
                abs(
                    validation_metrics[
                        "Macro F1"
                    ] - best_f1
                ) <= 1e-5
                and validation_metrics[
                    "Loss"
                ] < best_loss
            )
        )

        if improved:
            best_f1 = (
                validation_metrics[
                    "Macro F1"
                ]
            )

            best_loss = (
                validation_metrics[
                    "Loss"
                ]
            )

            no_improvement = 0

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict":
                        model.state_dict(),
                    "best_validation_macro_f1":
                        best_f1,
                    "best_validation_loss":
                        best_loss,
                },
                checkpoint_path,
            )

            print("  Best checkpoint saved.")

        else:
            no_improvement += 1

            print(
                "  No improvement: "
                f"{no_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        if (
            no_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                "  Early stopping activated."
            )
            break

    history_dataframe = pd.DataFrame(
        history
    )

    total_seconds = (
        time.time() - run_start
    )

    return (
        history_dataframe,
        total_seconds,
    )


# ============================================================
# 11. SAVE PREDICTIONS
# ============================================================

def save_predictions(
    labels,
    predictions,
    probabilities,
    class_names,
    path,
):
    dataframe = pd.DataFrame({
        "True Label": labels,

        "True Class": [
            class_names[index]
            for index in labels
        ],

        "Predicted Label":
            predictions,

        "Predicted Class": [
            class_names[index]
            for index in predictions
        ],

        "Correct":
            (
                labels
                == predictions
            ).astype(int),

        "Confidence":
            probabilities.max(
                axis=1
            ),
    })

    for class_index, class_name in enumerate(
        class_names
    ):
        dataframe[
            f"Probability {class_name}"
        ] = probabilities[
            :,
            class_index,
        ]

    dataframe.to_csv(
        path,
        index=False,
    )


# ============================================================
# 12. EXPERIMENT A: 3-FOLD CV
# ============================================================

def run_three_fold_cv():
    print("\n")
    print("=" * 90)
    print(
        "EXPERIMENT A: 3-FOLD CV ON AUGMENTED TRAINING PATH"
    )
    print("=" * 90)

    base_dataset = datasets.ImageFolder(
        root=str(
            CV_AUGMENTED_TRAIN_DIR
        )
    )

    samples = base_dataset.samples
    labels = np.asarray(
        base_dataset.targets,
        dtype=np.int64,
    )

    class_names = (
        base_dataset.classes
    )

    class_counts = np.bincount(
        labels,
        minlength=len(class_names),
    )

    if class_counts.min() < N_SPLITS:
        raise ValueError(
            f"Every class needs at least "
            f"{N_SPLITS} images for CV."
        )

    print(
        f"CV dataset: "
        f"{CV_AUGMENTED_TRAIN_DIR}"
    )
    print(
        f"Images: {len(samples)}"
    )
    print(
        f"Classes: {class_names}"
    )

    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=CV_RANDOM_SEED,
    )

    fold_results = []

    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        splitter.split(
            np.zeros(len(labels)),
            labels,
        ),
        start=1,
    ):
        fold_seed = (
            CV_RANDOM_SEED
            + fold_number * 1000
        )

        set_seed(fold_seed)

        fold_directory = (
            CV_OUTPUT_DIR
            / f"fold_{fold_number}"
        )

        fold_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        train_dataset = (
            IndexedImageFolderDataset(
                samples=samples,
                indices=train_indices,
                transform=train_transform,
            )
        )

        validation_dataset = (
            IndexedImageFolderDataset(
                samples=samples,
                indices=validation_indices,
                transform=evaluation_transform,
            )
        )

        train_loader = create_loader(
            train_dataset,
            shuffle=True,
            seed=fold_seed,
        )

        validation_loader = create_loader(
            validation_dataset,
            shuffle=False,
            seed=fold_seed,
        )

        (
            model,
            criterion,
            optimizer,
            scheduler,
            scaler,
        ) = create_training_objects(
            num_classes=len(
                class_names
            )
        )

        checkpoint_path = (
            fold_directory
            / "best_model.pth"
        )

        (
            history_dataframe,
            fold_seconds,
        ) = train_and_select_best(
            model=model,
            train_loader=train_loader,
            validation_loader=
                validation_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            checkpoint_path=
                checkpoint_path,
            run_label=(
                f"CV Fold "
                f"{fold_number}/{N_SPLITS}"
            ),
        )

        history_dataframe.to_csv(
            fold_directory
            / "training_history.csv",
            index=False,
        )

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
        )

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        (
            metrics,
            fold_labels,
            fold_predictions,
            fold_probabilities,
        ) = evaluate_model(
            model=model,
            loader=validation_loader,
            criterion=criterion,
        )

        save_predictions(
            labels=fold_labels,
            predictions=
                fold_predictions,
            probabilities=
                fold_probabilities,
            class_names=class_names,
            path=(
                fold_directory
                / "validation_predictions.csv"
            ),
        )

        confusion = confusion_matrix(
            fold_labels,
            fold_predictions,
            labels=np.arange(
                len(class_names)
            ),
        )

        pd.DataFrame(
            confusion,
            index=class_names,
            columns=class_names,
        ).to_csv(
            fold_directory
            / "confusion_matrix.csv"
        )

        fold_result = {
            "Fold": fold_number,
            "Best Epoch":
                checkpoint["epoch"],
            "Train Samples":
                len(train_dataset),
            "Validation Samples":
                len(validation_dataset),
            "Accuracy":
                metrics["Accuracy"],
            "Macro Precision":
                metrics[
                    "Macro Precision"
                ],
            "Macro F1":
                metrics["Macro F1"],
            "MCC":
                metrics["MCC"],
            "Cohen Kappa":
                metrics[
                    "Cohen Kappa"
                ],
            "Balanced Accuracy":
                metrics[
                    "Balanced Accuracy"
                ],
            "Macro Recall":
                metrics[
                    "Macro Recall"
                ],
            "Weighted F1":
                metrics[
                    "Weighted F1"
                ],
            "Validation Loss":
                metrics["Loss"],
            "Training Time Minutes":
                fold_seconds / 60.0,
        }

        fold_results.append(
            fold_result
        )

        print(
            "\nCV fold result | "
            f"Fold={fold_number} | "
            f"Acc="
            f"{metrics['Accuracy']:.6f} | "
            f"Pre="
            f"{metrics['Macro Precision']:.6f} | "
            f"F1="
            f"{metrics['Macro F1']:.6f} | "
            f"MCC="
            f"{metrics['MCC']:.6f} | "
            f"Kappa="
            f"{metrics['Cohen Kappa']:.6f}"
        )

        del model
        del optimizer
        del scheduler
        del scaler

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_dataframe = pd.DataFrame(
        fold_results
    )

    fold_dataframe.to_csv(
        CV_OUTPUT_DIR
        / "cv_fold_metrics.csv",
        index=False,
    )

    metric_names = [
        "Accuracy",
        "Macro Precision",
        "Macro F1",
        "MCC",
        "Cohen Kappa",
    ]

    summary_rows = []

    for metric in metric_names:
        summary_rows.append({
            "Metric": metric,
            "Mean":
                fold_dataframe[
                    metric
                ].mean(),
            "Standard Deviation":
                fold_dataframe[
                    metric
                ].std(ddof=1),
            "Minimum":
                fold_dataframe[
                    metric
                ].min(),
            "Maximum":
                fold_dataframe[
                    metric
                ].max(),
        })

    summary_dataframe = pd.DataFrame(
        summary_rows
    )

    summary_dataframe.to_csv(
        CV_OUTPUT_DIR
        / "cv_mean_std_summary.csv",
        index=False,
    )

    print("\n" + "-" * 96)
    print(
        "3-FOLD CV RESULTS"
    )
    print("-" * 96)

    print(
        fold_dataframe[
            [
                "Fold",
                "Best Epoch",
                "Accuracy",
                "Macro Precision",
                "Macro F1",
                "MCC",
                "Cohen Kappa",
            ]
        ].round(6).to_string(
            index=False
        )
    )

    print(
        "\n3-fold CV mean ± SD"
    )

    for _, row in (
        summary_dataframe.iterrows()
    ):
        print(
            f"{row['Metric']:16s}: "
            f"{row['Mean']:.6f} ± "
            f"{row['Standard Deviation']:.6f}"
        )

    return (
        fold_dataframe,
        summary_dataframe,
        class_names,
    )


# ============================================================
# 13. EXPERIMENT B: 3-SEED FIXED-SPLIT STABILITY
# ============================================================

def run_three_seed_stability():
    print("\n")
    print("=" * 90)
    print(
        "EXPERIMENT B: 3-SEED STABILITY ON FIXED TRAIN/VAL/TEST"
    )
    print("=" * 90)

    (
        _,
        _,
        _,
        class_names,
        class_to_idx,
    ) = create_standard_datasets()

    seed_results = []

    for seed in STABILITY_SEEDS:
        set_seed(seed)

        seed_directory = (
            SEED_OUTPUT_DIR
            / f"seed_{seed}"
        )

        seed_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        # Recreate datasets and loaders for each seed.
        (
            train_dataset,
            validation_dataset,
            test_dataset,
            current_classes,
            _,
        ) = create_standard_datasets()

        if current_classes != class_names:
            raise RuntimeError(
                "Class order changed "
                "between seed runs."
            )

        train_loader = create_loader(
            train_dataset,
            shuffle=True,
            seed=seed,
        )

        validation_loader = create_loader(
            validation_dataset,
            shuffle=False,
            seed=seed,
        )

        test_loader = create_loader(
            test_dataset,
            shuffle=False,
            seed=seed,
        )

        (
            model,
            criterion,
            optimizer,
            scheduler,
            scaler,
        ) = create_training_objects(
            num_classes=len(
                class_names
            )
        )

        checkpoint_path = (
            seed_directory
            / "best_model.pth"
        )

        (
            history_dataframe,
            seed_seconds,
        ) = train_and_select_best(
            model=model,
            train_loader=train_loader,
            validation_loader=
                validation_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            checkpoint_path=
                checkpoint_path,
            run_label=f"Seed {seed}",
        )

        history_dataframe.to_csv(
            seed_directory
            / "training_history.csv",
            index=False,
        )

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
        )

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        (
            validation_metrics,
            validation_labels,
            validation_predictions,
            validation_probabilities,
        ) = evaluate_model(
            model=model,
            loader=validation_loader,
            criterion=criterion,
        )

        (
            test_metrics,
            test_labels,
            test_predictions,
            test_probabilities,
        ) = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion,
        )

        save_predictions(
            labels=validation_labels,
            predictions=
                validation_predictions,
            probabilities=
                validation_probabilities,
            class_names=class_names,
            path=(
                seed_directory
                / "validation_predictions.csv"
            ),
        )

        save_predictions(
            labels=test_labels,
            predictions=
                test_predictions,
            probabilities=
                test_probabilities,
            class_names=class_names,
            path=(
                seed_directory
                / "test_predictions.csv"
            ),
        )

        validation_confusion = (
            confusion_matrix(
                validation_labels,
                validation_predictions,
                labels=np.arange(
                    len(class_names)
                ),
            )
        )

        test_confusion = confusion_matrix(
            test_labels,
            test_predictions,
            labels=np.arange(
                len(class_names)
            ),
        )

        pd.DataFrame(
            validation_confusion,
            index=class_names,
            columns=class_names,
        ).to_csv(
            seed_directory
            / "validation_confusion_matrix.csv"
        )

        pd.DataFrame(
            test_confusion,
            index=class_names,
            columns=class_names,
        ).to_csv(
            seed_directory
            / "test_confusion_matrix.csv"
        )

        seed_result = {
            "Seed": seed,
            "Best Epoch":
                checkpoint["epoch"],

            "Validation Accuracy":
                validation_metrics[
                    "Accuracy"
                ],

            "Validation Macro Precision":
                validation_metrics[
                    "Macro Precision"
                ],

            "Validation Macro F1":
                validation_metrics[
                    "Macro F1"
                ],

            "Validation MCC":
                validation_metrics[
                    "MCC"
                ],

            "Validation Cohen Kappa":
                validation_metrics[
                    "Cohen Kappa"
                ],

            "Test Accuracy":
                test_metrics[
                    "Accuracy"
                ],

            "Test Macro Precision":
                test_metrics[
                    "Macro Precision"
                ],

            "Test Macro F1":
                test_metrics[
                    "Macro F1"
                ],

            "Test MCC":
                test_metrics[
                    "MCC"
                ],

            "Test Cohen Kappa":
                test_metrics[
                    "Cohen Kappa"
                ],

            "Test Balanced Accuracy":
                test_metrics[
                    "Balanced Accuracy"
                ],

            "Test Macro Recall":
                test_metrics[
                    "Macro Recall"
                ],

            "Test Weighted F1":
                test_metrics[
                    "Weighted F1"
                ],

            "Test Loss":
                test_metrics[
                    "Loss"
                ],

            "Training Time Minutes":
                seed_seconds / 60.0,
        }

        seed_results.append(
            seed_result
        )

        pd.DataFrame(
            [seed_result]
        ).to_csv(
            seed_directory
            / "seed_metrics.csv",
            index=False,
        )

        print(
            "\nSeed test result | "
            f"Seed={seed} | "
            f"Acc="
            f"{test_metrics['Accuracy']:.6f} | "
            f"Pre="
            f"{test_metrics['Macro Precision']:.6f} | "
            f"F1="
            f"{test_metrics['Macro F1']:.6f} | "
            f"MCC="
            f"{test_metrics['MCC']:.6f} | "
            f"Kappa="
            f"{test_metrics['Cohen Kappa']:.6f}"
        )

        del model
        del optimizer
        del scheduler
        del scaler

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    seed_dataframe = pd.DataFrame(
        seed_results
    )

    seed_dataframe.to_csv(
        SEED_OUTPUT_DIR
        / "three_seed_metrics.csv",
        index=False,
    )

    test_metric_columns = [
        "Test Accuracy",
        "Test Macro Precision",
        "Test Macro F1",
        "Test MCC",
        "Test Cohen Kappa",
    ]

    summary_rows = []

    for metric in test_metric_columns:
        summary_rows.append({
            "Metric": metric,
            "Mean":
                seed_dataframe[
                    metric
                ].mean(),
            "Standard Deviation":
                seed_dataframe[
                    metric
                ].std(ddof=1),
            "Minimum":
                seed_dataframe[
                    metric
                ].min(),
            "Maximum":
                seed_dataframe[
                    metric
                ].max(),
            "Range":
                (
                    seed_dataframe[
                        metric
                    ].max()
                    - seed_dataframe[
                        metric
                    ].min()
                ),
        })

    summary_dataframe = pd.DataFrame(
        summary_rows
    )

    summary_dataframe.to_csv(
        SEED_OUTPUT_DIR
        / "three_seed_stability_summary.csv",
        index=False,
    )

    print("\n" + "-" * 112)
    print(
        "3-SEED FIXED-SPLIT TEST RESULTS"
    )
    print("-" * 112)

    print(
        seed_dataframe[
            [
                "Seed",
                "Best Epoch",
                "Test Accuracy",
                "Test Macro Precision",
                "Test Macro F1",
                "Test MCC",
                "Test Cohen Kappa",
            ]
        ].round(6).to_string(
            index=False
        )
    )

    print(
        "\nThree-seed test stability mean ± SD"
    )

    for _, row in (
        summary_dataframe.iterrows()
    ):
        print(
            f"{row['Metric']:22s}: "
            f"{row['Mean']:.6f} ± "
            f"{row['Standard Deviation']:.6f}"
        )

    return (
        seed_dataframe,
        summary_dataframe,
        class_names,
        class_to_idx,
    )


# ============================================================
# 14. SUMMARY PLOTS
# ============================================================

def plot_cv_results(
    fold_dataframe,
):
    metrics = [
        "Accuracy",
        "Macro Precision",
        "Macro F1",
        "MCC",
        "Cohen Kappa",
    ]

    for metric in metrics:
        plt.figure(figsize=(7, 5))

        plt.plot(
            fold_dataframe["Fold"],
            fold_dataframe[metric],
            marker="o",
        )

        plt.xlabel("Fold")
        plt.ylabel(metric)

        plt.title(
            f"3-fold CV: {metric}"
        )

        plt.xticks(
            range(
                1,
                N_SPLITS + 1
            )
        )

        plt.grid(
            True,
            linestyle="--",
            alpha=0.5,
        )

        plt.tight_layout()

        filename = (
            metric.lower()
            .replace(" ", "_")
        )

        plt.savefig(
            CV_OUTPUT_DIR
            / f"{filename}_by_fold.png",
            dpi=300,
        )

        plt.close()


def plot_seed_results(
    seed_dataframe,
):
    metrics = [
        "Test Accuracy",
        "Test Macro Precision",
        "Test Macro F1",
        "Test MCC",
        "Test Cohen Kappa",
    ]

    for metric in metrics:
        plt.figure(figsize=(7, 5))

        plt.plot(
            seed_dataframe["Seed"].astype(
                str
            ),
            seed_dataframe[metric],
            marker="o",
        )

        plt.xlabel("Random seed")
        plt.ylabel(metric)

        plt.title(
            f"Random-seed stability: "
            f"{metric}"
        )

        plt.grid(
            True,
            linestyle="--",
            alpha=0.5,
        )

        plt.tight_layout()

        filename = (
            metric.lower()
            .replace(" ", "_")
        )

        plt.savefig(
            SEED_OUTPUT_DIR
            / f"{filename}_by_seed.png",
            dpi=300,
        )

        plt.close()


# ============================================================
# 15. MAIN
# ============================================================

def main():
    set_seed(CV_RANDOM_SEED)
    validate_directories()

    total_start = time.time()

    print("=" * 90)
    print(MODEL_DISPLAY_NAME)
    print("=" * 90)
    print(f"Device        : {DEVICE}")
    print(f"AMP enabled   : {USE_AMP}")
    print(f"Epochs        : {EPOCHS}")
    print(
        f"CV train path : "
        f"{CV_AUGMENTED_TRAIN_DIR}"
    )
    print(
        f"Seed train    : "
        f"{TRAIN_DIR}"
    )
    print(
        f"Seed val      : "
        f"{VAL_DIR}"
    )
    print(
        f"Seed test     : "
        f"{TEST_DIR}"
    )

    (
        cv_fold_dataframe,
        cv_summary_dataframe,
        cv_class_names,
    ) = run_three_fold_cv()

    (
        seed_dataframe,
        seed_summary_dataframe,
        seed_class_names,
        class_to_idx,
    ) = run_three_seed_stability()

    if cv_class_names != seed_class_names:
        print(
            "\nWarning: CV class ordering "
            "and fixed-split class ordering "
            "are different."
        )

    plot_cv_results(
        cv_fold_dataframe
    )

    plot_seed_results(
        seed_dataframe
    )

    total_seconds = (
        time.time() - total_start
    )

    run_summary = {
        "Model":
            MODEL_DISPLAY_NAME,

        "Epochs":
            EPOCHS,

        "CV Folds":
            N_SPLITS,

        "CV Random Seed":
            CV_RANDOM_SEED,

        "Stability Seeds":
            STABILITY_SEEDS,

        "CV Augmented Training Path":
            CV_AUGMENTED_TRAIN_DIR,

        "Fixed Training Path":
            TRAIN_DIR,

        "Fixed Validation Path":
            VAL_DIR,

        "Fixed Test Path":
            TEST_DIR,

        "Classes":
            seed_class_names,

        "Class Mapping":
            str(class_to_idx),

        "Total Runtime Hours":
            total_seconds / 3600.0,

        "Output Directory":
            OUTPUT_DIR,
    }

    save_json(
        run_summary,
        OUTPUT_DIR
        / "combined_run_summary.json",
    )

    print("\n")
    print("=" * 90)
    print("COMPLETED SUCCESSFULLY")
    print("=" * 90)
    print(
        f"Total runtime: "
        f"{total_seconds / 3600.0:.3f} hours"
    )
    print(
        f"Results: {OUTPUT_DIR}"
    )


if __name__ == "__main__":
    main()


In [ ]:
# ============================================================
# TRAIN FOUR TRANSFORMER BASELINES
#
# 1. MobileViT-S
# 2. Swin-Tiny
# 3. DeiT-Tiny
# 4. TinyViT
#
# Printed metrics:
# - Accuracy
# - Macro Precision
# - Macro Recall
# - Macro F1-score
# ============================================================

import contextlib
import gc
import os
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. INSTALL AND IMPORT TIMM
# ============================================================

try:
    import timm
except ImportError:
    print("Installing timm...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "timm"]
    )
    import timm


# ============================================================
# 2. CONFIGURATION
# ============================================================

SEED = 42

TRAIN_DIR = Path("/kaggle/working/PVMD_Train_Augmented")
VAL_DIR = Path("/kaggle/working/PVMD_Split/validation")
TEST_DIR = Path("/kaggle/working/PVMD_Split/test")

# Support folders named either validation or val
if not VAL_DIR.exists():
    alternative_val_dir = Path("/kaggle/working/PVMD_Split/val")

    if alternative_val_dir.exists():
        VAL_DIR = alternative_val_dir

OUTPUT_DIR = Path(
    "/kaggle/working/Transformer_Baseline_Comparison"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

EPOCHS = 35

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.10
GRADIENT_CLIP = 1.0

EARLY_STOPPING_PATIENCE = 7
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-7

PRETRAINED = True

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)


# ============================================================
# 4. VALIDATE DATASET DIRECTORIES
# ============================================================

def validate_directories():
    required_directories = [
        TRAIN_DIR,
        VAL_DIR,
        TEST_DIR,
    ]

    missing_directories = [
        str(directory)
        for directory in required_directories
        if not directory.exists()
    ]

    if missing_directories:
        raise FileNotFoundError(
            "The following dataset directories were not found:\n"
            + "\n".join(missing_directories)
        )


validate_directories()


# ============================================================
# 5. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomVerticalFlip(p=0.10),
    transforms.RandomRotation(degrees=10),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05,
        hue=0.02,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),

    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.08),
        ratio=(0.5, 2.0),
    ),
])


evaluation_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# ============================================================
# 6. DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    root=str(TRAIN_DIR),
    transform=train_transform,
)

validation_dataset = datasets.ImageFolder(
    root=str(VAL_DIR),
    transform=evaluation_transform,
)

test_dataset = datasets.ImageFolder(
    root=str(TEST_DIR),
    transform=evaluation_transform,
)

class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx
number_of_classes = len(class_names)

if validation_dataset.classes != class_names:
    raise ValueError(
        "Validation classes do not match training classes.\n"
        f"Training classes: {class_names}\n"
        f"Validation classes: {validation_dataset.classes}"
    )

if test_dataset.classes != class_names:
    raise ValueError(
        "Test classes do not match training classes.\n"
        f"Training classes: {class_names}\n"
        f"Test classes: {test_dataset.classes}"
    )

print("\nDataset information")
print("-------------------")
print(f"Classes            : {class_names}")
print(f"Class mapping      : {class_to_idx}")
print(f"Number of classes  : {number_of_classes}")
print(f"Training images    : {len(train_dataset)}")
print(f"Validation images  : {len(validation_dataset)}")
print(f"Test images        : {len(test_dataset)}")


# ============================================================
# 7. DATA LOADERS
# ============================================================

common_loader_arguments = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **common_loader_arguments,
)

validation_loader = DataLoader(
    validation_dataset,
    shuffle=False,
    **common_loader_arguments,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **common_loader_arguments,
)


# ============================================================
# 8. MODEL CANDIDATES
# ============================================================
#
# Multiple candidate names are supplied because timm model
# identifiers can vary slightly between timm versions.
# The first available model is automatically selected.
# ============================================================

MODEL_CANDIDATES = {
    "MobileViT-S": [
        "mobilevit_s.cvnets_in1k",
        "mobilevit_s",
    ],

    "Swin-Tiny": [
        "swin_tiny_patch4_window7_224.ms_in1k",
        "swin_tiny_patch4_window7_224",
    ],

    "DeiT-Tiny": [
        "deit_tiny_patch16_224.fb_in1k",
        "deit_tiny_patch16_224",
    ],

    "TinyViT": [
        "tiny_vit_5m_224.dist_in22k_ft_in1k",
        "tiny_vit_5m_224",
    ],
}


def resolve_timm_model_name(candidate_names):
    """
    Return the first candidate model available in the installed
    version of timm.
    """
    available_models = set(
        timm.list_models(pretrained=False)
    )

    for candidate_name in candidate_names:
        if candidate_name in available_models:
            return candidate_name

    # Also search by the base name before the first dot
    for candidate_name in candidate_names:
        base_name = candidate_name.split(".")[0]

        matching_models = timm.list_models(
            f"*{base_name}*",
            pretrained=False,
        )

        if matching_models:
            return matching_models[0]

    raise ValueError(
        "None of the following timm models were found:\n"
        + "\n".join(candidate_names)
    )


resolved_models = {}

print("\nResolved timm model names")
print("-------------------------")

for display_name, candidates in MODEL_CANDIDATES.items():
    resolved_name = resolve_timm_model_name(candidates)
    resolved_models[display_name] = resolved_name

    print(f"{display_name:12s}: {resolved_name}")


# ============================================================
# 9. AMP CONTEXT
# ============================================================

def amp_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        )

    return contextlib.nullcontext()


# ============================================================
# 10. CREATE BASELINE MODEL
# ============================================================

def create_baseline_model(
    timm_model_name,
    number_of_classes,
):
    """
    Create a pretrained timm classification model.
    """
    try:
        model = timm.create_model(
            timm_model_name,
            pretrained=PRETRAINED,
            num_classes=number_of_classes,
        )

    except Exception as error:
        if not PRETRAINED:
            raise

        print(
            "\nWarning: pretrained weights could not be loaded "
            f"for {timm_model_name}."
        )
        print(f"Reason: {error}")
        print("The model will be created without pretrained weights.")

        model = timm.create_model(
            timm_model_name,
            pretrained=False,
            num_classes=number_of_classes,
        )

    return model.to(DEVICE)


# ============================================================
# 11. METRIC FUNCTION
# ============================================================

def calculate_four_metrics(
    labels,
    predictions,
):
    """
    Calculate only the four requested metrics.
    """
    return {
        "Accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "Precision": precision_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),

        "Recall": recall_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),

        "F1-score": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }


# ============================================================
# 12. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
):
    model.train()

    labels_all = []
    predictions_all = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with amp_context():
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP,
        )

        scaler.step(optimizer)
        scaler.update()

        predictions = torch.argmax(
            logits,
            dim=1,
        )

        labels_all.extend(
            labels.detach().cpu().numpy()
        )

        predictions_all.extend(
            predictions.detach().cpu().numpy()
        )

    return calculate_four_metrics(
        np.asarray(labels_all),
        np.asarray(predictions_all),
    )


# ============================================================
# 13. EVALUATION
# ============================================================

@torch.inference_mode()
def evaluate_model(
    model,
    loader,
):
    model.eval()

    labels_all = []
    predictions_all = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        with amp_context():
            logits = model(images)

        predictions = torch.argmax(
            logits,
            dim=1,
        )

        labels_all.extend(
            labels.numpy()
        )

        predictions_all.extend(
            predictions.cpu().numpy()
        )

    labels_all = np.asarray(
        labels_all
    )

    predictions_all = np.asarray(
        predictions_all
    )

    metrics = calculate_four_metrics(
        labels_all,
        predictions_all,
    )

    return metrics, labels_all, predictions_all


# ============================================================
# 14. TRAIN ONE BASELINE
# ============================================================

def train_baseline(
    display_name,
    timm_model_name,
):
    print(
        "\n"
        + "=" * 75
        + f"\nTRAINING: {display_name}"
        + f"\nTIMM MODEL: {timm_model_name}"
        + "\n"
        + "=" * 75
    )

    # Reinitialize the random seed before every model
    set_seed(SEED)

    model = create_baseline_model(
        timm_model_name=timm_model_name,
        number_of_classes=number_of_classes,
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MIN_LEARNING_RATE,
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    safe_model_name = (
        display_name
        .replace(" ", "_")
        .replace("-", "_")
    )

    best_model_path = (
        OUTPUT_DIR
        / f"best_{safe_model_name}.pth"
    )

    best_validation_f1 = -np.inf
    epochs_without_improvement = 0

    training_start_time = time.time()

    for epoch in range(1, EPOCHS + 1):

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
        )

        validation_metrics, _, _ = evaluate_model(
            model=model,
            loader=validation_loader,
        )

        scheduler.step(
            validation_metrics["F1-score"]
        )

        current_learning_rate = (
            optimizer.param_groups[0]["lr"]
        )

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Acc: {train_metrics['Accuracy']:.4f} | "
            f"Train Pre: {train_metrics['Precision']:.4f} | "
            f"Train Rec: {train_metrics['Recall']:.4f} | "
            f"Train F1: {train_metrics['F1-score']:.4f} | "
            f"Val Acc: {validation_metrics['Accuracy']:.4f} | "
            f"Val Pre: {validation_metrics['Precision']:.4f} | "
            f"Val Rec: {validation_metrics['Recall']:.4f} | "
            f"Val F1: {validation_metrics['F1-score']:.4f} | "
            f"LR: {current_learning_rate:.2e}"
        )

        if (
            validation_metrics["F1-score"]
            > best_validation_f1 + 1e-5
        ):
            best_validation_f1 = (
                validation_metrics["F1-score"]
            )

            epochs_without_improvement = 0

            torch.save(
                {
                    "model_name": display_name,
                    "timm_model_name": timm_model_name,
                    "model_state_dict": model.state_dict(),
                    "class_names": class_names,
                    "class_to_idx": class_to_idx,
                    "best_validation_f1": best_validation_f1,
                    "epoch": epoch,
                },
                best_model_path,
            )

            print("  Best model saved.")

        else:
            epochs_without_improvement += 1

            print(
                "  No validation F1 improvement: "
                f"{epochs_without_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print("  Early stopping activated.")
            break

    training_seconds = (
        time.time() - training_start_time
    )

    # --------------------------------------------------------
    # Load the best validation checkpoint
    # --------------------------------------------------------

    checkpoint = torch.load(
        best_model_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    # --------------------------------------------------------
    # Final unseen-test evaluation
    # --------------------------------------------------------

    test_metrics, test_labels, test_predictions = (
        evaluate_model(
            model=model,
            loader=test_loader,
        )
    )

    print(
        "\n"
        + "-" * 55
        + f"\nFINAL TEST RESULT: {display_name}"
        + "\n"
        + "-" * 55
    )

    print(
        f"Accuracy  : "
        f"{test_metrics['Accuracy']:.4f}"
    )

    print(
        f"Precision : "
        f"{test_metrics['Precision']:.4f}"
    )

    print(
        f"Recall    : "
        f"{test_metrics['Recall']:.4f}"
    )

    print(
        f"F1-score  : "
        f"{test_metrics['F1-score']:.4f}"
    )

    # Save test predictions
    prediction_dataframe = pd.DataFrame({
        "True Label": test_labels,
        "Predicted Label": test_predictions,
    })

    prediction_dataframe.to_csv(
        OUTPUT_DIR
        / f"{safe_model_name}_test_predictions.csv",
        index=False,
    )

    result = {
        "Model": display_name,
        "TIMM Model": timm_model_name,
        "Accuracy": test_metrics["Accuracy"],
        "Precision": test_metrics["Precision"],
        "Recall": test_metrics["Recall"],
        "F1-score": test_metrics["F1-score"],
        "Best Validation F1": best_validation_f1,
        "Training Time Minutes": training_seconds / 60.0,
        "Best Model Path": str(best_model_path),
    }

    # Clear GPU memory before training the next model
    del model
    del optimizer
    del scheduler
    del scaler

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# 15. TRAIN ALL FOUR BASELINES
# ============================================================

baseline_results = []

for display_name, timm_model_name in resolved_models.items():

    try:
        result = train_baseline(
            display_name=display_name,
            timm_model_name=timm_model_name,
        )

        baseline_results.append(result)

    except RuntimeError as error:

        if "out of memory" in str(error).lower():
            print(
                f"\nGPU out-of-memory error while training "
                f"{display_name}."
            )

            print(
                "Reduce BATCH_SIZE, for example from 16 to 8."
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        else:
            print(
                f"\nTraining failed for {display_name}:"
            )
            print(error)

    except Exception as error:
        print(
            f"\nTraining failed for {display_name}:"
        )
        print(error)


# ============================================================
# 16. FINAL BASELINE COMPARISON
# ============================================================

if len(baseline_results) == 0:
    raise RuntimeError(
        "None of the baseline models completed successfully."
    )

results_dataframe = pd.DataFrame(
    baseline_results
)

results_dataframe = results_dataframe.sort_values(
    by="F1-score",
    ascending=False,
).reset_index(drop=True)

# Only the requested metrics are printed
print(
    "\n\n"
    + "=" * 75
    + "\nFINAL BASELINE COMPARISON ON UNSEEN TEST DATA"
    + "\n"
    + "=" * 75
)

print(
    results_dataframe[
        [
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# Complete information is saved, while the console prints
# only the four requested performance metrics.
results_dataframe.to_csv(
    OUTPUT_DIR / "baseline_comparison_results.csv",
    index=False,
)

results_dataframe[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
    ]
].to_csv(
    OUTPUT_DIR / "baseline_four_metrics_only.csv",
    index=False,
)

print("\nSaved:")
print(
    OUTPUT_DIR
    / "baseline_comparison_results.csv"
)

print(
    OUTPUT_DIR
    / "baseline_four_metrics_only.csv"
)

In [ ]:
# ============================================================
# RAW MAXVIT-TINY BASELINE
# No GRFM or custom feature module
#
# Printed metrics:
# - Accuracy
# - Macro F1-score
# ============================================================

import contextlib
import gc
import os
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")


# ============================================================
# 1. INSTALL AND IMPORT TIMM
# ============================================================

try:
    import timm
except ImportError:
    print("Installing timm...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "timm"]
    )
    import timm


# ============================================================
# 2. CONFIGURATION
# ============================================================

SEED = 42

TRAIN_DIR = Path("/kaggle/working/PVMD_Train_Augmented")
VAL_DIR = Path("/kaggle/working/PVMD_Split/validation")
TEST_DIR = Path("/kaggle/working/PVMD_Split/test")

# Support validation folder named "val"
if not VAL_DIR.exists():
    alternative_val_dir = Path(
        "/kaggle/working/PVMD_Split/val"
    )

    if alternative_val_dir.exists():
        VAL_DIR = alternative_val_dir

OUTPUT_DIR = Path(
    "/kaggle/working/Raw_MaxViT_Tiny_Results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_raw_maxvit_tiny.pth"
)

MODEL_CANDIDATES = [
    "maxvit_tiny_tf_224.in1k",
    "maxvit_tiny_tf_224",
]

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

EPOCHS = 35

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.10
GRADIENT_CLIP = 1.0

EARLY_STOPPING_PATIENCE = 7

SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-7

PRETRAINED = True

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)


# ============================================================
# 4. VALIDATE DATASET DIRECTORIES
# ============================================================

def validate_directories():
    required_directories = [
        TRAIN_DIR,
        VAL_DIR,
        TEST_DIR,
    ]

    missing_directories = [
        str(directory)
        for directory in required_directories
        if not directory.exists()
    ]

    if missing_directories:
        raise FileNotFoundError(
            "The following dataset directories were not found:\n- "
            + "\n- ".join(missing_directories)
        )


validate_directories()


# ============================================================
# 5. DATA TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.50
    ),

    transforms.RandomVerticalFlip(
        p=0.10
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.05,
        hue=0.02,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),

    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.08),
        ratio=(0.5, 2.0),
    ),
])


evaluation_transform = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# ============================================================
# 6. DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    root=str(TRAIN_DIR),
    transform=train_transform,
)

validation_dataset = datasets.ImageFolder(
    root=str(VAL_DIR),
    transform=evaluation_transform,
)

test_dataset = datasets.ImageFolder(
    root=str(TEST_DIR),
    transform=evaluation_transform,
)

class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

number_of_classes = len(class_names)

if validation_dataset.classes != class_names:
    raise ValueError(
        "Validation classes do not match training classes.\n"
        f"Training classes: {class_names}\n"
        f"Validation classes: {validation_dataset.classes}"
    )

if test_dataset.classes != class_names:
    raise ValueError(
        "Test classes do not match training classes.\n"
        f"Training classes: {class_names}\n"
        f"Test classes: {test_dataset.classes}"
    )

print("\nDataset information")
print("-------------------")
print(f"Classes           : {class_names}")
print(f"Class mapping     : {class_to_idx}")
print(f"Number of classes : {number_of_classes}")
print(f"Training images   : {len(train_dataset)}")
print(f"Validation images : {len(validation_dataset)}")
print(f"Test images       : {len(test_dataset)}")


# ============================================================
# 7. DATA LOADERS
# ============================================================

loader_settings = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_settings,
)

validation_loader = DataLoader(
    validation_dataset,
    shuffle=False,
    **loader_settings,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_settings,
)


# ============================================================
# 8. RESOLVE MAXVIT-TINY MODEL NAME
# ============================================================

def resolve_model_name(candidate_names):
    available_models = set(
        timm.list_models(
            pretrained=False
        )
    )

    for candidate_name in candidate_names:
        if candidate_name in available_models:
            return candidate_name

    for candidate_name in candidate_names:
        base_name = candidate_name.split(".")[0]

        matching_models = timm.list_models(
            f"*{base_name}*",
            pretrained=False
        )

        if matching_models:
            return matching_models[0]

    raise ValueError(
        "MaxViT-Tiny was not found in the installed timm version.\n"
        f"Candidates checked: {candidate_names}"
    )


MODEL_NAME = resolve_model_name(
    MODEL_CANDIDATES
)

print(f"\nResolved MaxViT model: {MODEL_NAME}")


# ============================================================
# 9. AMP CONTEXT
# ============================================================

def amp_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True
        )

    return contextlib.nullcontext()


# ============================================================
# 10. CREATE RAW MAXVIT-TINY
# ============================================================

def create_raw_maxvit_tiny(
    model_name,
    num_classes
):
    try:
        model = timm.create_model(
            model_name,
            pretrained=PRETRAINED,
            num_classes=num_classes
        )

    except Exception as error:
        if not PRETRAINED:
            raise

        print(
            "\nWarning: pretrained weights could not be loaded."
        )

        print(f"Reason: {error}")
        print("Using pretrained=False.")

        model = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=num_classes
        )

    return model.to(DEVICE)


model = create_raw_maxvit_tiny(
    model_name=MODEL_NAME,
    num_classes=number_of_classes
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("\nRaw MaxViT-Tiny parameter summary")
print("---------------------------------")
print(f"Total parameters    : {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")


# ============================================================
# 11. LOSS, OPTIMIZER, SCHEDULER
# ============================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    min_lr=MIN_LEARNING_RATE
)

scaler = torch.cuda.amp.GradScaler(
    enabled=USE_AMP
)


# ============================================================
# 12. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    labels,
    predictions
):
    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "Accuracy": accuracy,
        "F1-score": macro_f1,
    }


# ============================================================
# 13. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):
    model.train()

    running_loss = 0.0

    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with amp_context():
            logits = model(images)
            loss = criterion(
                logits,
                labels
            )

        scaler.scale(loss).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    average_loss = (
        running_loss
        / len(loader.dataset)
    )

    metrics = calculate_metrics(
        np.asarray(all_labels),
        np.asarray(all_predictions)
    )

    return {
        "Loss": average_loss,
        "Accuracy": metrics["Accuracy"],
        "F1-score": metrics["F1-score"],
    }


# ============================================================
# 14. VALIDATION / TEST EVALUATION
# ============================================================

@torch.inference_mode()
def evaluate_model(
    model,
    loader,
    criterion=None
):
    model.eval()

    running_loss = 0.0

    all_labels = []
    all_predictions = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels_device = labels.to(
            DEVICE,
            non_blocking=True
        )

        with amp_context():
            logits = model(images)

            if criterion is not None:
                loss = criterion(
                    logits,
                    labels_device
                )

                running_loss += (
                    loss.item()
                    * images.size(0)
                )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

    all_labels = np.asarray(
        all_labels
    )

    all_predictions = np.asarray(
        all_predictions
    )

    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )

    average_loss = (
        running_loss / len(loader.dataset)
        if criterion is not None
        else np.nan
    )

    return {
        "Loss": average_loss,
        "Accuracy": metrics["Accuracy"],
        "F1-score": metrics["F1-score"],
        "Labels": all_labels,
        "Predictions": all_predictions,
    }


# ============================================================
# 15. TRAINING LOOP
# ============================================================

history = []

best_validation_f1 = -np.inf
best_validation_loss = np.inf

epochs_without_improvement = 0

training_start_time = time.time()

print(
    "\n"
    + "=" * 75
    + "\nTRAINING RAW MAXVIT-TINY"
    + "\n"
    + "=" * 75
)


for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()

    train_results = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler
    )

    validation_results = evaluate_model(
        model=model,
        loader=validation_loader,
        criterion=criterion
    )

    scheduler.step(
        validation_results["F1-score"]
    )

    current_learning_rate = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_seconds = (
        time.time() - epoch_start_time
    )

    history.append({
        "Epoch": epoch,

        "Train Loss": (
            train_results["Loss"]
        ),

        "Train Accuracy": (
            train_results["Accuracy"]
        ),

        "Train F1-score": (
            train_results["F1-score"]
        ),

        "Validation Loss": (
            validation_results["Loss"]
        ),

        "Validation Accuracy": (
            validation_results["Accuracy"]
        ),

        "Validation F1-score": (
            validation_results["F1-score"]
        ),

        "Learning Rate": (
            current_learning_rate
        ),

        "Epoch Time Seconds": (
            epoch_seconds
        ),
    })

    # Print only accuracy and F1, plus loss/LR for training control
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Acc: {train_results['Accuracy']:.4f} | "
        f"Train F1: {train_results['F1-score']:.4f} | "
        f"Val Acc: {validation_results['Accuracy']:.4f} | "
        f"Val F1: {validation_results['F1-score']:.4f} | "
        f"LR: {current_learning_rate:.2e}"
    )

    improved = (
        validation_results["F1-score"]
        > best_validation_f1 + 1e-5
    ) or (
        abs(
            validation_results["F1-score"]
            - best_validation_f1
        ) <= 1e-5
        and validation_results["Loss"]
        < best_validation_loss
    )

    if improved:
        best_validation_f1 = (
            validation_results["F1-score"]
        )

        best_validation_loss = (
            validation_results["Loss"]
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,

                "model_name": MODEL_NAME,

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "best_validation_f1": (
                    best_validation_f1
                ),

                "best_validation_loss": (
                    best_validation_loss
                ),

                "class_names": class_names,

                "class_to_idx": class_to_idx,

                "configuration": {
                    "image_size": IMG_SIZE,
                    "batch_size": BATCH_SIZE,
                    "epochs": EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "label_smoothing": LABEL_SMOOTHING,
                    "seed": SEED,
                    "pretrained": PRETRAINED,
                },
            },
            BEST_MODEL_PATH
        )

        print("  Best checkpoint saved.")

    else:
        epochs_without_improvement += 1

        print(
            "  No validation F1 improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("\nEarly stopping activated.")
        break


total_training_seconds = (
    time.time() - training_start_time
)

history_dataframe = pd.DataFrame(
    history
)

history_path = (
    OUTPUT_DIR / "training_history.csv"
)

history_dataframe.to_csv(
    history_path,
    index=False
)

print(
    f"\nTraining completed in "
    f"{total_training_seconds / 60:.2f} minutes."
)


# ============================================================
# 16. LOAD BEST MODEL
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print(
    f"\nLoaded best checkpoint from epoch "
    f"{checkpoint['epoch']}."
)

print(
    f"Best validation F1-score: "
    f"{checkpoint['best_validation_f1']:.4f}"
)


# ============================================================
# 17. FINAL UNSEEN TEST EVALUATION
# ============================================================

test_results = evaluate_model(
    model=model,
    loader=test_loader,
    criterion=None
)

test_accuracy = (
    test_results["Accuracy"]
)

test_f1 = (
    test_results["F1-score"]
)

print(
    "\n"
    + "=" * 75
    + "\nRAW MAXVIT-TINY FINAL TEST RESULT"
    + "\n"
    + "=" * 75
)

print(
    f"Accuracy : {test_accuracy:.4f}"
)

print(
    f"F1-score : {test_f1:.4f}"
)


# ============================================================
# 18. SAVE FINAL RESULTS
# ============================================================

final_results_dataframe = pd.DataFrame([
    {
        "Model": "Raw MaxViT-Tiny",
        "TIMM Model": MODEL_NAME,
        "Accuracy": test_accuracy,
        "F1-score": test_f1,
        "Best Validation F1": (
            checkpoint["best_validation_f1"]
        ),
        "Best Epoch": checkpoint["epoch"],
        "Training Time Minutes": (
            total_training_seconds / 60
        ),
    }
])

final_results_path = (
    OUTPUT_DIR / "raw_maxvit_tiny_test_results.csv"
)

final_results_dataframe.to_csv(
    final_results_path,
    index=False
)


# Save unseen-test predictions
predictions_dataframe = pd.DataFrame({
    "True Label": (
        test_results["Labels"]
    ),

    "Predicted Label": (
        test_results["Predictions"]
    ),

    "Correct": (
        test_results["Labels"]
        == test_results["Predictions"]
    ),
})

predictions_path = (
    OUTPUT_DIR / "raw_maxvit_tiny_test_predictions.csv"
)

predictions_dataframe.to_csv(
    predictions_path,
    index=False
)


# Print only the requested final comparison columns
print("\nFinal result table")
print("------------------")

print(
    final_results_dataframe[
        [
            "Model",
            "Accuracy",
            "F1-score",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\nSaved:")
print(history_path)
print(final_results_path)
print(predictions_path)
print(BEST_MODEL_PATH)


# ============================================================
# 19. CLEAN MEMORY
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()